In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 110)

# Notebook tự tìm file ở nhiều chỗ: thư mục hiện tại, và đường dẫn Kaggle.
CANDIDATES = ['.', '/mnt/user-data/uploads', '/kaggle/input/competitions/rossmann-store-sales',
              '/kaggle/input/rossmann-store-sales', '../input/rossmann-store-sales']
def find(*names):
    for d in CANDIDATES:
        for n in names:
            p = Path(d) / n
            if p.exists(): return p
    raise FileNotFoundError(f'Không tìm thấy {names} trong {CANDIDATES}')

OUT_DIR = Path('.')      # nơi lưu các file output F02_*.csv (đổi thành /kaggle/working nếu chạy trên Kaggle)

## 0. Load dữ liệu
Dùng đúng 3 file đã có trong U02. `StateHoliday` ép về chuỗi (`'0','a','b','c'`) ngay lúc đọc để tránh lỗi 0 (số) và '0' (chuỗi) bị tách thành 2 nhóm (đã phát hiện ở U02 mục 1.2).

In [ ]:
train = pd.read_csv(find('train_ross_1.csv', 'train_ross.csv', 'train.csv'), parse_dates=['Date'], dtype={'StateHoliday': str})
test  = pd.read_csv(find('test_ross_1.csv',  'test_ross.csv',  'test.csv'),  parse_dates=['Date'], dtype={'StateHoliday': str})
store = pd.read_csv(find('store_ross_1.csv', 'store_ross.csv', 'store.csv'))
for df in (train, test):
    df['StateHoliday'] = df['StateHoliday'].astype(str).replace('0.0', '0')

print('train', train.shape, '| test', test.shape, '| store', store.shape)

---
## 1. Bài toán forecasting được định nghĩa chính xác như thế nào?

**Vì sao cần làm bước này:** mọi luật phía sau (lag bao nhiêu ngày, chia fold ra sao) đều suy ra từ 3 con số: *horizon* (dự báo bao xa), *gap* (khoảng cách giữa cuối train và đầu test) và *phạm vi cửa hàng*.

In [ ]:
TRAIN_START, TRAIN_END = train.Date.min(), train.Date.max()
TEST_START,  TEST_END  = test.Date.min(),  test.Date.max()
HORIZON = (TEST_END - TEST_START).days + 1          # số ngày phải dự báo
GAP     = (TEST_START - TRAIN_END).days - 1         # số ngày "trống" giữa train và test
test_stores = np.sort(test.Store.unique())

print(f'Train: {TRAIN_START.date()} -> {TRAIN_END.date()} ({(TRAIN_END-TRAIN_START).days+1} ngày)')
print(f'Test : {TEST_START.date()} -> {TEST_END.date()}  => HORIZON = {HORIZON} ngày ({HORIZON/7:.1f} tuần)')
print(f'GAP giữa cuối train và đầu test = {GAP} ngày (test bắt đầu NGAY SAU train)')
print(f'Số cửa hàng trong test: {len(test_stores)} / {store.Store.nunique()} (thiếu {store.Store.nunique()-len(test_stores)} CH)')
print(f'Số dòng test: {len(test):,} | Open=1: {(test.Open==1).sum():,} | Open=0: {(test.Open==0).sum():,} | Open thiếu: {test.Open.isna().sum()}')

**Kết luận mục 1 (nền tảng cho các luật sau):**
- **HORIZON = 48 ngày**, **GAP = 0** → ngày dự báo đầu tiên là ngày ngay sau ngày cuối cùng có dữ liệu. Ngày dự báo thứ 48 cách ngày cuối có dữ liệu 48 ngày.
- Nghĩa là khi dự báo, ta **không hề biết Sales của bất kỳ ngày nào trong 48 ngày đó** → mọi biến "lấy từ Sales" phải được tính từ dữ liệu **trước** thời điểm dự báo. Đây là gốc của luật chống rò rỉ ở mục 5.
- Model chỉ cần (và chỉ nên được chấm trên) **856 cửa hàng** có trong test; 259 CH còn lại vẫn dùng được để **train** (nhiều dữ liệu hơn cho model chung) nhưng **không đưa vào điểm validation** (R9).
- Điểm chấm là **RMSPE** (sai số theo %), bỏ qua các dòng có `Sales = 0` (xem mục 8).

---
## 2. R1 - Biến nào được phép dùng? (biết trước vs chỉ biết quá khứ)

**Giải thích 30 giây:** khi dự báo cho ngày D trong tương lai, có 2 loại thông tin:
- **Biết trước:** những thứ đã nằm sẵn trong file test (lịch, khuyến mãi, ngày lễ, cửa hàng có mở không, đặc điểm cửa hàng).
- **Chỉ biết quá khứ:** `Sales`, `Customers`. Tương lai của chúng chính là thứ đang phải dự báo.

Nếu đưa vào model một biến mà lúc dự báo thật **không có**, điểm validation sẽ đẹp giả (data leakage).

In [ ]:
tr_cols, te_cols, st_cols = set(train.columns), set(test.columns), set(store.columns)
rows = []
for c in sorted(tr_cols | te_cols | st_cols):
    in_tr, in_te, in_st = c in tr_cols, c in te_cols, c in st_cols
    if c == 'Id':                       kind = 'Chỉ là mã dòng để nộp bài, KHÔNG phải feature'
    elif c in ('Sales', 'Customers'):   kind = 'CHỈ BIẾT QUÁ KHỨ (không có trong test) -> không dùng trực tiếp'
    elif in_te or in_st:                kind = 'BIẾT TRƯỚC (có trong test hoặc là thuộc tính cố định của cửa hàng)'
    else:                               kind = '?'
    rows.append((c, 'x' if in_tr else '', 'x' if in_te else '', 'x' if in_st else '', kind))
audit = pd.DataFrame(rows, columns=['Cột', 'train', 'test', 'store', 'Phân loại'])
audit

**Kết luận mục 2 → Luật R1:**
- **R1.1** Chỉ dùng trực tiếp các cột có trong `test.csv` và `store.csv`.
- **R1.2** `Customers` **bị chặn** (không có trong test). Chỉ được dùng dưới dạng *lịch sử* (ví dụ trung bình `Customers` của khoảng thời gian cách ít nhất 48 ngày) và đó cũng chỉ là biến tùy chọn.
- **R1.3** `Sales` chỉ được dùng làm (a) biến mục tiêu và (b) nguyên liệu để tạo biến lịch sử theo luật R5.
- **R1.4** Mọi thống kê theo cửa hàng (ví dụ "Sales trung bình của cửa hàng") **phải tính trong phạm vi thời gian được phép của từng fold** (R6), không tính trên toàn bộ train.

---
## 3. R2 - R3: Quy tắc cho `Open` và `Sales = 0`

**Vấn đề:** `Sales = 0` có 2 nghĩa khác nhau: (1) cửa hàng đóng cửa, (2) cửa hàng mở nhưng không bán được gì (rất hiếm, nhiều khả năng lỗi dữ liệu). Còn trong test thì có 5.984 dòng đóng cửa và 11 dòng `Open` bị thiếu.

In [ ]:
# --- (a) train: 2 loại Sales = 0
closed_rows = (train.Open == 0).sum()
open_zero   = ((train.Open == 1) & (train.Sales == 0)).sum()
print(f'train: Open=0 -> {closed_rows:,} dòng | Open=1 nhưng Sales=0 -> {open_zero} dòng')
print('Open=0 mà Sales != 0:', ((train.Open == 0) & (train.Sales != 0)).sum(), '(kỳ vọng 0)')

# --- (b) test: Open=0 rơi vào thứ mấy?
print('\ntest: Open=0 theo DayOfWeek (1=T2 ... 7=CN):')
print(test[test.Open == 0].DayOfWeek.value_counts().sort_index().to_dict())

# --- (c) 11 dòng Open thiếu
miss = test[test.Open.isna()]
print('\n11 dòng Open thiếu thuộc cửa hàng:', miss.Store.unique().tolist(), '| thứ trong tuần:', sorted(miss.DayOfWeek.unique()))
hist622 = train[train.Store == 622].groupby('DayOfWeek').Open.mean().round(3)
print('Tỷ lệ mở cửa lịch sử của Store 622 theo thứ:', hist622.to_dict())

**Nhận xét:** Open=0 trong test chủ yếu là Chủ nhật (5.803/5.984 dòng); còn lại là 165 dòng T7 (một số cửa hàng nghỉ T7) và 16 dòng T2-T6 (nghỉ lễ hoặc đóng cửa tạm). Store 622 trong lịch sử mở cửa 94-100% các ngày T2-T7 và đóng cửa 100% Chủ nhật; 11 dòng thiếu đều rơi vào T2-T7 (khớp U02 mục 1.12).

### Vì sao phải làm đúng? Thử tính "đoán nhầm 0" tốn bao nhiêu điểm RMSPE
RMSPE chia cho giá trị thật, nên nếu cửa hàng **thực sự mở** mà ta dự báo `0` thì sai số của dòng đó là 100% (bằng 1.0 trong công thức). Dòng đóng cửa thì **bị bỏ qua** khi chấm. Cho nên: *đoán 0 cho ngày đóng cửa thì an toàn, đoán 0 cho ngày mở cửa thì rất đắt.*

In [ ]:
n_eval = int(((test.Open == 1) | test.Open.isna()).sum())    # số dòng sẽ được chấm (xấp xỉ)
n_bad  = int(test.Open.isna().sum())                        # 11 dòng nếu lỡ gán Open=0
sim = []
for base in (0.10, 0.12, 0.15, 0.22):
    new = np.sqrt(base**2 + n_bad / n_eval)
    sim.append((base, round(new, 4), round(new - base, 4)))
pd.DataFrame(sim, columns=['RMSPE của model (giả định)', 'RMSPE nếu gán nhầm Open=0 cho 11 dòng', 'Điểm bị cộng thêm'])

**Kết luận mục 3 → Luật R2, R3:**
- **R2 (khi train):** chỉ train trên dòng `Open = 1` **và** `Sales > 0`. Loại toàn bộ dòng `Open = 0` và 54 dòng `Open = 1 & Sales = 0` (outlier hiếm, không đại diện hành vi bình thường).
- **R3.1 (khi dự báo):** `Open = 0` → gán `Sales = 0` (không cần model).
- **R3.2** 11 dòng `Open` thiếu (toàn bộ Store 622) → **gán `Open = 1`** (rule đã chốt ở U02 mục 1.12). Gán nhầm 0 làm RMSPE tăng thêm khoảng 0.001-0.002 (bảng trên): nhỏ nhưng là lỗi hoàn toàn tránh được.
- **R3.3** Các dòng `Open = 1` còn lại mới đưa vào model.

---
## 4. R4 - Quy tắc cho `Promo` (khuyến mãi ngắn hạn)

**Vấn đề:** `Promo` là biến mạnh nhất (U02: +38% Sales), và nó **có sẵn trong test** (biết trước). Nhưng nếu hiểu sai lịch của nó thì sẽ tạo biến sai. Cần xem: Promo chạy vào thứ mấy, và lịch Promo trong test trông như thế nào.

In [ ]:
print('Promo trung bình theo DayOfWeek')
print(pd.DataFrame({'train': train.groupby('DayOfWeek').Promo.mean().round(3),
                    'test':  test.groupby('DayOfWeek').Promo.mean().round(3)}).T)
print('\nPromo=1 vào T7/CN: train =', train[train.DayOfWeek >= 6].Promo.sum(), '| test =', test[test.DayOfWeek >= 6].Promo.sum())

# Lịch Promo trong test theo tuần (T2-T6)
t = test[test.DayOfWeek <= 5].copy(); t['week_start'] = t.Date - pd.to_timedelta(t.Date.dt.dayofweek, unit='D')
wk = t.groupby('week_start').Promo.agg(['mean', 'nunique'])
wk.columns = ['tỷ lệ cửa hàng có Promo', 'số giá trị Promo khác nhau trong tuần']
print('\nLịch Promo trong test theo tuần:'); print(wk)

**Nhận xét:** Promo **chỉ chạy T2-T6**, không bao giờ vào T7/CN. Trong test, lịch Promo là **luân phiên từng tuần** (tuần bật, tuần tắt) và **cả 856 cửa hàng cùng bật/tắt trong cùng một tuần** → với test, `Promo` thực chất là một **lịch chung của cả chuỗi**, biết trước cho cả 48 ngày.

Hệ quả quan trọng: vì khuyến mãi luân phiên theo tuần, nếu lấy "doanh số cùng thứ tuần trước" làm dự báo mà không để ý tuần đó có Promo hay không thì sẽ sai rất nhiều (tuần trước thường ở trạng thái Promo **ngược** với tuần cần dự báo). Ở mục 6 ta sẽ đo con số này.

**Kết luận mục 4 → Luật R4:**
- **R4.1** Dùng nguyên `Promo` trong test như đã cho, **không tự tạo/gán lại Promo** (đặc biệt không gán Promo=1 cho T7/CN).
- **R4.2** Có thể tạo thêm biến từ lịch Promo vì nó **biết trước cả tương lai**: `IsPromoWeek` (tuần đó có Promo không), `Promo_prev7` và `Promo_next7` (Promo cùng thứ tuần trước / tuần sau), `DaysSincePromoStart`. Ở biên cuối test thì `Promo_next7` = NaN, để model tự xử lý hoặc đặt = -1.
- **R4.3** Mọi biến lịch sử của Sales **phải được "ghép theo trạng thái Promo"**: so sánh ngày Promo với các ngày Promo trong quá khứ, ngày không Promo với ngày không Promo (xem `dow_promo_mean` ở mục 5).

---
## 5. R5 - R8: Biến lag / rolling, kiểm tra rò rỉ và dữ liệu thiếu cấu trúc

### 5.1 Ý tưởng bằng 1 hình
```
 ... quá khứ ...  |<------------------ 48 ngày cần dự báo ------------------>|
 ---------------[T = ngày cuối có dữ liệu]--[D+1]--[D+2]-- ... --[D+47]--[D+48]
                      ^
                      |  Ngày dự báo thứ h (h = 1..48) chỉ được dùng Sales của
                      |  các ngày <= T. Muốn dùng 1 luật chung cho cả 48 ngày:
                      |  CHỈ ĐƯỢC DÙNG Sales của các ngày <= (ngày cần dự báo - 48).
```
**Luật R5 (lag >= 48):** với dòng (cửa hàng s, ngày t), biến lịch sử chỉ lấy từ ngày **≤ t − 48**. Khi đó ngày t = T+h luôn dùng được vì t − 48 ≤ T. Cùng một công thức áp cho **cả train lẫn test** nên phân phối biến ở train và test giống nhau.

*Vì sao không dùng "hôm qua" hay "tuần trước"?* Với ngày dự báo thứ 10, "hôm qua" nằm trong khoảng ta chưa biết Sales (chỉ dùng được nếu dự báo từng ngày một và đưa kết quả dự báo ngược lại làm đầu vào, gọi là dự báo đệ quy, sai số cộng dồn nên nhóm KHÔNG dùng).

### 5.2 Dựng "ma trận" ngày x cửa hàng để tính lag cho nhanh
Mỗi ô = log(Sales+1) của cửa hàng đó ngày đó, và là `NaN` nếu cửa hàng đóng cửa, thiếu ngày, hay Sales=0. Nhờ vậy khi tính trung bình, các ngày đóng cửa **không kéo trung bình xuống** (đúng với R2).

In [ ]:
ALL_DATES = pd.date_range(TRAIN_START, TEST_END, name='Date')
STORES = pd.Index(np.sort(store.Store.unique()), name='Store')

def to_mat(df, col):
    return df.pivot(index='Date', columns='Store', values=col).reindex(index=ALL_DATES, columns=STORES)

S      = to_mat(train, 'Sales')
O      = to_mat(train, 'Open')
Promo  = pd.concat([to_mat(train, 'Promo').loc[:TRAIN_END], to_mat(test, 'Promo').loc[TEST_START:]])   # lịch Promo biết trước cả test
valid  = (O == 1) & (S > 0)
logS   = np.log1p(S.where(valid))            # NaN nếu đóng cửa / thiếu ngày / Sales=0
print('Ma trận ngày x cửa hàng:', logS.shape, '| tỷ lệ ô có giá trị:', round(logS.notna().mean().mean(), 3))

### 5.3 Hàm tạo biến lịch sử (đã tuân thủ R5) + bộ biến đề xuất

| Biến | Ý nghĩa (nói đơn giản) | Cửa sổ dữ liệu được dùng |
|---|---|---|
| `dow_promo_mean` | Doanh số trung bình (log) của **cùng cửa hàng, cùng thứ, cùng trạng thái Promo** trong ~11 tuần gần nhất cách ít nhất 48 ngày | lag 49, 56, ..., 119 |
| `dow_promo_n` | Có bao nhiêu mẫu để tính biến trên (đo độ tin cậy) | như trên |
| `roll28`, `roll91` | Mức bán trung bình gần đây của cửa hàng (28 / 91 ngày), chỉ tính ngày mở cửa | kết thúc ở t − 48 |
| `open28` | Số ngày mở cửa trong 28 ngày đó (phát hiện cửa hàng mới đóng cửa dài ngày) | kết thúc ở t − 48 |
| `promo_uplift91` | Cửa hàng này tăng bao nhiêu (log) khi có Promo so với không Promo trong 91 ngày | kết thúc ở t − 48 |
| `ly364`, `ly728` | Doanh số cùng thứ **1 năm / 2 năm trước** (364 / 728 ngày, giữ nguyên thứ trong tuần) | lag 364, 728 |

In [ ]:
def first_multiple_of_7_at_least(h):
    return int(np.ceil(h / 7) * 7)

def same_dow_same_promo_mean(logS, Promo, lags):
    # Trung bình log-sales của các ngày CÙNG THỨ (lag bội của 7) và CÙNG trạng thái Promo với ngày cần dự báo.
    num, den = 0, 0
    for k in lags:
        v  = logS.shift(k)
        ok = v.notna() & (Promo.shift(k) == Promo)
        num = num + v.where(ok).fillna(0); den = den + ok.astype(float)
    return num / den.replace(0, np.nan), den

def build_history_features(logS, Promo, horizon=HORIZON):
    first = first_multiple_of_7_at_least(horizon)             # 49 khi horizon = 48
    lags  = list(range(first, first + 7 * 11, 7))             # 49, 56, ..., 119
    assert min(lags) >= horizon, 'Vi phạm luật R5: lag phải >= horizon'
    L = logS.shift(horizon)                                   # mọi cửa sổ rolling kết thúc ở t - horizon
    wk = pd.Series(ALL_DATES.dayofweek < 5, index=ALL_DATES).shift(horizon).fillna(False).values[:, None]
    f = {}
    f['dow_promo_mean'], f['dow_promo_n'] = same_dow_same_promo_mean(logS, Promo, lags)
    f['roll28'] = L.rolling(28, min_periods=7).mean()
    f['roll91'] = L.rolling(91, min_periods=20).mean()
    f['open28'] = logS.notna().astype(float).shift(horizon).rolling(28, min_periods=1).sum()
    mp = L.where(Promo.shift(horizon) == 1).rolling(91, min_periods=5).mean()
    mn = L.where((Promo.shift(horizon) == 0) & wk).rolling(91, min_periods=5).mean()
    f['promo_uplift91'] = mp - mn
    f['ly364'] = logS.shift(364)
    f['ly728'] = logS.shift(728)
    return f

HIST = build_history_features(logS, Promo)
print('Đã tạo', len(HIST), 'biến lịch sử, mỗi biến là ma trận', logS.shape)

### 5.4 Test chống rò rỉ (quan trọng nhất của F02)
**Cách kiểm tra "cắt dữ liệu":** giả vờ ta chỉ biết dữ liệu đến ngày `C` (xóa hết Sales sau `C`), tính lại toàn bộ biến, rồi so với biến tính từ dữ liệu đầy đủ.
- Với mọi ngày `t ≤ C + 48`, biến **phải y hệt nhau** (vì luật R5 nói các ngày này chỉ dùng dữ liệu ≤ C).
- Ngược lại, với `t > C + 48` biến **nên khác** (chứng tỏ test đủ nhạy để bắt lỗi).
- Làm thêm 1 biến **cố tình rò rỉ** (`lag7`: Sales 7 ngày trước) để chứng minh test này bắt được lỗi thật.

In [ ]:
def leakage_test(cutoff, features_fn, horizon=HORIZON):
    cutoff = pd.Timestamp(cutoff)
    logS_cut = logS.copy(); logS_cut.loc[logS_cut.index > cutoff] = np.nan       # "chỉ biết dữ liệu đến cutoff"
    full, cut = features_fn(logS), features_fn(logS_cut)
    inside  = slice(None, cutoff + pd.Timedelta(days=horizon))
    outside = slice(cutoff + pd.Timedelta(days=horizon + 1), TRAIN_END)
    res = []
    for k in full:
        same_inside = np.allclose(full[k].loc[inside].values, cut[k].loc[inside].values, equal_nan=True)
        diff_after  = not np.allclose(full[k].loc[outside].values, cut[k].loc[outside].values, equal_nan=True)
        res.append((k, 'ĐẠT' if same_inside else 'RÒ RỈ!', 'có' if diff_after else 'không (n/a)'))
    return pd.DataFrame(res, columns=['Biến', f'Giống hệt khi cắt tại {cutoff.date()} (t <= C+48)?', 'Có đổi khi t > C+48 (test nhạy)?'])

fn_good  = lambda ls: build_history_features(ls, Promo)
fn_leaky = lambda ls: {'lag7_RÒ_RỈ': ls.shift(7), 'lag1_RÒ_RỈ': ls.shift(1)}
display(leakage_test('2015-03-09', fn_good))
display(leakage_test('2015-03-09', fn_leaky))

**Kết luận 5.4:** toàn bộ biến theo R5 đều **ĐẠT**; hai biến cố tình rò rỉ bị bắt ngay (`RÒ RỈ!`). (Với `ly364`, `ly728` cột "có đổi" ghi *n/a* vì tham chiếu 364 / 728 ngày trước nằm ngoài phạm vi dữ liệu khi cắt tại 03/2015, không phải lỗi.)

→ **Luật R5 kèm test này sẽ được F04 dùng lại nguyên văn** như một bước kiểm tra bắt buộc sau khi tạo dataset.

### 5.5 Phát hiện: dữ liệu thiếu có cấu trúc (cần xử lý riêng)
U02 mục 1.10 ghi 181 cửa hàng chỉ có 758/942 ngày và nhận định là "bắt đầu có dữ liệu muộn hơn". Kiểm tra lại cho F02 thì **không phải lúc bắt đầu, mà là một khoảng trống ở giữa**:

In [ ]:
cnt = train.groupby('Store').Date.count()
short_stores = cnt[cnt < cnt.max()].index
print('Số cửa hàng thiếu ngày:', len(short_stores), '|', cnt[short_stores].value_counts().to_dict())
alld = set(ALL_DATES[ALL_DATES <= TRAIN_END])
gaps = {}
for s in short_stores:
    miss_days = sorted(alld - set(train.loc[train.Store == s, 'Date']))
    gaps[s] = (miss_days[0].date(), miss_days[-1].date(), len(miss_days))
gdf = pd.DataFrame(gaps, index=['thiếu từ', 'thiếu đến', 'số ngày thiếu']).T
print(gdf.groupby(['thiếu từ', 'thiếu đến', 'số ngày thiếu']).size().rename('số cửa hàng'))
print('\nSố cửa hàng thiếu này nằm trong test:', len(set(short_stores) & set(test_stores)), '/', len(short_stores))
# Hệ quả: tỷ lệ NaN của roll28 ở 180 cửa hàng mất 184 ngày, theo tháng
gap_stores = [s for s, g in gaps.items() if g[2] == 184]
nan_m = HIST['roll28'][gap_stores].isna().mean(axis=1).groupby(ALL_DATES.to_period('M')).mean()
print('Tỷ lệ NaN của roll28 ở 180 CH này theo tháng:', {str(k): round(float(v), 2) for k, v in nan_m.loc['2014-07':'2015-04'].items()})
n_by_date = S.loc[:TRAIN_END].notna().sum(axis=1)
fig, ax = plt.subplots(figsize=(9, 3.2)); n_by_date.plot(ax=ax, color='#4472C4')
ax.set_title('Daily Active Stores'); ax.set_ylabel('Store Count'); plt.tight_layout(); plt.show()


**Phát hiện (sửa lại nhận định ở U02 mục 1.10):** **180 cửa hàng mất đúng 184 ngày liên tục (01/07/2014 → 31/12/2014)**, 1 cửa hàng thiếu 1 ngày lẻ. Cả 181 cửa hàng này **đều nằm trong test** (chiếm 21% số cửa hàng cần dự báo).

**Hệ quả trực tiếp cho dự báo (đây là chỗ dễ bị bỏ sót):**
1. Test là 01/08 → 17/09/**2015**; "cùng kỳ năm ngoái" là 01/08 → 17/09/**2014** - đúng khoảng bị thiếu của 180 cửa hàng này → `ly364` = NaN cho họ khi dự báo.
2. Các cửa sổ rolling của họ cũng trống theo (kết thúc ở t − 48 rơi vào nửa cuối 2014): `roll28` NaN gần 100% từ tháng 10/2014 đến hết tháng 1/2015 và ~93% tháng 2/2015 (bảng ở trên), nên các dòng train của 180 cửa hàng này trong giai đoạn đó gần như không có biến lịch sử; `dow_promo_mean`, `roll91` cũng bị ảnh hưởng tương tự.
3. Fold validation "cùng mùa năm trước" (F4, mục 6) chỉ chấm được trên các cửa hàng còn lại (~676 CH trong tập test).

**Luật R6 - R8:**
- **R6 (tính biến theo từng fold):** mọi thống kê cửa hàng (profile) phải được tính **chỉ từ dữ liệu trước mốc cắt của fold**. Cấm tính "Sales trung bình của cửa hàng" trên toàn bộ train rồi dùng cho validation (sẽ minh họa ở mục 6).
- **R7 (dữ liệu thiếu cấu trúc):** **không impute bừa** bằng 0 hay trung bình toàn chuỗi. (a) Giữ `NaN` + thêm cờ `has_ly364 = ly364.notna()`; (b) dùng `ly728` (2 năm trước, các cửa hàng này vẫn có dữ liệu 08-09/2013) như biến dự phòng; (c) Gradient Boosting xử lý NaN trực tiếp, còn Random Forest cần điền trước: dùng `roll91`, nếu vẫn thiếu thì dùng trung bình theo `StoreType x Assortment` cùng thời điểm.
- **R8 (warm-up):** các ngày đầu train chưa đủ lịch sử để tạo biến → loại khỏi tập train (giữ lại làm nguyên liệu).

In [ ]:
# R8: warm-up = ngày đầu tiên mà (trung vị theo cửa hàng) có >= 4 mẫu cho dow_promo_mean và roll91 khác NaN ở >= 90% cửa hàng
n_med = HIST['dow_promo_n'].loc[:TRAIN_END].median(axis=1)
cov91 = HIST['roll91'].loc[:TRAIN_END].notna().mean(axis=1)
warm_start = ALL_DATES[ALL_DATES <= TRAIN_END][(n_med >= 4).values & (cov91 >= 0.9).values][0]
print('Ngày đầu tiên đủ lịch sử (warm-up kết thúc):', warm_start.date(), f'-> loại {(warm_start - TRAIN_START).days} ngày đầu khỏi tập train')

# Độ phủ NaN của từng biến trên các dòng TEST sẽ được dự báo (856 CH, Open=1)
tt = test[(test.Open != 0)][['Store', 'Date']]
ri, ci = ALL_DATES.get_indexer(tt.Date), STORES.get_indexer(tt.Store)     # vị trí (ngày, cửa hàng) của từng dòng test trong ma trận
cov = {k: float(np.isnan(m.values[ri, ci]).mean()) for k, m in HIST.items()}
pd.Series(cov).round(3).rename('Tỷ lệ NaN trên dòng test (Open!=0)').to_frame()

**Nhận xét:** `ly364` thiếu ở khoảng 1/5 dòng test (đúng là 180 cửa hàng bị khoảng trống 2014), còn `ly728` lại phủ được hầu hết các cửa hàng đó. Các biến còn lại phủ gần đầy đủ. Đây là lý do R7 bắt buộc giữ cả `ly364` và `ly728` cùng cờ thiếu.

---
## 6. R9 - Validation Strategy: chia fold walk-forward + mức sàn (baseline)

**Vấn đề:** điểm thật chỉ có khi nộp lên Kaggle (Sales của test nằm ở Kaggle). Vì vậy ta phải **tự dựng các "bài thi thử" giống test** để chọn và tune model, và chỉ nộp Kaggle như **kỳ thi cuối**.

**Nguyên tắc (nói đơn giản):**
- **KHÔNG chia ngẫu nhiên (random K-fold).** Chia ngẫu nhiên sẽ để model nhìn thấy ngày thứ 10 và thứ 12 của một tuần rồi đoán ngày thứ 11 → quá dễ so với test thật (test hoàn toàn nằm ở tương lai).
- Mỗi fold mô phỏng đúng test: *train đến mốc T, dự báo 48 ngày sau T* (**walk-forward**). Làm **4 fold**, mỗi fold dài đúng 48 ngày.
- 1 fold phải **cùng mùa** với test (01/08 → 17/09/2014) vì test rơi vào cuối hè/đầu thu.

In [ ]:
FOLD_ORIGINS = {
    'F1 (gần test nhất)':       pd.Timestamp('2015-06-13'),   # dự báo 14/06 -> 31/07/2015
    'F2':                       pd.Timestamp('2015-04-26'),   # dự báo 27/04 -> 13/06/2015
    'F3 (có lễ Phục sinh)':     pd.Timestamp('2015-03-09'),   # dự báo 10/03 -> 26/04/2015
    'F4 (cùng mùa năm trước)':  pd.Timestamp('2014-07-31'),   # dự báo 01/08 -> 17/09/2014
}
cols = [s for s in STORES if s in set(test_stores)]            # chỉ chấm trên 856 cửa hàng có trong test

def rmspe(y, yhat):
    y = np.asarray(y, float); yhat = np.asarray(yhat, float); m = y != 0       # Kaggle bỏ qua dòng Sales = 0
    return float(np.sqrt(np.mean(((y[m] - yhat[m]) / y[m]) ** 2)))

rows = []
for name, T in FOLD_ORIGINS.items():
    v0, v1 = T + pd.Timedelta(days=1), T + pd.Timedelta(days=HORIZON)
    Sv, Ov = S.loc[v0:v1, cols], O.loc[v0:v1, cols]
    ok = (Ov == 1) & (Sv > 0)
    vd = train[(train.Date >= v0) & (train.Date <= v1) & train.Store.isin(cols) & (train.Open == 1)]
    rows.append({'Fold': name, 'Mốc cắt T (hết dữ liệu train)': T.date(), 'Giai đoạn validate': f'{v0.date()} -> {v1.date()}',
                 'Số ngày': (v1 - v0).days + 1, 'Số CH có Sales': int(Sv.notna().any().sum()), 'Số dòng được chấm': int(ok.values.sum()),
                 'Tỷ lệ dòng Promo': round(vd.Promo.mean(), 2), 'Số dòng ngày lễ nhà nước': int((vd.StateHoliday != '0').sum())})
folds_df = pd.DataFrame(rows)
folds_df

**Nhận xét về các fold:**
- Mọi fold dài đúng **48 ngày** như test, và **không có khoảng trống (gap = 0)** giữa cuối train và đầu validate, y như test.
- **F4** chỉ chấm được ~676/856 cửa hàng vì 180 cửa hàng thiếu dữ liệu giai đoạn đó (mục 5.5).
- **F2 và F3** có ngày lễ nhà nước (bảng trên: 80 và 39 dòng mở cửa trong ngày lễ; F3 có Phục sinh), còn **F1 và F4 không có ngày lễ nào**. Test chỉ có **đúng 1 ngày lễ cấp bang** (15/08/2015, 180 cửa hàng, chỉ 26 dòng mở cửa) → F1 và F4 giống test nhất về yếu tố lễ; với F2, F3 nên báo cáo thêm điểm "loại ngày lễ".
- **F1** gần test nhất về thời gian, F4 gần test nhất về mùa vụ → **tune dựa trên trung bình 4 fold, nhưng luôn báo cáo từng fold riêng**.

### 6.2 Mức sàn: dự báo "ngây thơ" không cần ML (và minh họa rò rỉ)
5 dự báo đơn giản, đo trên đúng 4 fold ở trên. Mục đích: (1) cho biết **mức điểm tối thiểu mà ML phải vượt qua**; (2) chứng minh Promo là yếu tố quyết định; (3) cho thấy rò rỉ làm điểm đẹp giả **bao nhiêu**.

| Mã | Dự báo | Hợp lệ? |
|---|---|---|
| `B1` | Trung bình 8 tuần gần nhất của **cùng cửa hàng, cùng thứ** (tính đến mốc T) | Hợp lệ |
| `B2` | Như B1 nhưng **cùng trạng thái Promo** với ngày cần dự báo | Hợp lệ - **baseline chính** |
| `B3` | `dow_promo_mean` theo luật R5 (lag ≥ 49 ngày) | Hợp lệ |
| `X1` | Như B3 nhưng dùng lag 7-49 ngày (lấy cả Sales trong giai đoạn đang dự báo) | **RÒ RỈ** |
| `X2` | Như B2 nhưng profile tính trên **toàn bộ train** (gồm cả giai đoạn validate) | **RÒ RỈ** (vi phạm R6) |

In [ ]:
def profile_pred(T, val_index, cols, n_weeks, use_promo):
    # Dự báo = trung bình log-sales của n_weeks tuần gần nhất (<= T) cùng thứ (và cùng Promo nếu use_promo). Trả về ma trận val x cols.
    LT, PT = logS.loc[:T, cols], Promo.loc[:T, cols]
    pv = Promo.loc[val_index, cols].values
    out = np.full((len(val_index), len(cols)), np.nan)
    for d in range(7):
        hist = LT[LT.index.dayofweek == d].tail(n_weeks); hp = PT.loc[hist.index]
        rows_d = np.where(val_index.dayofweek == d)[0]
        if use_promo:
            for pr in (0, 1):
                mu = hist.where(hp == pr).mean().values
                for r in rows_d: out[r, pv[r] == pr] = mu[pv[r] == pr]
        else:
            mu = hist.mean().values
            for r in rows_d: out[r, :] = mu
    return out

def eval_fold(T):
    v0, v1 = T + pd.Timedelta(days=1), T + pd.Timedelta(days=HORIZON)
    vi = ALL_DATES[(ALL_DATES >= v0) & (ALL_DATES <= v1)]
    Sv, Ov = S.loc[vi, cols], O.loc[vi, cols]
    y = Sv.where((Ov == 1) & (Sv > 0)).values
    preds = {
        'B1 cùng thứ (8 tuần)':        profile_pred(T, vi, cols, 8, False),
        'B2 cùng thứ + cùng Promo':    profile_pred(T, vi, cols, 16, True),
        'B3 dow_promo_mean (lag>=49)': HIST['dow_promo_mean'].loc[vi, cols].values,
    }
    leaky_feat, _ = same_dow_same_promo_mean(logS, Promo, range(7, 57, 7))                  # X1: lag 7..49 -> rò rỉ
    preds['X1 RÒ RỈ lag 7-49'] = leaky_feat.loc[vi, cols].values
    preds['X2 RÒ RỈ profile toàn train'] = profile_pred(TRAIN_END, vi, cols, 10_000, True)  # X2: dùng cả giai đoạn validate
    res = {}
    for k, p in preds.items():
        pe = np.expm1(p); m = ~np.isnan(y) & ~np.isnan(pe)
        res[k] = round(rmspe(y[m], pe[m]), 4)
    return res

floor = pd.DataFrame({name: eval_fold(T) for name, T in FOLD_ORIGINS.items()})
floor['Trung bình 4 fold'] = floor.mean(axis=1).round(4)
floor

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
SCENARIOS_EN = {
    'B1 cùng thứ (8 tuần)': 'B1 DOW Mean (8w)',
    'B2 cùng thứ + cùng Promo': 'B2 DOW + Promo (16w)',
    'B3 dow_promo_mean (lag>=49)': 'B3 dow_promo_mean (Valid)',
    'X1 RÒ RỈ lag 7-49': 'X1 Target Leakage (lag 7-49)',
    'X2 RÒ RỈ profile toàn train': 'X2 Target Leakage (Full Train)'
}
FOLDS_EN = {
    'F1 (gần test nhất)': 'F1 (Near Test)',
    'F2': 'F2 (Spring)',
    'F3 (có lễ Phục sinh)': 'F3 (Easter)',
    'F4 (cùng mùa năm trước)': 'F4 (Prior Year)'
}
plot_df = floor.drop(columns='Trung bình 4 fold', errors='ignore').rename(index=SCENARIOS_EN).rename(columns=FOLDS_EN).T
plot_df.plot(kind='bar', ax=ax, color=['#A6A6A6', '#4472C4', '#2E75B6', '#ED7D31', '#C00000'], width=0.8)
ax.set_ylabel('RMSPE')
ax.set_title('Baseline and Leakage RMSPE Across Folds')
ax.legend(fontsize=8, loc='upper right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Nhận xét (đọc từ bảng và biểu đồ):**
1. **B1 → B2:** chỉ cần "ghép theo trạng thái Promo" là RMSPE trung bình 4 fold giảm rất mạnh (từ ~0.217 xuống ~0.150). Đây là lý do luật **R4.3** (ghép theo Promo) là bắt buộc và là cải tiến lớn nhất trong cả F02.
2. **B2 và B3 gần nhau** (trung bình ~0.150 so với ~0.158) → luật R5 (lag ≥ 49) chỉ **mất ít thông tin** so với dùng profile đến sát mốc T, nghĩa là cách làm "mỗi dòng độc lập, dùng được cho cả train và test" là chấp nhận được.
3. **Rò rỉ làm điểm đẹp hơn bản hợp lệ tương ứng:** X1 so với B3 (cùng công thức, chỉ khác lag) tốt hơn ở **cả 4 fold** (trung bình ~0.147 so với ~0.158; F3: ~0.154 so với ~0.182); X2 so với B2 tốt hơn ở 3/4 fold (trung bình ~0.144 so với ~0.150). Mức "đẹp giả" ở baseline đơn giản chỉ khoảng 0.006-0.011 vì baseline không có nhiều chỗ để ăn gian; với model ML nhiều biến, khoảng cách có thể lớn hơn nhưng **chưa đo**, sẽ đo ở M02. **Luật bắt buộc: biến nào trượt test 5.4 thì loại.**
4. **Mức sàn = B2 ≈ 0.14-0.16 (trung bình ~0.150)** → model ML chỉ có ý nghĩa nếu vượt rõ rệt mức này ở **cả 4 fold** (không chỉ trung bình).

**Kết luận mục 6 → Luật R9 (Validation Strategy):**
- **R9.1** Walk-forward 4 fold như bảng trên; mỗi fold 48 ngày; chấm trên **856 cửa hàng** trong test; metric chính = **RMSPE bỏ qua Sales=0**; báo cáo thêm trung bình và độ lệch giữa các fold.
- **R9.2** Tuyệt đối không K-fold ngẫu nhiên, không xáo trộn theo thời gian.
- **R9.3** Mọi thống kê cửa hàng/target encoding phải tính **trong từng fold** (chỉ từ dữ liệu ≤ T của fold đó).
- **R9.4** Điểm Kaggle (Sales thật của test) chỉ dùng như **kỳ thi cuối**: nộp rất ít lần (1-2 lần), **không tune model theo điểm Kaggle** (nếu tune theo đó thì test trở thành validation và mất giá trị đánh giá khách quan).
- **R9.5** Trước khi nộp: train lại trên **toàn bộ dữ liệu đến 31/07/2015** với tham số đã chốt, rồi dự báo 48 ngày test.
- **R9.6** Mức sàn B2 và các con số ở bảng trên là **benchmark** để M02 so sánh.

---
## 7. R10 - R12: Competition, Promo2 và lịch lễ

### 7.1 Competition (đối thủ cạnh tranh)

In [ ]:
c = store[['Store', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']].copy()
c['comp_open'] = pd.to_datetime(dict(year=c.CompetitionOpenSinceYear, month=c.CompetitionOpenSinceMonth, day=1), errors='coerce')
facts = {
    'CompetitionDistance thiếu (cửa hàng)': c[c.CompetitionDistance.isna()].Store.tolist(),
    'Có khoảng cách nhưng THIẾU ngày đối thủ mở (CH)': int((c.CompetitionDistance.notna() & c.comp_open.isna()).sum()),
    'Năm đối thủ mở < 1990 (nghi placeholder) (CH)': int((c.CompetitionOpenSinceYear < 1990).sum()),
    'Đối thủ mở SAU ngày cuối train (từ 08/2015) (CH)': int((c.comp_open >= TEST_START).sum()),
    'Đối thủ mở trong 12 tháng cuối của train (CH)': int(((c.comp_open >= TEST_START - pd.DateOffset(years=1)) & (c.comp_open < TEST_START)).sum()),
    'CompetitionDistance lớn nhất (m)': float(c.CompetitionDistance.max()),
}
pd.Series(facts).to_frame('giá trị')

In [ ]:
def add_competition_features(df, store):
    s = store[['Store', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']].copy()
    s.loc[s.CompetitionOpenSinceYear < 1990, ['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']] = np.nan   # placeholder -> coi như chưa biết
    s['comp_open'] = pd.to_datetime(dict(year=s.CompetitionOpenSinceYear, month=s.CompetitionOpenSinceMonth, day=1), errors='coerce')
    s['CompetitionDistanceMissing'] = s.CompetitionDistance.isna().astype('int8')
    s['CompetitionDistance'] = s.CompetitionDistance.fillna(s.CompetitionDistance.max()).astype('float32')   # không có đối thủ gần -> coi như xa nhất
    out = df.merge(s[['Store', 'CompetitionDistance', 'CompetitionDistanceMissing', 'comp_open']], on='Store', how='left')
    months = (out.Date.dt.year - out.comp_open.dt.year) * 12 + (out.Date.dt.month - out.comp_open.dt.month)
    out['CompetitionSinceUnknown'] = out.comp_open.isna().astype('int8')
    out['CompetitorNotOpenYet']    = (months < 0).astype('int8')                     # đối thủ chưa mở tại thời điểm của dòng này
    out['MonthsSinceCompetitionOpened'] = months.clip(lower=0).astype('float32')     # NaN giữ nguyên khi chưa biết
    return out.drop(columns='comp_open')

tc = add_competition_features(test[['Store', 'Date']], store)
print('Tỷ lệ dòng có cờ - TEST :', tc[['CompetitionDistanceMissing', 'CompetitionSinceUnknown', 'CompetitorNotOpenYet']].mean().round(4).to_dict())
trc = add_competition_features(train[['Store', 'Date']], store)
print('Tỷ lệ dòng có cờ - TRAIN:', trc[['CompetitionDistanceMissing', 'CompetitionSinceUnknown', 'CompetitorNotOpenYet']].mean().round(4).to_dict())
tc.MonthsSinceCompetitionOpened.describe().round(1)

**Luật R10 (Competition):**
- **R10.1** `MonthsSinceCompetitionOpened` được tính **theo ngày của từng dòng** (không phải 1 số cố định), nên biến đổi theo thời gian với ~17% cửa hàng có đối thủ mới mở trong 2013-2015 (U02 mục 1.13).
- **R10.2** Giá trị âm (đối thủ chưa mở tại ngày của dòng) → kẹp về 0 + cờ `CompetitorNotOpenYet`. Cờ này xuất hiện ở **train** (đối thủ mở sau ngày của dòng) nhưng ở **test = 0%** (xem 2 dòng thống kê trên), nên model chỉ học nó từ train; vẫn cần để không biến giá trị âm thành thông tin sai.
- **R10.3** 351 cửa hàng có khoảng cách nhưng không có ngày mở → để `NaN` + cờ `CompetitionSinceUnknown` (không đoán bừa).
- **R10.4** Năm < 1990 (2 cửa hàng) coi là placeholder → thành `NaN`.
- **R10.5** `CompetitionDistance` thiếu (3 cửa hàng) → điền bằng giá trị lớn nhất + cờ `CompetitionDistanceMissing`.
- Nhắc lại: U02 mục 2.2 cho thấy `CompetitionDistance` gần như không tương quan với Sales, nên đừng kỳ vọng nó là driver chính (xem ở xAI).

### 7.2 Promo2 (khuyến mãi dài hạn): biến `IsPromo2ActiveThisMonth` làm biến dự báo

In [ ]:
MONTHS = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,'Sep':9,'Sept':9,'Oct':10,'Nov':11,'Dec':12}   # lưu ý 'Sept'

def add_promo2_features(df, store):
    s = store[['Store', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']].copy()
    for m in range(1, 13):
        s[f'm{m}'] = s.PromoInterval.fillna('').apply(lambda x: m in {MONTHS[t.strip()] for t in x.split(',') if t.strip()})
    s['p2_start'] = pd.to_datetime(s.Promo2SinceYear.astype('Int64').astype(str) + '-W' + s.Promo2SinceWeek.astype('Int64').astype(str).str.zfill(2) + '-1',
                                   format='%G-W%V-%u', errors='coerce')          # tuần ISO, khớp cách làm ở U02 mục 3.3
    out = df.merge(s.drop(columns=['PromoInterval', 'Promo2SinceWeek', 'Promo2SinceYear']), on='Store', how='left')
    mm = out[[f'm{m}' for m in range(1, 13)]].values
    in_interval = mm[np.arange(len(out)), out.Date.dt.month.values - 1]
    started = (out.Date >= out.p2_start).values
    out['Promo2Started'] = (out.Promo2.eq(1).values & started).astype('int8')
    out['IsPromo2ActiveThisMonth'] = (out.Promo2Started.values.astype(bool) & in_interval).astype('int8')
    out['Promo2WeeksSinceStart'] = np.where(out.Promo2Started.values == 1, ((out.Date - out.p2_start).dt.days // 7).values, np.nan).astype('float32')
    return out.drop(columns=[f'm{m}' for m in range(1, 13)] + ['p2_start'])

# --- đối chiếu độc lập với logic U02 mục 3.3 (so sánh năm-tuần ISO): số dòng khác nhau phải bằng 0
p2_ids = store[store.Promo2 == 1].Store
d = add_promo2_features(train[train.Store.isin(p2_ids)][['Store', 'Date']], store)
iso = d.Date.dt.isocalendar()
sy = d.merge(store[['Store', 'Promo2SinceYear', 'Promo2SinceWeek']], on='Store', how='left')
u02_past = ((iso.year.values > sy.Promo2SinceYear.values) | ((iso.year.values == sy.Promo2SinceYear.values) & (iso.week.values >= sy.Promo2SinceWeek.values)))
print('Đối chiếu với U02 mục 3.3 (Promo2Started):', int((u02_past.astype(int) != d.Promo2Started.values).sum()), 'dòng khác nhau trên', f'{len(d):,}', 'dòng')

# --- test nhanh: cửa hàng chu kỳ 'Jan,Apr,Jul,Oct' chỉ được active vào các tháng 1,4,7,10 (và sau mốc bắt đầu)
chk = add_promo2_features(train[train.Store.isin(store[store.PromoInterval == 'Jan,Apr,Jul,Oct'].Store.head(30))][['Store', 'Date']], store)
act_months = sorted(chk[chk.IsPromo2ActiveThisMonth == 1].Date.dt.month.unique())
print('Các tháng có IsPromo2ActiveThisMonth=1 của nhóm Jan,Apr,Jul,Oct:', act_months, '=> ĐẠT' if act_months == [1, 4, 7, 10] else '=> LỖI')

tp = add_promo2_features(test[['Store', 'Date']], store)
p2s = store[store.Promo2 == 1].copy()
p2s['start'] = pd.to_datetime(p2s.Promo2SinceYear.astype(int).astype(str) + '-W' + p2s.Promo2SinceWeek.astype(int).astype(str).str.zfill(2) + '-1', format='%G-W%V-%u')
print('\nPromo2 bắt đầu sau 01/08/2015:', int((p2s.start >= TEST_START).sum()), '| bắt đầu trong 12 tháng cuối của train:', int(((p2s.start >= TEST_START - pd.DateOffset(years=1)) & (p2s.start < TEST_START)).sum()))
print('Test: số dòng IsPromo2ActiveThisMonth=1 theo tháng:', tp[tp.IsPromo2ActiveThisMonth == 1].Date.dt.month.value_counts().sort_index().to_dict())

**Luật R11 (Promo2):**
- **R11.1** Dùng đúng định nghĩa đã thống nhất của nhóm (U02 mục 3.3): `IsPromo2ActiveThisMonth = 1` khi đồng thời `Promo2 = 1`, đã qua mốc `Promo2SinceWeek/Year` (tuần ISO), và tháng thuộc `PromoInterval` ('Sept' ≠ 'Sep' đã được xử lý bằng bảng ánh xạ).
- **R11.2** Thêm `Promo2Started` và `Promo2WeeksSinceStart` (số tuần kể từ khi tham gia) cho model dự báo.
- **R11.3** Không có cửa hàng nào bắt đầu Promo2 sau cuối train → biến này **đủ ổn định giữa train và test** (không có trường hợp "mới xuất hiện ở test").
- **R11.4 (quan trọng về diễn giải):** trong F02, `IsPromo2ActiveThisMonth` chỉ là **biến dự báo**. Mọi kết luận về "tác động của Promo2" nằm ngoài phạm vi notebook này và **không** suy ra nhân quả từ độ quan trọng biến của model.

### 7.3 Lịch lễ: `StateHoliday` và `SchoolHoliday`

In [ ]:
print('Test: StateHoliday =', test.StateHoliday.value_counts().to_dict(), '| SchoolHoliday = 1:', round(test.SchoolHoliday.mean(), 3))
print(test[test.StateHoliday != '0'].groupby('Date').agg(số_dòng=('Store', 'count'), số_dòng_mở_cửa=('Open', 'sum')))

allcal = pd.concat([train[['Store', 'Date', 'StateHoliday']], test[['Store', 'Date', 'StateHoliday']]])
share = allcal.assign(h=allcal.StateHoliday != '0').groupby('Date').h.mean()
print('\nSố ngày mà StateHoliday khác nhau giữa các cửa hàng (lễ theo bang):', int(((share > 0) & (share < 1)).sum()))
nat = share[share >= 0.5].index
print('Số ngày lễ "toàn chuỗi" (>=50% cửa hàng):', len(nat), '| trong giai đoạn test:', int((nat >= TEST_START).sum()))

# Rule cho biến lịch lễ
def add_holiday_features(df, nat_dates, cap=30):
    d = df.copy(); dates = pd.DatetimeIndex(sorted(nat_dates))
    idx_prev = dates.searchsorted(d.Date.values, side='right') - 1
    last = pd.Series(dates[np.clip(idx_prev, 0, None)], index=d.index).where(idx_prev >= 0)
    d['DaysSinceLastHoliday'] = (d.Date - last).dt.days.clip(upper=cap).astype('float32')
    d['IsDayAfterHoliday']  = d.Date.isin(dates + pd.Timedelta(days=1)).astype('int8')
    d['IsDayBeforeHoliday'] = d.Date.isin(dates - pd.Timedelta(days=1)).astype('int8')
    return d
th = add_holiday_features(test[['Store', 'Date']], nat)
print('\nTest: DaysSinceLastHoliday (cap 30):', th.DaysSinceLastHoliday.describe().round(1).to_dict())

**Nhận xét & luật R12 (lịch lễ):**
- Trong test **chỉ có 1 ngày lễ cấp bang** (15/08/2015, 180 cửa hàng) và **không có ngày lễ toàn chuỗi nào**; hệ quả: `DaysSinceLastHoliday` = 30 (mức cap) cho **mọi** dòng test và 2 cờ `IsDay...Holiday` = 0 → các biến này **không mang thông tin gì ở test**, nhưng vẫn cần cho train (nếu không model sẽ nhầm các ngày lễ trong quá khứ là ngày thường).
- **R12.1** `StateHoliday` dùng nguyên giá trị theo từng cửa hàng (lễ theo bang nên khác nhau giữa các cửa hàng ở ~vài ngày), **không** suy ra từ ngày đơn thuần.
- **R12.2** Biến khoảng cách ngày lễ (`DaysSinceLastHoliday`, `IsDayBeforeHoliday`, `IsDayAfterHoliday`) tính từ **lịch lễ toàn chuỗi** (ngày có ≥ 50% cửa hàng nghỉ lễ), cap 30 ngày. Dùng được vì lịch lễ là thông tin biết trước. Không dùng "số ngày tới lễ tiếp theo" vì ở cuối test không thể biết lễ sau đó từ dữ liệu.
- **R12.3** F2, F3 có ngày lễ còn test chỉ có 1 ngày lễ cấp bang → báo cáo thêm RMSPE **loại bỏ ngày lễ** ở mọi fold để so sánh công bằng (F1, F4 vốn không có ngày lễ).
- **R12.4** `SchoolHoliday` giữ nguyên (44% dòng test = 1). U02 mục 2.5 cho con số gộp +4.4% nhưng cảnh báo hiệu ứng có thể khác nhau giữa các cửa hàng, nên **đừng diễn giải là tăng đều** → để xAI kiểm tra theo `StoreType`.

---
## 8. R13 - Biến mục tiêu `log(Sales+1)` và hậu xử lý

**Vì sao dùng log?** RMSPE đo sai số theo **phần trăm**: lệch 500€ ở cửa hàng bán 2.000€ là sai 25%, gấp 5 lần sai 5% khi lệch 500€ ở cửa hàng bán 10.000€ (U02 mục 1.14), và RMSPE còn bình phương sai số nên chênh lệch càng lớn. Khi huấn luyện trên `log(Sales+1)`, sai số log xấp xỉ sai số phần trăm, nên tối ưu MSE trên log ≈ tối ưu RMSPE; đồng thời Sales hết lệch phải (skew +1.59 → -0.11 sau khi loại 54 dòng Sales=0).

In [ ]:
# (1) Minh họa: sai số log ≈ sai số phần trăm khi sai số nhỏ
y = np.array([2000., 2000., 10000., 10000.]); yhat = np.array([1500., 2500., 9500., 10500.])
print(pd.DataFrame({'y': y, 'ŷ': yhat, 'sai số %': ((y-yhat)/y).round(3), 'sai số log': (np.log(y)-np.log(yhat)).round(3)}))

# (2) Hàm hậu xử lý CHUẨN để dùng cho mọi model (R3 + R13)
def postprocess(pred_log, open_flag, c=1.0):
    # pred_log: dự báo trên thang log1p | open_flag: cột Open của test | c: hệ số hiệu chỉnh (mặc định 1.0, tune ở M02)
    pred = np.expm1(np.asarray(pred_log, float)) * c
    open_flag = np.where(pd.isna(open_flag), 1, open_flag)         # R3.2: Open thiếu -> 1
    pred = np.where(open_flag == 1, pred, 0.0)                      # R3.1: đóng cửa -> 0
    return np.clip(pred, 0, None)

# (3) Unit test nhỏ
out = postprocess([np.log1p(1000), np.log1p(1000), np.log1p(1000)], [1, 0, np.nan])
assert np.allclose(out, [1000, 0, 1000]), out
print('postprocess OK:', out)

# (4) Định dạng file nộp: Id, Sales (đúng thứ tự Id của test)
sub_tmpl = pd.read_csv(find('sample_submission_ross.csv', 'sample_submission.csv'))
assert list(sub_tmpl.columns) == ['Id', 'Sales'] and set(sub_tmpl.Id) == set(test.Id)
def make_submission(ids, pred): return pd.DataFrame({'Id': ids, 'Sales': pred}).sort_values('Id').reset_index(drop=True)
print('Định dạng nộp bài hợp lệ:', sub_tmpl.shape, '| Id test khớp sample_submission: True')

**Luật R13:**
- **R13.1** Train trên `y = log1p(Sales)` (chỉ dòng thỏa R2).
- **R13.2** Dự báo: `Sales_hat = expm1(ŷ) × c`, trong đó `c` là hệ số hiệu chỉnh nhỏ (khoảng 0.97-1.02) vì trung bình trên thang log thường lệch nhẹ so với thang gốc; **chỉ tune `c` trên các fold validation** (1 tham số nên ít rủi ro overfit).
- **R13.3** Gán `Sales_hat = 0` khi `Open = 0`; `Open` thiếu → 1 (hàm `postprocess` ở trên là nơi duy nhất làm việc này, dùng chung cho validation và nộp Kaggle để không lệch).
- **R13.4** Metric: `rmspe()` ở mục 6 (bỏ dòng Sales=0), thang gốc (không phải log).

---
## 9. Tổng hợp: kiểm tra nhất quán train-test, bảng luật, đặc tả biến, folds → lưu CSV

### 9.1 Nhất quán kiểu dữ liệu giữa train và test (F04 phải làm)
Lỗi hay gặp: `StateHoliday` trong test chỉ có {'0','a'} còn train có {'0','a','b','c'} → nếu mã hóa độc lập, 'a' ở train và test có thể ra 2 con số khác nhau. **Luật: khai báo sẵn danh sách mức (category) cố định cho cả hai.**

In [ ]:
CATS = {'StateHoliday': ['0', 'a', 'b', 'c'], 'StoreType': list('abcd'), 'Assortment': list('abc')}
def align_categories(df):
    df = df.copy()
    for col, levels in CATS.items():
        if col in df: df[col] = pd.Categorical(df[col], categories=levels)
    return df

print('StateHoliday mức xuất hiện - train:', sorted(train.StateHoliday.unique()), '| test:', sorted(test.StateHoliday.unique()))
a, b = align_categories(train[['StateHoliday']]), align_categories(test[['StateHoliday']])
print('Mã số của mức "a" - train:', int(a.StateHoliday.cat.codes[a.StateHoliday == 'a'].iloc[0]), '| test:', int(b.StateHoliday.cat.codes[b.StateHoliday == 'a'].iloc[0]), '(phải bằng nhau)')

In [ ]:
RULES = pd.DataFrame([
 ('R1',  'Biến',        'Chỉ dùng cột có trong test/store; Customers bị chặn (chỉ được dùng dạng lịch sử ≥48 ngày, tùy chọn); Sales chỉ làm target và nguyên liệu cho biến lịch sử', 'Mục 2: Customers và Sales không có trong test', 'F04, M02'),
 ('R2',  'Dòng train',  'Chỉ train trên Open=1 và Sales>0 (loại Open=0 và 54 dòng Open=1&Sales=0)', 'Mục 3', 'F04'),
 ('R3',  'Open',        'Open=0 -> Sales=0; 11 dòng Open thiếu (Store 622) -> Open=1', 'Mục 3: Store 622 mở 94-100% T2-T7; đoán 0 nhầm làm RMSPE tăng ~0.001-0.002', 'Hậu xử lý'),
 ('R4',  'Promo',       'Dùng nguyên lịch Promo của test (chỉ T2-T6); tạo IsPromoWeek/Promo_prev7/Promo_next7; mọi biến lịch sử ghép theo trạng thái Promo', 'Mục 4 và 6: ghép Promo giảm RMSPE baseline từ ~0.2 xuống ~0.15', 'F04'),
 ('R5',  'Lag/rolling', 'Biến lịch sử chỉ dùng Sales của ngày <= t-48 (lag >= 48, cửa sổ rolling kết thúc ở t-48); bắt buộc qua leakage_test', 'Mục 5.4: các biến R5 ĐẠT, biến lag1/lag7 bị bắt RÒ RỈ', 'F04'),
 ('R6',  'Profile CH',  'Mọi thống kê theo cửa hàng/target encoding chỉ tính từ dữ liệu <= mốc cắt của fold; cấm tính trên toàn bộ train', 'Mục 6: X2 (profile toàn train) cho điểm đẹp giả', 'F04, M02'),
 ('R7',  'Dữ liệu thiếu','180 CH mất 01/07-31/12/2014: giữ NaN + cờ has_ly364, thêm ly728 làm dự phòng; GB nhận NaN, RF phải điền (roll91 rồi trung bình StoreType x Assortment)', 'Mục 5.5: 180/181 CH thiếu đều nằm trong test (21%)', 'F04, M02'),
 ('R8',  'Warm-up',     'Loại các ngày đầu train chưa đủ lịch sử khỏi tập train', 'Mục 5.5', 'F04'),
 ('R9',  'Validation',  'Walk-forward 4 fold x 48 ngày (3 fold gần + 1 fold cùng mùa 2014); chấm trên 856 CH; RMSPE bỏ Sales=0; không K-fold ngẫu nhiên; Kaggle chỉ là kỳ thi cuối, không tune theo Kaggle', 'Mục 6', 'M02, E02'),
 ('R10', 'Competition', 'MonthsSinceCompetitionOpened tính theo ngày dòng; âm -> 0 + cờ; thiếu -> NaN + cờ; năm<1990 -> NaN; thiếu Distance -> max + cờ', 'Mục 7.1: 351 CH thiếu ngày mở, 3 CH thiếu Distance, 3 CH đối thủ mở sau train', 'F04'),
 ('R11', 'Promo2',      'IsPromo2ActiveThisMonth theo định nghĩa thống nhất ở U02 mục 3.3 (tuần ISO, ánh xạ Sept); thêm Promo2Started, Promo2WeeksSinceStart; chỉ là biến dự báo, không suy nhân quả', 'Mục 7.2: không CH nào bắt đầu Promo2 sau train', 'F04, M03'),
 ('R12', 'Lịch lễ',     'StateHoliday theo từng CH; khoảng cách lễ tính từ lịch lễ toàn chuỗi (cap 30); báo cáo thêm RMSPE bỏ ngày lễ', 'Mục 7.3: test chỉ có 1 ngày lễ cấp bang (180 CH), 0 lễ toàn chuỗi; F1, F4 không có ngày lễ', 'F04, E02'),
 ('R13', 'Target',       'Train trên log1p(Sales); Sales_hat = expm1(ŷ) x c (c tune trên fold); đóng cửa -> 0; dùng chung hàm postprocess', 'Mục 8', 'M02, E02'),
 ('R14', 'Nhất quán',   'Train và test dùng CÙNG hàm tạo biến, CÙNG danh sách category, CÙNG thứ tự cột; kiểm tra tỷ lệ NaN từng cột giữa train và test', 'Mục 9.1', 'F04'),
 ('R15', 'Đóng cửa dài', 'Từ lịch Open (biết trước ở cả test): DaysSinceReopen = số ngày kể từ lần mở lại sau đợt đóng cửa liên tục >= 10 ngày (cap 120); LastClosureLen = độ dài đợt đóng cửa đó. Chỉ nhìn ngược về quá khứ, không nhìn trước. Giữ lại hay không do kiểm tra A/B ở mục 12.2', 'U02 mục 1.7: 72 CH từng đóng liên tục >= 10 ngày (tối đa 185 ngày); doanh số sau mở lại có thể lệch mạnh', 'F04, M02'),
 ('R16', 'Mùa vụ chuỗi', 'chain_delta = chênh lệch log-sales trung bình toàn chuỗi giữa tuần cùng kỳ năm trước và cửa sổ 91 ngày dùng làm mốc (cùng kỳ năm trước); chỉ dùng Sales <= t-361. Giữ lại hay không do kiểm tra A/B ở mục 12.2', 'U02 mục 3.5: tháng 12 cao (1.24), tháng 1-2 và 8-10 thấp (~0.94-0.96)', 'F04, M02'),
], columns=['Mã', 'Nhóm', 'Luật', 'Bằng chứng (từ notebook này)', 'Áp dụng ở task'])
RULES

In [ ]:
SPEC = pd.DataFrame([
 # feature, nhóm, lúc dự báo, cách tính / luật, dtype, xử lý NaN, RF, GB
 ('Store',                        'Định danh', 'Biết trước', 'Mã cửa hàng. GB dùng như category; RF KHÔNG one-hot 1.115 mức mà dùng các biến profile bên dưới', 'category (int16)', 'không có NaN', 'Không', 'Có'),
 ('DayOfWeek',                    'Lịch', 'Biết trước', '1-7', 'int8', 'không có NaN', 'Có', 'Có'),
 ('Month, WeekOfYear(ISO), DayOfMonth, DaysToMonthEnd', 'Lịch', 'Biết trước', 'Tách từ Date. KHÔNG dùng Year hoặc Date thô (cây không ngoại suy được sang giá trị mới)', 'int8', 'không có NaN', 'Có', 'Có'),
 ('Promo',                        'Khuyến mãi', 'Biết trước', 'Nguyên từ test (R4.1), chỉ T2-T6', 'int8', 'không có NaN', 'Có', 'Có'),
 ('IsPromoWeek, Promo_prev7, Promo_next7', 'Khuyến mãi', 'Biết trước', 'Từ lịch Promo (R4.2); Promo_next7 ở cuối test = -1', 'int8', 'điền -1 ở biên', 'Có', 'Có'),
 ('StateHoliday',                 'Lịch lễ', 'Biết trước', "Mức cố định ['0','a','b','c'] -> mã 0-3 (R14)", 'category (int8)', 'không có NaN', 'Có', 'Có'),
 ('SchoolHoliday',                'Lịch lễ', 'Biết trước', 'Nguyên từ test', 'int8', 'không có NaN', 'Có', 'Có'),
 ('DaysSinceLastHoliday, IsDayBeforeHoliday, IsDayAfterHoliday', 'Lịch lễ', 'Biết trước', 'Từ lịch lễ toàn chuỗi, cap 30 ngày (R12.2)', 'float32 / int8', 'không có NaN', 'Có', 'Có'),
 ('StoreType, Assortment',        'Đặc điểm CH', 'Biết trước', 'Mức cố định a-d, a-c (R14). Nhóm b rất nhỏ (n=17 và n=9): chỉ kết luận kèm n', 'category (int8)', 'không có NaN', 'Có', 'Có'),
 ('CompetitionDistance, CompetitionDistanceMissing', 'Đối thủ', 'Biết trước', 'Thiếu -> giá trị lớn nhất + cờ (R10.5)', 'float32 / int8', 'đã điền', 'Có', 'Có'),
 ('MonthsSinceCompetitionOpened, CompetitionSinceUnknown, CompetitorNotOpenYet', 'Đối thủ', 'Biết trước', 'Tính theo ngày dòng; âm -> 0 + cờ; thiếu -> NaN + cờ (R10)', 'float32 / int8', 'NaN (RF: điền -1 + giữ cờ)', 'Có', 'Có'),
 ('Promo2, Promo2Started, IsPromo2ActiveThisMonth', 'Promo2', 'Biết trước', 'Định nghĩa U02 mục 3.3 (R11)', 'int8', 'không có NaN', 'Có', 'Có'),
 ('Promo2WeeksSinceStart',        'Promo2', 'Biết trước', 'Số tuần kể từ mốc tham gia; NaN nếu chưa/không tham gia', 'float32', 'NaN (RF: điền -1)', 'Có', 'Có'),
 ('DaysSinceReopen, LastClosureLen', 'Đóng cửa dài (R15)', 'Biết trước (từ lịch Open)', 'Số ngày kể từ lần mở lại sau đợt đóng cửa >= 10 ngày liên tục (cap 120) và độ dài đợt đó (cap 200)', 'float32', 'không có NaN (không có đợt đóng cửa -> 120 / 0)', 'Tùy (A/B)', 'Tùy (A/B)'),
 ('chain_delta',                  'Mùa vụ chuỗi (R16)', 'Quá khứ >= 361 ngày', 'Log-sales TB toàn chuỗi (panel cố định) của tuần cùng kỳ năm trước trừ cửa sổ mốc 91 ngày cùng kỳ năm trước', 'float32', 'NaN trước 05/2014 (RF: điền 0)', 'Tùy (A/B)', 'Tùy (A/B)'),
 ('dow_promo_mean, dow_promo_n',  'Lịch sử Sales', 'Quá khứ >= 49 ngày', 'Trung bình log-sales cùng thứ và cùng Promo, lag 49-119 (R5); dow_promo_n = số mẫu', 'float32 / int8', 'NaN (RF: điền roll91 -> trung bình StoreType x Assortment)', 'Có', 'Có'),
 ('roll28, roll91, open28',       'Lịch sử Sales', 'Quá khứ >= 48 ngày', 'Rolling ngày mở cửa, cửa sổ kết thúc ở t-48; open28 = số ngày mở trong 28 ngày đó', 'float32 / int8', 'NaN (RF: như trên)', 'Có', 'Có'),
 ('promo_uplift91',               'Lịch sử Sales', 'Quá khứ >= 48 ngày', 'Mức tăng (log) khi có Promo so với không Promo của riêng cửa hàng', 'float32', 'NaN (RF: điền 0)', 'Có', 'Có'),
 ('ly364, ly728, has_ly364',      'Lịch sử Sales', 'Quá khứ >= 364 ngày', 'Log-sales cùng thứ 1 và 2 năm trước; has_ly364 = ly364 có giá trị (R7)', 'float32 / int8', 'NaN + cờ (RF: điền dow_promo_mean)', 'Có', 'Có'),
 ('(tùy chọn) cust_roll28',       'Lịch sử Customers', 'Quá khứ >= 48 ngày', 'Trung bình Customers kết thúc ở t-48 (R1.2); chỉ thêm nếu cải thiện validation', 'float32', 'NaN', 'Tùy', 'Tùy'),
 ('Customers (trực tiếp)',        'BỊ CHẶN', 'KHÔNG có lúc dự báo', 'Không có trong test (R1.2)', '-', '-', 'Không', 'Không'),
 ('Sales (thô), Id',              'BỊ CHẶN', 'Target / mã nộp bài', 'Sales chỉ là target; Id chỉ để nộp bài', '-', '-', 'Không', 'Không'),
], columns=['Feature', 'Nhóm', 'Có sẵn lúc dự báo?', 'Cách tính / luật', 'Dtype đề xuất', 'Xử lý NaN', 'Dùng cho RF', 'Dùng cho GB'])
SPEC

### 9.2 Đối chiếu: các phát hiện ở bước EDA (U02) đã được đưa vào luật chưa?
Mình rà lại **toàn bộ kết luận của U02** và đối chiếu từng mục với luật trong notebook này. Cột *Trạng thái* cho biết mục nào đã có, mục nào **thiếu ở bản trước và được bổ sung ở bản này**, mục nào không thuộc phạm vi dự báo.

In [ ]:
EDA_MAP = pd.DataFrame([
 ('1.1',  'Missing: Open test thiếu 11 dòng; Competition thiếu; Promo2 thiếu có cấu trúc', 'R3.2, R10, R11', 'Đã có'),
 ('1.2',  'StateHoliday: 0 (số) và "0" (chuỗi) bị tách nhóm', 'Ép chuỗi lúc đọc + danh sách category cố định (R14)', 'Đã có'),
 ('1.3',  'Open=0 -> Sales=0; 54 dòng Open=1 & Sales=0', 'R2, R3.1', 'Đã có'),
 ('1.4',  'PromoInterval chỉ có 3 chu kỳ; "Sept" khác "Sep"', 'R11 (bảng ánh xạ tháng)', 'Đã có'),
 ('1.5/2.3', 'StoreType b (17 CH) và Assortment b (9 CH) quá nhỏ', 'Ghi n khi báo cáo; bảng sai số theo nhóm ở E02', 'Đã có'),
 ('1.6',  'test chỉ có 856/1.115 cửa hàng', 'R9: chỉ chấm trên 856 CH; train dùng tất cả CH', 'Đã có'),
 ('1.7',  '72 CH từng đóng cửa liên tục >= 10 ngày (tối đa 185 ngày); doanh số sau mở lại và trung bình cuốn bị méo', 'Trước đây chỉ có open28 và cờ báo cáo ở E02', 'THIẾU -> bổ sung R15 (DaysSinceReopen, LastClosureLen), kiểm tra A/B ở 12.2'),
 ('1.8',  'CompetitionOpenSinceYear có giá trị 1900, 1961', 'R10.4 (năm < 1990 -> NaN)', 'Đã có'),
 ('1.9',  'Không trùng lặp; (Store, Date) là khóa duy nhất', 'Thêm assert kiểm tra khóa duy nhất trong F04 (mục 11.1)', 'Đã bổ sung kiểm tra'),
 ('1.10', '181 CH thiếu ngày liên tục', 'R7 (đã sửa: mất 01/07-31/12/2014, không phải bắt đầu muộn)', 'Đã có (và đã sửa nhận định của U02)'),
 ('1.11', 'Promo chỉ chạy T2-T6', 'R4.1', 'Đã có'),
 ('1.12', '11 dòng Open thiếu đều là Store 622 -> Open=1', 'R3.2', 'Đã có'),
 ('1.13', '191 CH (17,1%) có đối thủ mới mở trong giai đoạn train', 'R10: MonthsSinceCompetitionOpened tính theo ngày dòng', 'Đã có'),
 ('1.14', 'Sales lệch phải; RMSPE; log và gán 0 khi đóng cửa', 'R13', 'Đã có'),
 ('1.15', '225 CH tham gia Promo2 trong giai đoạn train', 'Chỉ phục vụ phân tích Promo2, không dùng cho dự báo', 'Không áp dụng (phân tích Promo2 đã bỏ khỏi notebook)'),
 ('2.1',  'Promo +38%; nguy cơ mua dồn / dịch chuyển nhu cầu (forward-buying)', 'R4.2: Promo_prev7, Promo_next7, IsPromoWeek', 'Đã có'),
 ('2.2',  'CompetitionDistance gần như không liên quan Sales (r ~ -0,04)', 'Giữ làm biến phụ, không kỳ vọng nhiều', 'Đã có'),
 ('2.4',  'Chủ nhật hiếm khi mở nhưng khi mở thì doanh số cao; Thứ 7 thấp nhất', 'DayOfWeek + biến lịch sử cùng thứ; báo cáo riêng nhóm Chủ nhật ở E02', 'Đã có'),
 ('2.5',  'SchoolHoliday +4,4% chỉ là số gộp', 'R12.4', 'Đã có'),
 ('2.6',  'StateHoliday: nghịch lý Simpson (StoreType b tăng, a/d giảm)', 'R12; mô hình cây tự học tương tác; E02 báo cáo riêng, đây vẫn là điểm yếu', 'Đã có (còn hạn chế)'),
 ('2.7',  'Customers không có trong test', 'R1.2', 'Đã có'),
 ('3.5',  'Mùa vụ thị trường rõ (tháng 12 cao; tháng 1-2 và 8-10 thấp)', 'Trước đây chỉ có Month, WeekOfYear, ly364', 'THIẾU -> bổ sung R16 (chain_delta), kiểm tra A/B ở 12.2'),
 ('3.1-3.4, 3.6', 'Điều tra Promo2 (selection bias, within-store, mùa vụ)', 'Thuộc phân tích Promo2', 'Không áp dụng (đã bỏ khỏi notebook)'),
 ('4',    'Root cause 2: cần mô hình dự báo tập trung, walk-forward, kiểm soát leakage, đúng phạm vi 856 CH', 'R5, R6, R9', 'Đã có'),
], columns=['Mục U02', 'Phát hiện ở EDA', 'Xử lý trong pipeline', 'Trạng thái'])
EDA_MAP

In [ ]:
RULES.to_csv(OUT_DIR / 'F02_rules.csv', index=False, encoding='utf-8-sig')
SPEC.to_csv(OUT_DIR / 'F02_feature_spec.csv', index=False, encoding='utf-8-sig')
folds_df.merge(floor.drop(columns='Trung bình 4 fold').T.reset_index().rename(columns={'index': 'Fold'}), on='Fold').to_csv(OUT_DIR / 'F02_validation_folds_and_baseline_floor.csv', index=False, encoding='utf-8-sig')
EDA_MAP.to_csv(OUT_DIR / 'F02_eda_to_rules_map.csv', index=False, encoding='utf-8-sig')
print('Đã lưu: F02_rules.csv | F02_feature_spec.csv | F02_validation_folds_and_baseline_floor.csv | F02_eda_to_rules_map.csv  ->', OUT_DIR.resolve())

---
# 10. Kế hoạch pipeline hoàn chỉnh cho Forecasting (F04 → M02 → E02 → M03 → E03 → D02)
> **Cập nhật:** F04, M02, E02 đã được **triển khai bằng code ở phần B (mục 11-14)**. Mục 10 giữ lại làm bản mô tả kế hoạch; xAI (M03, E03) đã được triển khai ở **phần D (mục 15-16)**; D02 **chưa làm**.

## 10.0 Bức tranh tổng thể
```
train.csv / test.csv / store.csv
        │
        ▼
[F04] Dựng bảng dữ liệu cho model   (áp luật R1-R14, chạy leakage_test, kiểm tra nhất quán train-test)
        │
        ▼
[M02a] Baseline B1, B2, B3          (mức sàn, đã có số ở mục 6)
        │
        ▼
[M02b] Model ML: Linear (log), Random Forest, Gradient Boosting   (CÙNG bộ biến, CÙNG 4 fold)
        │
        ▼
[M02c] Tuning (random search, chấm bằng trung bình 4 fold) + hệ số c
        │
        ▼
[E02] Đánh giá: RMSPE từng fold, theo loại cửa hàng / thứ / ngày dự báo thứ h; phân tích lỗi; nộp Kaggle 1-2 lần
        │
        ▼
[M03] xAI: importance, SHAP, PDP/ICE  →  [E03] kiểm tra độ tin cậy xAI  →  [D02] Business memo (dùng dự báo làm gì)
```

## 10.1 F04: Input, kiểu dữ liệu và thứ tự xử lý
**Input:** `train.csv` (1.017.209 dòng), `test.csv` (41.088 dòng), `store.csv` (1.115 dòng). Kiểu dữ liệu của **từng cột đầu ra** nằm trong `F02_feature_spec.csv` (cột *Dtype đề xuất*): `Date` là datetime64, mã/cờ là `int8`, biến liên tục là `float32`, biến phân loại có **danh sách mức cố định**.

**Thứ tự làm (quan trọng, đừng đảo):**
1. Đọc 3 file, ép `StateHoliday` thành chuỗi, parse `Date`.
2. Tạo biến cố định theo cửa hàng và theo ngày: Competition (R10), Promo2 (R11), lịch (R4.2, R12). **Dùng đúng 1 hàm cho cả train và test.**
3. Dựng ma trận ngày x cửa hàng và biến lịch sử (R5). **Chạy `leakage_test` - phải ĐẠT hết mới đi tiếp.**
4. Ghép biến lịch sử vào từng dòng theo cặp `(Date, Store)`.
5. Lọc dòng train theo R2 và bỏ giai đoạn warm-up (R8). Tạo `y_log = log1p(Sales)`.
6. Ở test: `Open` thiếu → 1 (R3.2), giữ nguyên các dòng `Open = 0` (để hậu xử lý gán 0), tạo biến bằng đúng hàm ở bước 2-4.
7. **Kiểm tra nhất quán train-test (R14):** cùng tên cột và thứ tự cột; cùng dtype; cùng danh sách category; so sánh tỷ lệ NaN từng cột giữa train và test; **không có** `Customers`, `Sales` thô trong danh sách biến.
8. Lưu theo convention của nhóm, ví dụ `model_table_train_F04.csv`, `model_table_test_F04.csv` (không xóa bản cũ).

## 10.2 M02: Chọn model và benchmark ("bậc thang" từ đơn giản đến phức tạp)
Nguyên tắc: **mỗi bậc phải thắng bậc dưới thì mới có lý do lên bậc trên.** Nếu model phức tạp không thắng baseline thì quay lại kiểm tra dữ liệu/biến, chứ không nên tin ngay.

| Bậc | Model | Vai trò | Vì sao hợp lý | Lưu ý |
|---|---|---|---|---|
| B1, B2, B3 | Trung bình cùng thứ / cùng thứ + Promo / `dow_promo_mean` | Mức sàn | Không cần học; đã có số ở mục 6 | B2 ≈ 0.14-0.16 (trung bình ~0.150) là mốc phải vượt |
| L1 | Linear Regression hoặc Ridge trên `y_log` | "Thước đo độ phức tạp" | Nhanh, dễ diễn giải; cho biết quan hệ có gần tuyến tính không | Cần one-hot các biến phân loại **nhỏ** (StoreType, Assortment, StateHoliday, DayOfWeek); không one-hot `Store` |
| M1 | Random Forest (`RandomForestRegressor`) | Model phi tuyến thứ nhất, ít nhạy tham số | Bắt tương tác (Promo x thứ x loại cửa hàng) mà không cần tự tạo | Chậm với ~0,8 triệu dòng: dùng `max_samples` 0.2-0.4, `min_samples_leaf` 10-50, `n_jobs=-1`; điền NaN trước (bản sklearn cũ) |
| M2 | Gradient Boosting (LightGBM hoặc XGBoost) | Model chính dự kiến | Thường tốt nhất trên bảng dữ liệu có nhiều biến loại này; nhận NaN trực tiếp; SHAP hỗ trợ tốt | Cần tune; dễ overfit nếu không early stopping |

**Vì sao KHÔNG chọn:** ARIMA/Prophet cho từng cửa hàng (hơn 800 chuỗi riêng, khó đưa vào lịch Promo/lễ/đối thủ, mỗi chuỗi chỉ ~940 điểm); mạng nơ-ron/LSTM (không cần cho bài này, khó giải thích, vượt phạm vi môn). **Vì sao 1 model chung cho mọi cửa hàng:** học cấu trúc chung (thứ, Promo, mùa) từ cả chục nghìn ngày-cửa hàng; phần riêng của từng cửa hàng đã được các biến profile (`dow_promo_mean`, `roll91`, `ly364`...) đảm nhiệm.

**Kiểm tra "có hợp lý không" (checklist M02):**
1. Thắng **B2 ở cả 4 fold** (không chỉ trung bình). Nếu chỉ thắng 1-2 fold → chưa đáng tin.
2. Nếu RMSPE tụt xuống **thấp hơn B2 quá nhiều** (ví dụ giảm hơn một nửa) thì **nghi rò rỉ trước khi ăn mừng**: chạy lại `leakage_test`, kiểm tra không có `Customers` hay biến lag ngắn trong danh sách biến.
3. Sai số **không tăng vọt theo ngày dự báo thứ h** (h = 1..48) - vì biến theo R5 không phụ thuộc h nên đường sai số theo h nên khá phẳng.
4. **Bias ≈ 0:** tổng dự báo của từng fold không lệch hệ thống so với tổng thực tế (nếu lệch → dùng hệ số `c`).
5. Không có dự báo âm hoặc bất thường (đã chặn bằng `postprocess`).

## 10.3 Cách chạy validation (walk-forward) - khung chương trình
```python
for fold, T in FOLD_ORIGINS.items():                       # 4 fold ở mục 6
    tr = table[table.Date <= T]                            # chỉ dữ liệu đến mốc T (đã qua lọc R2, R8)
    va = table[(table.Date > T) & (table.Date <= T + 48 ngày) & table.Store.isin(test_stores)]   # gồm cả dòng Open=0 để hậu xử lý
    # Biến lịch sử đã đúng luật R5 nên không cần tính lại. Riêng bất kỳ target encoding / profile mới nào PHẢI fit trên `tr` (R6).
    model.fit(tr[FEATURES], tr['y_log'])
    pred  = postprocess(model.predict(va[FEATURES]), va['Open'], c)
    score[fold] = rmspe(va['Sales'], pred)                 # thang gốc, bỏ Sales=0
report(score)    # từng fold + trung bình + độ lệch chuẩn + điểm "bỏ ngày lễ"
```
**Trước khi nộp Kaggle:** train lại trên **toàn bộ dữ liệu đến 31/07/2015** với tham số đã chốt, dự báo 48 ngày test, `make_submission(test.Id, pred)`. Điểm Kaggle (Sales thật của test) là **kỳ thi cuối**: nộp 1-2 lần để kiểm tra, **không dùng để tune**. (Nếu cuộc thi chỉ nhận late submission thì vẫn nộp theo cách đó; điểm và cách tách public/private tùy cấu hình hiện tại trên Kaggle, nhóm nên xem trực tiếp trên trang cuộc thi.)

## 10.4 Tuning (chỉnh tham số) - làm gọn, tránh overfit vào fold
| Bước | Việc làm |
|---|---|
| 1 | Chạy **tham số mặc định** cho L1, M1, M2 trên 4 fold → ghi lại bảng điểm gốc |
| 2 | **Random search** 20-30 tổ hợp (tune trên mẫu ~300 cửa hàng cho nhanh, rồi kiểm lại bằng toàn bộ cửa hàng); điểm chọn = **trung bình RMSPE 4 fold** |
| 3 | Lấy top 3 cấu hình, xem **độ lệch chuẩn giữa các fold** và so từng fold với B2. Chọn cấu hình **đơn giản nhất** trong nhóm sai khác không đáng kể |
| 4 | Tune hệ số `c` (0.97-1.02, bước 0.005) trên 4 fold |
| 5 | Train lại toàn bộ dữ liệu → dự báo test → nộp Kaggle |

Tham số gợi ý: **Gradient Boosting** (LightGBM): `num_leaves` 31-255, `learning_rate` 0.03-0.1, `min_child_samples` 20-200, `feature_fraction` và `bagging_fraction` 0.6-1.0, `lambda_l2` 0-10, số cây chọn bằng early stopping. **Random Forest:** `n_estimators` 200-400, `min_samples_leaf` 5-50, `max_features` 0.3-0.7, `max_samples` 0.2-0.5. Lưu ý: dù train bằng MSE trên log, **luôn chọn tham số theo RMSPE thang gốc**.

## 10.5 E02: Đánh giá (ngoài điểm tổng)
- RMSPE **từng fold**, trung bình, độ lệch chuẩn; điểm "bỏ ngày lễ".
- RMSPE **theo `StoreType` / `Assortment` / thứ trong tuần / Promo vs không Promo / ngày dự báo thứ h** (tìm chỗ model yếu). Nhóm `b` nhỏ (n=17) → ghi rõ n.
- **Nhóm cửa hàng rủi ro:** 180 CH thiếu dữ liệu 2014 (không có `ly364`), cửa hàng vừa mở lại sau đóng cửa dài ngày, cửa hàng có đối thủ mới mở: sai số có cao hơn không?
- **Danh sách 20 cửa hàng sai nhiều nhất** và nguyên nhân (đóng cửa dài, dữ liệu thiếu, outlier).
- **Bias** và biểu đồ dự báo vs thực tế cho vài cửa hàng đại diện.

## 10.6 M03: xAI - làm gì, hỏi câu gì, cẩn thận điều gì
| Kỹ thuật | Trả lời câu hỏi nào | Cách làm | Cẩn thận |
|---|---|---|---|
| **Permutation importance theo nhóm biến** | Nhóm biến nào làm tăng RMSPE nhiều nhất nếu bị xáo trộn? | Trên 1 fold validation, xáo từng **nhóm** (Promo+DayOfWeek; Lịch sử Sales; Competition; Promo2; Lịch lễ) và đo ΔRMSPE | Biến tương quan (Promo, DayOfWeek, Open) chia nhau importance → xáo theo nhóm |
| **SHAP (TreeSHAP)** summary/beeswarm | Biến nào đẩy dự báo lên/xuống và theo chiều nào? | Chạy trên **mẫu 20-50k dòng của fold validation** (không dùng train), cho GB và RF | SHAP giải thích **model**, không phải nguyên nhân thật (thầy đã nói dataset này không đủ cho nhân quả) |
| **SHAP dependence theo `StoreType`** | `StateHoliday`, `SchoolHoliday`, `Promo` có hiệu ứng khác nhau theo loại cửa hàng không? | Vẽ riêng từng `StoreType` | Nghịch lý Simpson ở `StateHoliday` (U02 mục 2.6); `b` chỉ n=17 |
| **PDP / ICE** | Dự báo thay đổi thế nào khi chỉ đổi 1 biến (Promo, `MonthsSinceCompetitionOpened`, `CompetitionDistance`)? | PDP cho xu hướng chung, ICE để thấy sự khác nhau giữa các cửa hàng | Biến tương quan mạnh có thể tạo tổ hợp phi thực tế |
| **Giải thích cục bộ (waterfall)** | Vì sao ngày-cửa hàng này được dự báo như vậy? | Chọn 3 dòng: 1 điển hình, 1 sai nhiều nhất, 1 ngày lễ | Chỉ minh họa, không khái quát hóa |

**Mẹo để xAI đọc được "driver" thay vì chỉ thấy "mức nền của cửa hàng":** biến lịch sử (`dow_promo_mean`, `roll91`...) gần như chắc chắn sẽ chiếm importance lớn nhất vì chúng mã hóa "cửa hàng này vốn bán nhiều/ít". Để thấy driver kinh doanh, làm thêm **một góc nhìn thứ hai**: SHAP/importance **chỉ trên nhóm biến lịch + khuyến mãi + đối thủ**, hoặc huấn luyện phụ trên phần dư `y_log - dow_promo_mean`. Báo cáo cả hai góc nhìn và nói rõ là hai góc nhìn khác nhau.

## 10.7 E03: Kiểm tra độ tin cậy của xAI
- **Ổn định theo thời gian:** tính importance/SHAP trên **từng fold** rồi so thứ hạng (ví dụ Spearman giữa các fold). Gợi ý ngưỡng đạt: top-5 biến trùng ≥ 4/5, Spearman ≥ 0.8, dấu của hiệu ứng không đổi.
- **Ổn định theo model:** RF vs GB; thử vài `random_seed`.
- **Cảm biến rò rỉ:** nếu `Customers` hoặc biến lag < 48 ngày xuất hiện trong danh sách importance → **có lỗi ở F04**, dừng lại sửa.

## 10.8 D02: Business memo - dự báo dùng để làm gì
- Dự báo 48 ngày theo cửa hàng → **kế hoạch nhập hàng** và **xếp ca nhân viên** (đặc biệt các ngày Promo vs không Promo, các ngày Chủ nhật mở cửa).
- Nêu **độ tin cậy theo nhóm cửa hàng** (nhóm nào nên có hàng tồn an toàn lớn hơn, nhóm nào dự báo chắc).
- Nêu rõ **giới hạn:** dữ liệu quan sát, không có margin/chi phí, 180 CH thiếu dữ liệu 2014, 259 CH ngoài phạm vi test.

## 10.9 Những quyết định cần nhóm chốt sớm
| # | Quyết định | Gợi ý |
|---|---|---|
| 1 | Orange có phải chạy đủ pipeline không, hay chỉ trên mẫu? (mục 10.1) | Hỏi thầy; mặc định: Python cho toàn bộ, Orange cho mẫu theo cửa hàng |
| 2 | Giữ cách "lag ≥ 48" (mỗi dòng độc lập) hay chuyển sang cách "mốc cố định" (mỗi mốc T dựng 1 bộ biến, thêm biến *h* = số ngày kể từ T)? | Bắt đầu bằng "lag ≥ 48" (đơn giản, dễ kiểm tra rò rỉ). Chỉ thử cách mốc cố định nếu còn thời gian và M02 cho thấy sai số tăng theo h |
| 3 | Dùng `Customers` dạng lịch sử hay không? | Chỉ thêm nếu cải thiện validation rõ rệt |
| 4 | Fold F4 (cùng mùa năm trước) có đưa vào điểm tune không? | Có, nhưng báo cáo riêng vì chỉ ~676 cửa hàng |
| 5 | Phạm vi phân tích lỗi cho 180 CH thiếu dữ liệu | Báo cáo riêng một dòng trong E02 |

---
**Output của F02 (để F04 dùng ngay):** `F02_rules.csv` (14 luật), `F02_feature_spec.csv` (đặc tả biến và dtype), `F02_validation_folds_and_baseline_floor.csv` (4 fold + mức sàn), cùng các hàm trong notebook: `build_history_features`, `leakage_test`, `add_competition_features`, `add_promo2_features`, `add_holiday_features`, `postprocess`, `rmspe`, `make_submission`, `align_categories`.

---
# PHẦN B - TRIỂN KHAI: F04 → M02 → E02 (Model 2: Forecasting 48 ngày)

> **Nguyên tắc đánh giá của phần này (theo yêu cầu của nhóm):** mọi điểm số ở đây được tính trên **tệp validation** (4 fold walk-forward đã định nghĩa ở mục 6, Sales thật của các fold nằm trong `train.csv`). Đây là **đánh giá sơ bộ**. Điểm trên **tệp test của Kaggle** sẽ đến sau, khi nộp file dự báo ở mục cuối. Vì vậy mọi nhận xét dưới đây viết là *"ước lượng trên validation"*, không phải *"điểm cuối cùng"*.

## 11. F04 - Dựng bảng dữ liệu cho Model 2 (áp luật R1-R14)

**Việc cần làm (đơn giản):** gom mọi biến đã chốt ở F02 vào **một bảng duy nhất**, mỗi dòng = 1 cặp (cửa hàng, ngày); **train và test đi qua đúng cùng một đoạn code** (luật R14) nên không thể lệch nhau.

### 11.0 Cấu hình chung của phần B

In [ ]:
QUICK = False            # True = chạy nhanh để kiểm tra code (ít cây, ít cấu hình tuning, ít cửa hàng). False = cấu hình dùng cho báo cáo.
SEED  = 42
N_JOBS = -1
rng = np.random.default_rng(SEED)

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import time
print('LightGBM:', 'có' if HAS_LGB else 'KHÔNG có -> dùng HistGradientBoostingRegressor của sklearn thay thế', '| QUICK =', QUICK)

### 11.1 Ghép train + test thành một bảng, tạo biến lịch / Promo / lễ / cửa hàng

In [ ]:
base_cols = ['Store', 'Date', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']
tr_ = train[base_cols + ['Sales']].copy(); tr_['is_test'] = 0; tr_['Id'] = -1
te_ = test[base_cols + ['Id']].copy();     te_['Sales'] = np.nan; te_['is_test'] = 1
te_['Open'] = te_['Open'].fillna(1)                                  # R3.2: Open thiếu (Store 622) -> 1
TB = pd.concat([tr_, te_], ignore_index=True)
assert not TB.duplicated(['Store', 'Date']).any(), 'Vi phạm: (Store, Date) phải là khóa duy nhất (U02 mục 1.9)'
ri, ci = ALL_DATES.get_indexer(TB.Date), STORES.get_indexer(TB.Store)   # vị trí của từng dòng trong ma trận ngày x cửa hàng

# --- lịch
TB['Month']       = TB.Date.dt.month.astype('int8')
TB['WeekOfYear']  = TB.Date.dt.isocalendar().week.astype('int8')
TB['DayOfMonth']  = TB.Date.dt.day.astype('int8')
TB['DaysToMonthEnd'] = (TB.Date.dt.days_in_month - TB.Date.dt.day).astype('int8')

# --- lịch Promo biết trước (R4.2)
PM = Promo.values
week_block = pd.factorize(ALL_DATES.to_period('W-SUN'))[0]
IsPromoWeek = pd.DataFrame(PM).groupby(week_block).transform('max').values
TB['IsPromoWeek'] = np.nan_to_num(IsPromoWeek[ri, ci], nan=0).astype('int8')
TB['Promo_prev7'] = np.nan_to_num(pd.DataFrame(PM).shift(7).values[ri, ci],  nan=-1).astype('int8')
TB['Promo_next7'] = np.nan_to_num(pd.DataFrame(PM).shift(-7).values[ri, ci], nan=-1).astype('int8')

# --- lịch lễ (R12): StateHoliday mã hóa với danh sách mức cố định
TB = add_holiday_features(TB, nat)
TB['StateHoliday'] = pd.Categorical(TB.StateHoliday, categories=CATS['StateHoliday']).codes.astype('int8')

# --- đặc điểm cửa hàng cố định
st_small = store[['Store', 'StoreType', 'Assortment']].copy()
TB = TB.merge(st_small, on='Store', how='left')
TB['StoreType']  = pd.Categorical(TB.StoreType,  categories=CATS['StoreType']).codes.astype('int8')
TB['Assortment'] = pd.Categorical(TB.Assortment, categories=CATS['Assortment']).codes.astype('int8')
assert (ALL_DATES.get_indexer(TB.Date) == ri).all()          # merge không làm đổi thứ tự dòng

# --- Competition (R10) và Promo2 (R11): đúng hàm đã kiểm tra ở mục 7
cf = add_competition_features(TB[['Store', 'Date']], store)
for c_ in ['CompetitionDistance', 'CompetitionDistanceMissing', 'MonthsSinceCompetitionOpened', 'CompetitionSinceUnknown', 'CompetitorNotOpenYet']:
    TB[c_] = cf[c_].values
pf = add_promo2_features(TB[['Store', 'Date']], store)
for c_ in ['Promo2', 'Promo2Started', 'IsPromo2ActiveThisMonth', 'Promo2WeeksSinceStart']:
    TB[c_] = pf[c_].values
del cf, pf
print('Bảng sau bước 11.1:', TB.shape)

### 11.2 Ghép biến lịch sử Sales (R5-R7) và tạo biến mục tiêu
Biến lịch sử đã được tạo và kiểm tra rò rỉ ở mục 5 (`HIST`); ở đây chỉ **tra cứu** theo cặp (ngày, cửa hàng). Vì mọi biến đều chỉ dùng dữ liệu ≤ t − 48 nên **chỉ cần tạo 1 lần cho cả 4 fold**, không cần tính lại theo fold.

In [ ]:
for k, m in HIST.items():
    TB[k] = m.values[ri, ci].astype('float32')
TB['has_ly364'] = TB['ly364'].notna().astype('int8')                  # R7: cờ thiếu "cùng kỳ năm trước"
TB['dow_promo_n'] = TB['dow_promo_n'].fillna(0).astype('int8')
TB['open28'] = TB['open28'].fillna(0).astype('int8')

# biến mục tiêu và cờ dòng được phép train (R2, R8, R13)
TB['y_log']    = np.log1p(TB.Sales).astype('float32')
TB['valid_y']  = ((TB.is_test == 0) & (TB.Open == 1) & (TB.Sales > 0))
TB['train_ok'] = TB.valid_y & (TB.Date >= warm_start)

# kiểu dữ liệu gọn
for c_ in ['CompetitionDistance', 'MonthsSinceCompetitionOpened', 'Promo2WeeksSinceStart', 'DaysSinceLastHoliday']:
    TB[c_] = TB[c_].astype('float32')
for c_ in ['Open', 'Promo', 'SchoolHoliday', 'DayOfWeek', 'is_test', 'Promo2']:
    TB[c_] = TB[c_].astype('int8')
TB['Store'] = TB['Store'].astype('int16')
print(f'Bảng cuối: {TB.shape[0]:,} dòng x {TB.shape[1]} cột | bộ nhớ ~{TB.memory_usage(deep=True).sum()/1e6:.0f} MB')
print(f"Dòng được phép train (R2+R8): {int(TB.train_ok.sum()):,} | dòng test: {int((TB.is_test==1).sum()):,}")

### 11.2b Hai nhóm biến còn thiếu so với EDA (U02): đóng cửa dài ngày (R15) và mùa vụ thị trường (R16)
- **R15 - đóng cửa dài (U02 mục 1.7):** 72 cửa hàng từng đóng liên tục ≥ 10 ngày (có đợt tới 185 ngày). Sau khi mở lại doanh số có thể lệch mạnh nên baseline và biến cuốn bị méo. Tạo `DaysSinceReopen` (số ngày kể từ lần mở lại) và `LastClosureLen` (độ dài đợt đóng cửa đó). **Chỉ nhìn ngược về quá khứ** (dùng `Open` đến ngày hiện tại, vốn có sẵn trong test), nên không rò rỉ. Ngày thiếu dữ liệu (180 cửa hàng thiếu 2014) **không** bị coi là đóng cửa.
- **R16 - mùa vụ thị trường (U02 mục 3.5):** tháng 12 cao (chỉ số 1.24), tháng 1-2 và 8-10 thấp. Biến lịch sử cách ít nhất 48 ngày không "thấy" được mùa vụ ở ngày cần dự báo, nên thêm `chain_delta` = (log-sales TB toàn chuỗi của **tuần cùng kỳ năm trước**) trừ (log-sales TB toàn chuỗi của **cửa sổ 91 ngày mốc, cùng kỳ năm trước**). Dùng một **panel cố định** (bỏ 180 cửa hàng thiếu 2014 và StoreType b) để thành phần không đổi theo thời gian.
- Hai nhóm biến này **chưa chắc có ích**: ở mục 12.2 sẽ chạy A/B trên validation và chỉ giữ lại nếu thật sự cải thiện hơn mức nhiễu do seed.

In [ ]:
# --- R15: đóng cửa dài ngày. Ma trận Open (ngày x cửa hàng) gồm cả giai đoạn test (Open của test có sẵn).
OPEN_M = np.full((len(ALL_DATES), len(STORES)), np.nan); OPEN_M[ri, ci] = TB.Open.values.astype(float)
LONG_CLOSURE_DAYS, CAP_REOPEN = 10, 120

def closure_features(open_m, min_len=LONG_CLOSURE_DAYS, cap=CAP_REOPEN):
    nd, ns = open_m.shape; idx = np.arange(nd)
    since = np.full((nd, ns), float(cap), dtype='float32'); last_len = np.zeros((nd, ns), dtype='float32'); n_spells = 0
    for j in range(ns):
        closed = (open_m[:, j] == 0).astype(np.int8)                                  # NaN (thiếu ngày) không phải đóng cửa
        d = np.diff(np.concatenate([[0], closed, [0]])); starts = np.where(d == 1)[0]; ends = np.where(d == -1)[0] - 1
        marker = np.full(nd, -1); lens = np.zeros(nd)
        for s_, e_ in zip(starts, ends):
            L = e_ - s_ + 1
            if L >= min_len and e_ + 1 < nd: marker[e_ + 1] = e_ + 1; lens[e_ + 1] = L; n_spells += 1
        if (marker >= 0).any():
            pos = np.maximum.accumulate(marker); ok = pos >= 0
            since[ok, j] = np.minimum(idx[ok] - pos[ok], cap); last_len[ok, j] = np.minimum(lens[pos[ok]], 200)
    return since, last_len, n_spells

SINCE_M, LEN_M, n_sp = closure_features(OPEN_M)
TB['DaysSinceReopen'] = SINCE_M[ri, ci]; TB['LastClosureLen'] = LEN_M[ri, ci]
stores_closed = int((LEN_M.max(axis=0) > 0).sum())
print(f'R15: {n_sp} đợt đóng cửa >= {LONG_CLOSURE_DAYS} ngày liên tục, {stores_closed} cửa hàng (U02 mục 1.7 ghi 72 CH >= 10 ngày trong train); đợt dài nhất = {int(LEN_M.max())} ngày')
print('Dòng test có DaysSinceReopen < 120 (vừa mở lại gần đây):', int((TB.loc[TB.is_test == 1, 'DaysSinceReopen'] < CAP_REOPEN).sum()))

# --- R16: mùa vụ thị trường
panel_stores = [s_ for s_ in STORES if (s_ not in set(gap_stores)) and store.set_index('Store').loc[s_, 'StoreType'] != 'b']
def chain_delta_series(logS_, horizon=HORIZON):
    chain = logS_[panel_stores].mean(axis=1)                                          # log-sales TB toàn chuỗi mỗi ngày (ngày mở cửa)
    A = chain.rolling(7, center=True, min_periods=4).mean().shift(364)                # tuần cùng kỳ năm trước (cửa sổ tâm, dùng tới t-361)
    B = chain.rolling(91, min_periods=60).mean().shift(364 + horizon)                 # cửa sổ mốc 91 ngày cùng kỳ năm trước
    return (A - B)
CHAIN_DELTA = chain_delta_series(logS)
TB['chain_delta'] = CHAIN_DELTA.values[ri].astype('float32')
print(f'R16: panel {len(panel_stores)} cửa hàng | chain_delta có giá trị ở {TB.chain_delta.notna().mean()*100:.1f}% dòng | test: {TB.loc[TB.is_test == 1, "chain_delta"].notna().mean()*100:.1f}%')
print('Khoảng giá trị chain_delta (test):', TB.loc[TB.is_test == 1, 'chain_delta'].describe()[['min', 'mean', 'max']].round(3).to_dict())

In [ ]:
CAT_FEATURES = ['Store', 'StoreType', 'Assortment', 'StateHoliday']           # biến phân loại (Gradient Boosting dùng natively)
FEATURES = ['Store', 'DayOfWeek', 'Month', 'WeekOfYear', 'DayOfMonth', 'DaysToMonthEnd',
            'Promo', 'IsPromoWeek', 'Promo_prev7', 'Promo_next7',
            'StateHoliday', 'SchoolHoliday', 'DaysSinceLastHoliday', 'IsDayBeforeHoliday', 'IsDayAfterHoliday',
            'StoreType', 'Assortment',
            'CompetitionDistance', 'CompetitionDistanceMissing', 'MonthsSinceCompetitionOpened', 'CompetitionSinceUnknown', 'CompetitorNotOpenYet',
            'Promo2', 'Promo2Started', 'IsPromo2ActiveThisMonth', 'Promo2WeeksSinceStart',
            'dow_promo_mean', 'dow_promo_n', 'roll28', 'roll91', 'open28', 'promo_uplift91', 'ly364', 'ly728', 'has_ly364']
FORBIDDEN = {'Customers', 'Sales', 'Id', 'y_log', 'Date'}
assert not (set(FEATURES) & FORBIDDEN), 'Có biến bị cấm trong FEATURES (R1)'
assert all(f in TB.columns for f in FEATURES)
print(len(FEATURES), 'biến đưa vào model; không có Customers / Sales thô / Id (R1).')

### 11.3 Kiểm tra nhất quán train - test (R14) và kiểm tra rò rỉ cuối cùng
So sánh tỷ lệ NaN từng biến giữa dòng train (dùng để học) và dòng test (cần dự báo). **Chênh lệch lớn không phải lúc nào cũng là lỗi** (ví dụ `ly364` thiếu nhiều ở train vì năm 2013 không có "năm trước"), nhưng mỗi chênh lệch lớn phải **giải thích được**.

In [ ]:
trn = TB[TB.train_ok]; tst = TB[(TB.is_test == 1) & (TB.Open == 1)]
cmp_nan = pd.DataFrame({'NaN% train': trn[FEATURES].isna().mean(), 'NaN% test': tst[FEATURES].isna().mean()})
cmp_nan['chênh lệch'] = (cmp_nan['NaN% test'] - cmp_nan['NaN% train']).round(3)
display(cmp_nan[(cmp_nan[['NaN% train', 'NaN% test']].max(axis=1) > 0)].round(3))

# phân phối trung bình các biến số chính: train vs test (kiểm tra lệch phân phối)
chk = ['dow_promo_mean', 'roll91', 'CompetitionDistance', 'MonthsSinceCompetitionOpened', 'Promo']
display(pd.DataFrame({'train (trung bình)': trn[chk].mean(), 'test (trung bình)': tst[chk].mean()}).round(3))

# kiểm tra rò rỉ lần cuối trên chính bảng cuối: không có biến nào mượn Sales của ngày > t-48 (đã test ở 5.4); không có Customers
print('Customers trong bảng cuối?', 'Customers' in TB.columns, '| Sales thô nằm trong FEATURES?', 'Sales' in FEATURES)

**Kiểm tra rò rỉ ở ĐÚNG 4 mốc cắt của 4 fold** (không chỉ mốc 09/03/2015 ở mục 5.4): với mỗi mốc T, giả vờ chỉ có dữ liệu đến T, tính lại mọi biến lịch sử (và `chain_delta`), rồi so với bản đầy đủ trên các ngày ≤ T + 48. Phải **giống hệt**.

In [ ]:
def leak_all_origins(fn):
    out = {}
    for fname, T0 in FOLD_ORIGINS.items():
        res = leakage_test(T0, fn); out[fname] = bool((res.iloc[:, 1] == 'ĐẠT').all())
    return out
fn_all = lambda ls: {**build_history_features(ls, Promo), 'chain_delta': chain_delta_series(ls).to_frame('chain_delta')}
LEAK_RES = leak_all_origins(fn_all)
print('Kết quả test rò rỉ tại 4 mốc T của 4 fold (True = ĐẠT, mọi biến giống hệt):'); display(pd.Series(LEAK_RES).to_frame('ĐẠT?'))
assert all(LEAK_RES.values()), 'Có biến rò rỉ dữ liệu sau mốc T - dừng lại và sửa trước khi huấn luyện'

**Nhận xét F04 (đọc kết quả trên):**
- **Kích thước:** bảng gồm **1.058.297 dòng**; **774.324 dòng** đủ điều kiện để học (R2 + warm-up R8) và **41.088 dòng test**. Sau bước 11.1-11.2 có 43 cột, thêm 3 biến bổ sung ở 11.2b (R15, R16) thành 46 cột. (Ở mục 11.2 vẫn in "35 biến" vì đó là bộ biến gốc; số biến thực sự đưa vào model được chốt ở mục 12.2.)
- **Code kiểm tra khóa duy nhất `(Store, Date)` chạy qua:** không có dòng trùng (khớp U02 mục 1.9).
- **R15 tái hiện đúng EDA:** tìm ra **74 đợt đóng cửa liên tục ≥ 10 ngày ở 72 cửa hàng**, đợt dài nhất **185 ngày**, đúng con số U02 mục 1.7 đã ghi. Có 96 dòng test thuộc cửa hàng vừa mở lại trong vòng 120 ngày, nên nhóm này nhỏ ở test.
- **R16:** `chain_delta` có giá trị ở 50,4% dòng (vì cần dữ liệu "cùng kỳ năm trước", nên 2013 và đầu 2014 để trống) nhưng ở **100% dòng test**. Khoảng giá trị ở test từ −0,13 đến +0,25 (log), tức độ lệch mùa vụ cùng kỳ năm trước dao động khoảng −12% đến +29% (`exp(−0,134)−1` đến `exp(0,254)−1`).
- **Kiểm tra rò rỉ ở đúng 4 mốc T của 4 fold: cả 4 đều ĐẠT** (mọi biến lịch sử và `chain_delta` giống hệt khi chỉ có dữ liệu đến T). Chạy chốt chặn `assert` nên nếu lỗi thì notebook dừng.
- **Chỗ lệch giữa train và test (giải thích được, không phải lỗi):** `ly728` thiếu 75,5% (train) so với 0,7% (test) vì train năm 2013-2014 chưa có "2 năm trước"; `ly364` thiếu 37,4% so với 20,8% (phần thiếu ở test là 180 cửa hàng mất dữ liệu 2014); `Promo2WeeksSinceStart` thiếu 56% so với 42%; `MonthsSinceCompetitionOpened` trung bình 60 (train) so với 74 tháng (test). Mức nền doanh số gần nhau: `dow_promo_mean` 8,756 so với 8,804; tỷ lệ dòng có Promo 45,1% so với 46,3%. Chỉ riêng `ly728` là khác biệt lớn đáng nhớ: model được học trên dữ liệu mà biến này thiếu phần lớn, nhưng khi dùng thật gần như luôn có.

### 11.3 Kiểm tra Tương quan (Correlation Check) & Biện luận Đa cộng tuyến
Trước khi đưa các biến vào mô hình, ta phân tích ma trận tương quan giữa các đặc trưng để:
1. **Phát hiện đa cộng tuyến:** Nhận diện các cặp biến có hệ số tương quan rất cao ($|r| > 0.80$).
2. **Biện luận giữ/loại biến:** Giải thích vì sao mô hình cây (LightGBM) vẫn hưởng lợi khi giữ các biến tương quan bổ trợ (như `roll28` bắt nhịp ngắn hạn 4 tuần kết hợp `roll91` neo mức nền trung hạn 13 tuần).
3. **Cơ sở khoa học cho xAI:** Chứng minh vì sao ở bước giải thích mô hình (xAI Tầng 1), ta bắt buộc phải **xáo trộn theo nhóm (Group Permutation)** thay vì xáo từng biến đơn lẻ.

In [ ]:
# 11.3 Biểu đồ Ma trận Tương quan (Correlation Heatmap) giữa các biến chính
key_feats = [
    "dow_promo_mean", "roll28", "roll91", "ly364", "ly728",
    "Promo", "IsPromoWeek", "Promo_prev7", "Promo_next7",
    "DayOfWeek", "DaysToMonthEnd", "DayOfMonth", "WeekOfYear",
    "chain_delta", "CompetitionDistance", "Store"
]
corr_sub = TB.loc[TB.train_ok, key_feats].sample(min(10000, len(TB)), random_state=SEED).corr()

fig, ax = plt.subplots(figsize=(11, 9))
cax = ax.matshow(corr_sub, cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(cax, fraction=0.046, pad=0.04)
ax.set_xticks(range(len(key_feats))); ax.set_yticks(range(len(key_feats)))
ax.set_xticklabels(key_feats, rotation=45, ha="left", fontsize=9)
ax.set_yticklabels(key_feats, fontsize=9)
for i in range(len(key_feats)):
    for j in range(len(key_feats)):
        val = corr_sub.iloc[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color="white" if abs(val) > 0.55 else "black", fontsize=8)
ax.set_title("Feature Correlation Matrix (Key Drivers)", fontsize=12, fontweight="bold", pad=25)
plt.tight_layout(); plt.show()

# In các cặp tương quan mạnh nhất
corr_pairs = []
for i in range(len(key_feats)):
    for j in range(i+1, len(key_feats)):
        r = corr_sub.iloc[i, j]
        if abs(r) >= 0.70:
            corr_pairs.append((key_feats[i], key_feats[j], r))
print("Các cặp biến tương quan mạnh nhất (|r| >= 0.70):")
for f1, f2, r in sorted(corr_pairs, key=lambda x: abs(x[2]), reverse=True):
    print(f"  - {f1:<22} vs {f2:<22}: r = {r:+.2f}")


---
## 12. M02 - Model 2: baseline → mô hình ML → tuning → so sánh
**Cách làm (đúng "bậc thang" ở mục 10.2):** mỗi mô hình đều chạy trên **cùng 4 fold, cùng bộ biến, cùng hàm chấm điểm** để so sánh công bằng. Mọi điểm RMSPE dưới đây là **điểm trên validation**, thang gốc (Sales), bỏ dòng Sales=0, chỉ các cửa hàng trong test.

### 12.1 Chuẩn bị 4 fold
Mỗi fold: **train = các dòng đủ điều kiện (R2, R8) có Date ≤ T**, **validate = 48 ngày sau T** của các cửa hàng trong test.

In [ ]:
FOLD_NAMES = list(FOLD_ORIGINS)
def fold_masks(T0):
    T1 = T0 + pd.Timedelta(days=HORIZON)
    m_tr = (TB.train_ok & (TB.Date <= T0)).values
    m_va = ((TB.is_test == 0) & (TB.Date > T0) & (TB.Date <= T1) & TB.Store.isin(test_stores)).values
    return m_tr, m_va

meta_cols = ['Store', 'Date', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'StoreType', 'Assortment', 'Promo2',
             'IsPromo2ActiveThisMonth', 'has_ly364', 'MonthsSinceCompetitionOpened', 'CompetitionSinceUnknown', 'Sales']
VAL = []
for name, T0 in FOLD_ORIGINS.items():
    _, m_va = fold_masks(T0)
    v = TB.loc[m_va, meta_cols].copy(); v['fold'] = name; v['h'] = (v.Date - T0).dt.days     # h = ngày dự báo thứ mấy (1..48)
    VAL.append(v)
VAL = pd.concat(VAL, ignore_index=True)
ROWS = {f: np.where(VAL.fold.values == f)[0] for f in FOLD_NAMES}
valid_val = ((VAL.Open == 1) & (VAL.Sales > 0)).values

summary = VAL.assign(chấm_điểm=valid_val).groupby('fold').agg(số_dòng=('Store', 'size'), số_dòng_chấm=('chấm_điểm', 'sum'), số_CH=('Store', 'nunique'))
print('Tập validation (đã loại khỏi quá trình học của chính fold đó):'); display(summary.loc[FOLD_NAMES])

PRED = {}            # PRED[model] = vector dự báo trên thang log, cùng thứ tự với VAL
TIMES = {}
def score_table(models, c_map=None, mask=None, decimals=4):
    c_map = c_map or {}; out = {}
    for m_ in models:
        ps = postprocess(PRED[m_], VAL.Open.values, c_map.get(m_, 1.0)); res = {}
        for f in FOLD_NAMES:
            mk = (VAL.fold.values == f) & (mask if mask is not None else True)
            res[f] = rmspe(VAL.Sales.values[mk], ps[mk])
        vals = [res[f] for f in FOLD_NAMES]; res['Trung bình'] = float(np.mean(vals)); res['Độ lệch chuẩn'] = float(np.std(vals))
        out[m_] = res
    return pd.DataFrame(out).T.round(decimals)

### 12.2 Kiểm tra A/B: hai nhóm biến bổ sung từ EDA (R15, R16) có thật sự giúp không?
Ở mục 11.2b mình thêm 3 biến lấy từ các phát hiện EDA mà bản trước còn thiếu. **Không mặc định giữ lại**: chạy Gradient Boosting (tham số mặc định) trên đúng 4 fold với 3 bộ biến và so sánh.
- **V0:** 35 biến gốc (theo F02) | **V1:** V0 + đóng cửa dài (`DaysSinceReopen`, `LastClosureLen`) | **V2:** V1 + `chain_delta`.
- Mỗi lần huấn luyện có yếu tố ngẫu nhiên (hạt giống `seed`), nên chạy **V0 với 3 hạt giống** để đo **mức nhiễu của chính phép so sánh**. Quy tắc chọn: lấy bộ biến **đơn giản nhất** có RMSPE trung bình **không kém bộ tốt nhất quá 1 độ lệch chuẩn nhiễu**. Cải thiện nhỏ hơn mức nhiễu thì **không được coi là cải thiện**.

In [ ]:
DEFAULT_GB = dict(n_estimators=(60 if QUICK else 300), learning_rate=0.10, num_leaves=63, min_child_samples=50,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=0.0)

class GB:
    # Bọc LightGBM (ưu tiên) hoặc HistGradientBoosting (dự phòng) để dùng chung một giao diện.
    def __init__(self, params, use_store=True, seed=SEED, feats=None):
        self.p, self.seed = dict(params), seed
        base = list(feats) if feats is not None else list(FEATURES)
        self.feats = base if (use_store and HAS_LGB) else [f for f in base if f != 'Store']
        self.cats = [c for c in CAT_FEATURES if c in self.feats]
    def fit(self, df, y):
        if HAS_LGB:
            self.m = lgb.LGBMRegressor(**self.p, n_jobs=N_JOBS, verbose=-1, random_state=self.seed)
            self.m.fit(df[self.feats], y, categorical_feature=self.cats)
        else:
            p = self.p; mask = [c in self.cats for c in self.feats]
            self.m = HistGradientBoostingRegressor(max_iter=p['n_estimators'], learning_rate=p['learning_rate'], max_leaf_nodes=p['num_leaves'],
                                                   min_samples_leaf=p['min_child_samples'], l2_regularization=p.get('reg_lambda', 0.0),
                                                   categorical_features=mask, random_state=self.seed)
            self.m.fit(df[self.feats], y)
        return self
    def predict(self, df): return self.m.predict(df[self.feats])

def gb_fold_scores(feats, seed=SEED, params=None, train_stores=None, use_store=True):
    # RMSPE (thang gốc, c = 1) của GB trên 4 fold với bộ biến `feats`; train_stores = chỉ huấn luyện trên các cửa hàng này (nếu có)
    params = params or DEFAULT_GB; res = {}
    for name, T0 in FOLD_ORIGINS.items():
        m_tr, m_va = fold_masks(T0)
        if train_stores is not None: m_tr = m_tr & TB.Store.isin(train_stores).values
        tr, va = TB.loc[m_tr], TB.loc[m_va]
        model = GB(params, use_store=use_store, seed=seed, feats=feats).fit(tr, tr.y_log.values)
        res[name] = rmspe(va.Sales.values, postprocess(model.predict(va), va.Open.values))
    return res

FEATURES_V0 = list(FEATURES)
FEATURES_V1 = FEATURES_V0 + ['DaysSinceReopen', 'LastClosureLen']
FEATURES_V2 = FEATURES_V1 + ['chain_delta']
VARIANTS = {'V0 bộ biến gốc (35)': FEATURES_V0, 'V1 + đóng cửa dài (R15)': FEATURES_V1, 'V2 + mùa vụ chuỗi (R16)': FEATURES_V2}
t0 = time.time()
AB = {vn: gb_fold_scores(fs) for vn, fs in VARIANTS.items()}
V0_NAME = 'V0 bộ biến gốc (35)'
seed_runs = {SEED: AB[V0_NAME]}
for sd in ([SEED + 1] if QUICK else [SEED + 1, SEED + 2]): seed_runs[sd] = gb_fold_scores(FEATURES_V0, seed=sd)
ab_df = pd.DataFrame(AB).T[FOLD_NAMES]; ab_df['Trung bình'] = ab_df.mean(axis=1)
seed_df = pd.DataFrame(seed_runs).T[FOLD_NAMES]; seed_df['Trung bình'] = seed_df.mean(axis=1); seed_df.index = [f'V0, seed {s}' for s in seed_df.index]
NOISE_SD = float(seed_df['Trung bình'].std(ddof=1))
best_mean = ab_df['Trung bình'].min()
CHOSEN = next(vn for vn in VARIANTS if ab_df.loc[vn, 'Trung bình'] <= best_mean + NOISE_SD)
print(f'A/B xong trong {time.time()-t0:.0f}s'); print('RMSPE theo bộ biến (GB mặc định, seed ' + str(SEED) + '):'); display(ab_df.round(4))
print('Mức nhiễu do hạt giống (V0 chạy nhiều seed):'); display(seed_df.round(4))
print(f'Độ lệch chuẩn nhiễu của RMSPE trung bình = {NOISE_SD:.4f}')
print(f'=> Bộ biến được chọn: {CHOSEN}  (bộ tốt nhất {ab_df["Trung bình"].idxmin()} = {best_mean:.4f}; bộ đơn giản nhất nằm trong 1 độ lệch chuẩn nhiễu của bộ tốt nhất)')
FEATURES = list(VARIANTS[CHOSEN]); FEATURES_FINAL = FEATURES
print(len(FEATURES), 'biến được dùng cho mọi mô hình ở các mục sau.')

In [ ]:
# Lưu mẫu cho Orange (P06) với bộ biến đã chốt
orange_stores = stratified_store_sample()
orange_df = TB[TB.Store.isin(orange_stores) & (TB.is_test == 0)][['Store', 'Date'] + FEATURES[1:] + ['Open', 'Sales', 'y_log']]
orange_df.to_csv(OUT_DIR / 'F04_orange_sample.csv', index=False)
print(f'Mẫu cho Orange: {len(orange_stores)} cửa hàng, {len(orange_df):,} dòng x {orange_df.shape[1]} cột -> F04_orange_sample.csv')
print('Phân bố StoreType trong mẫu:', store[store.Store.isin(orange_stores)].StoreType.value_counts().to_dict())

**Nhận xét A/B (R15, R16):**
- **Kết quả:** V0 (bộ gốc) **0,1142**, V1 (+ đóng cửa dài) **0,1148**, V2 (+ đóng cửa dài + mùa vụ chuỗi) **0,1132**. Mức nhiễu do hạt giống: độ lệch chuẩn **0,0004** (V0 chạy 3 seed cho 0,1142 / 0,1144 / 0,1150).
- **Quy tắc đã chọn V2 (38 biến).** V2 tốt hơn V0 khoảng **0,0010** (khoảng 2,5 lần mức nhiễu), tốt hơn V1 khoảng **0,0016** (khoảng 4 lần mức nhiễu). Cải thiện nhỏ (khoảng 1%) nhưng lớn hơn nhiễu.
- **R15 (đóng cửa dài) tự nó không giúp:** V1 kém V0 một chút (0,0006, trong khoảng nhiễu) và tệ hơn rõ ở F3 (0,1232 so với 0,1210). Mức cải thiện của V2 nhiều khả năng đến từ `chain_delta` (R16), nhưng **mình chưa chạy riêng bộ "V0 + chain_delta"** nên chưa tách được chắc chắn; V2 vẫn mang theo 2 biến R15. Nếu muốn gọn, đây là thử nghiệm đáng làm tiếp.
- **Vì sao R15 có thể không giúp:** chỉ 259 dòng validation thuộc cửa hàng vừa đóng cửa dài (E02), nên model ít dữ liệu để học; chưa kiểm chứng thêm.

### 12.3 Baseline (mức sàn): B1, B2, B3
Dùng lại đúng 3 dự báo đơn giản của mục 6.2, nhưng bây giờ chạy trên **cùng tập dòng** với các model ML. Dòng nào baseline không có số (cửa hàng thiếu lịch sử) thì **lùi về phương án kế tiếp** (B2 → B1 → B3 → `roll91` → trung vị toàn chuỗi) để mọi mô hình được chấm trên đúng cùng một tập dòng.

In [ ]:
def chain(*arrs, default):
    out = np.array(arrs[0], float).copy()
    for a in arrs[1:]: out = np.where(np.isnan(out), np.asarray(a, float), out)
    return np.where(np.isnan(out), default, out)
GLOBAL_MED = float(TB.loc[TB.train_ok, 'y_log'].median())
cols_idx = pd.Index(cols)
fallback_share = {}
for name, T0 in FOLD_ORIGINS.items():
    _, m_va = fold_masks(T0); va = TB.loc[m_va]; r = ROWS[name]
    vi = ALL_DATES[(ALL_DATES > T0) & (ALL_DATES <= T0 + pd.Timedelta(days=HORIZON))]
    rp, cp = vi.get_indexer(va.Date), cols_idx.get_indexer(va.Store)
    b1 = profile_pred(T0, vi, cols, 8,  False)[rp, cp]
    b2 = profile_pred(T0, vi, cols, 16, True)[rp, cp]
    b3 = va.dow_promo_mean.values
    for k, v in {'B1 cùng thứ': chain(b1, b3, va.roll91.values, default=GLOBAL_MED),
                 'B2 cùng thứ + Promo': chain(b2, b1, b3, va.roll91.values, default=GLOBAL_MED),
                 'B3 dow_promo_mean (lag>=49)': chain(b3, b2, b1, va.roll91.values, default=GLOBAL_MED)}.items():
        PRED.setdefault(k, np.full(len(VAL), np.nan))[r] = v
    mk = (va.Open.values == 1) & (va.Sales.values > 0)
    fallback_share[name] = round(float(np.isnan(b2[mk]).mean()), 3)
print('Tỷ lệ dòng B2 phải lùi về phương án dự phòng, theo fold:', fallback_share)
BASELINES = ['B1 cùng thứ', 'B2 cùng thứ + Promo', 'B3 dow_promo_mean (lag>=49)']
score_table(BASELINES)

### 12.4 Mô hình ML với tham số mặc định: Linear (L1), Random Forest (M1), Gradient Boosting (M2)
- **L1 - Ridge trên `y_log`:** thước đo "quan hệ gần tuyến tính đến đâu". Biến phân loại nhỏ được one-hot; không dùng `Store`.
- **M1 - Random Forest:** điền NaN theo luật R7 (`dow_promo_mean` ← `roll91` ← `roll28`; `ly364`/`ly728` ← `dow_promo_mean`), không dùng `Store`. Mỗi cây chỉ học trên một phần mẫu (`max_samples`) cho nhanh.
- **M2 - Gradient Boosting:** LightGBM (nếu máy có), nhận NaN trực tiếp, dùng `Store` làm biến phân loại.
- Mỗi fold **huấn luyện lại từ đầu** chỉ với dữ liệu ≤ T của fold đó.

In [ ]:
NOSTORE = [f for f in FEATURES if f != 'Store']
def rf_fill(df):
    d = df[NOSTORE].copy()
    d['dow_promo_mean'] = d.dow_promo_mean.fillna(d.roll91).fillna(d.roll28).fillna(GLOBAL_MED)
    d['roll91'] = d.roll91.fillna(d.dow_promo_mean); d['roll28'] = d.roll28.fillna(d.roll91)
    d['ly364']  = d.ly364.fillna(d.dow_promo_mean);  d['ly728']  = d.ly728.fillna(d.dow_promo_mean)
    return d.fillna(-1).astype('float32')

NUM_L1 = [f for f in NOSTORE if f not in ('DayOfWeek', 'Month', 'StateHoliday', 'StoreType', 'Assortment')]
CAT_L1 = ['DayOfWeek', 'Month', 'StateHoliday', 'StoreType', 'Assortment']
def make_ridge(alpha=1.0):
    try: oh = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError: oh = OneHotEncoder(handle_unknown='ignore', sparse=False)
    pre = ColumnTransformer([('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), NUM_L1), ('cat', oh, CAT_L1)])
    return make_pipeline(pre, Ridge(alpha=alpha))

RF_PARAMS = dict(n_estimators=(20 if QUICK else 60), max_samples=0.10, min_samples_leaf=10, max_features=0.5)
def run_models(spec, store_filter=None, verbose=True):
    # spec: {tên_model: hàm tạo model có fit/predict và hàm chuẩn bị X}; trả về thời gian
    for name, T0 in FOLD_ORIGINS.items():
        m_tr, m_va = fold_masks(T0)
        if store_filter is not None:
            m_tr = m_tr & TB.Store.isin(store_filter).values; m_va = m_va & TB.Store.isin(store_filter).values
        tr, va = TB.loc[m_tr], TB.loc[m_va]; r = ROWS[name]
        va_rows = np.where(m_va)[0]                                       # vị trí trong TB
        for mname, (mk_model, prep) in spec.items():
            t0 = time.time(); model = mk_model(); model.fit(prep(tr), tr.y_log.values)
            pred = model.predict(prep(va))
            full = PRED.setdefault(mname, np.full(len(VAL), np.nan))
            if store_filter is None: full[r] = pred
            TIMES[(mname, name)] = round(time.time() - t0, 1)
            if verbose: print(f'  {mname:<22} {name:<26} {TIMES[(mname, name)]:>6.1f}s')

spec_default = {
    'L1 Ridge (log)':        (lambda: make_ridge(1.0), lambda d: d[NOSTORE]),
    'M1 Random Forest':      (lambda: RandomForestRegressor(**RF_PARAMS, n_jobs=N_JOBS, random_state=SEED), rf_fill),
    'M2 Gradient Boosting':  (lambda: GB(DEFAULT_GB, use_store=True), lambda d: d),
    'M2b GB không dùng Store': (lambda: GB(DEFAULT_GB, use_store=False), lambda d: d),
}
run_models(spec_default)
MODELS_DEFAULT = ['L1 Ridge (log)', 'M1 Random Forest', 'M2 Gradient Boosting', 'M2b GB không dùng Store']
score_table(BASELINES + MODELS_DEFAULT)

### 12.5 Tuning (chỉnh tham số)
Theo mục 10.4 (rút gọn: 6 cấu hình ngẫu nhiên): **random search trên mẫu cửa hàng** (chọn theo `StoreType` để vẫn có loại `b`) cho nhanh, **điểm chọn = RMSPE trung bình 4 fold**, sau đó **huấn luyện lại cấu hình tốt nhất trên toàn bộ cửa hàng** và chấm lại. Tham số mặc định luôn nằm trong danh sách thử để biết tuning có thật sự giúp không. Lưu ý: tham số chọn trên mẫu cửa hàng là **ước lượng gần đúng** cho toàn bộ dữ liệu.

In [ ]:
TUNE_STORES = stratified_store_sample(n_total=(60 if QUICK else 200), seed=SEED + 1)
print('Mẫu tuning:', len(TUNE_STORES), 'cửa hàng | phân bố StoreType:', store[store.Store.isin(TUNE_STORES)].StoreType.value_counts().to_dict())

SPACE = {'num_leaves': [31, 63, 127, 255], 'learning_rate': [0.05, 0.08, 0.10, 0.15], 'min_child_samples': [20, 50, 100, 200],
         'colsample_bytree': [0.6, 0.8, 1.0], 'subsample': [0.6, 0.8, 1.0], 'reg_lambda': [0.0, 1.0, 5.0, 10.0], 'n_estimators': [200, 300, 500]}
N_ITER = 3 if QUICK else 6
r_ = np.random.default_rng(SEED)
CONFIGS = [('default', dict(DEFAULT_GB))]
for i in range(N_ITER):
    cfg = {k: v[r_.integers(len(v))] for k, v in SPACE.items()}; cfg['subsample_freq'] = 1; CONFIGS.append((f'rs{i+1}', cfg))
if QUICK: [c[1].update(n_estimators=min(c[1]['n_estimators'], 80)) for c in CONFIGS]

def eval_on_sample(cfg, store_sample):
    # huấn luyện trên mẫu cửa hàng, chấm RMSPE trung bình 4 fold trên chính mẫu đó (mọi thứ nằm trong mốc thời gian của từng fold)
    scores = []
    for name, T0 in FOLD_ORIGINS.items():
        m_tr, m_va = fold_masks(T0); sm = TB.Store.isin(store_sample).values
        tr, va = TB.loc[m_tr & sm], TB.loc[m_va & sm]
        model = GB(cfg).fit(tr, tr.y_log.values)
        p = postprocess(model.predict(va), va.Open.values)
        scores.append(rmspe(va.Sales.values, p))
    return float(np.mean(scores)), [round(s, 4) for s in scores]

tune_rows = []
t0 = time.time()
for nm, cfg in CONFIGS:
    s, per = eval_on_sample(cfg, TUNE_STORES)
    tune_rows.append({'cấu hình': nm, 'RMSPE TB 4 fold (mẫu)': round(s, 4), 'từng fold': per, **{k: cfg[k] for k in SPACE}})
tune_df = pd.DataFrame(tune_rows).sort_values('RMSPE TB 4 fold (mẫu)').reset_index(drop=True)
print(f'Tuning xong trong {time.time()-t0:.0f}s')
display(tune_df)
BEST_NAME = tune_df.loc[0, 'cấu hình']; BEST_GB = dict(dict(CONFIGS)[BEST_NAME])
print('Cấu hình GB tốt nhất trên mẫu:', BEST_NAME, {k: BEST_GB[k] for k in SPACE})

In [ ]:
# Tuning nhỏ cho Random Forest (3 cấu hình) trên cùng mẫu cửa hàng
RF_CONFIGS = [dict(RF_PARAMS), dict(RF_PARAMS, min_samples_leaf=5), dict(RF_PARAMS, min_samples_leaf=20, max_features=0.7)]
if QUICK: RF_CONFIGS = RF_CONFIGS[:2]
rf_rows = []
for i, cfg in enumerate(RF_CONFIGS):
    sc = []
    for name, T0 in FOLD_ORIGINS.items():
        m_tr, m_va = fold_masks(T0); sm = TB.Store.isin(TUNE_STORES).values
        tr, va = TB.loc[m_tr & sm], TB.loc[m_va & sm]
        rf = RandomForestRegressor(**cfg, n_jobs=N_JOBS, random_state=SEED).fit(rf_fill(tr), tr.y_log.values)
        sc.append(rmspe(va.Sales.values, postprocess(rf.predict(rf_fill(va)), va.Open.values)))
    rf_rows.append({'cấu hình RF': i, 'min_samples_leaf': cfg['min_samples_leaf'], 'max_features': cfg['max_features'], 'RMSPE TB 4 fold (mẫu)': round(float(np.mean(sc)), 4)})
rf_df = pd.DataFrame(rf_rows).sort_values('RMSPE TB 4 fold (mẫu)').reset_index(drop=True); display(rf_df)
BEST_RF = RF_CONFIGS[int(rf_df.loc[0, 'cấu hình RF'])]

### 12.6 Huấn luyện lại bản tốt nhất trên toàn bộ cửa hàng + hệ số hiệu chỉnh `c` (R13.2) + bản kết hợp

In [ ]:
spec_tuned = {
    'M1 Random Forest (tuned)':     (lambda: RandomForestRegressor(**BEST_RF, n_jobs=N_JOBS, random_state=SEED), rf_fill),
    'M2 Gradient Boosting (tuned)': (lambda: GB(BEST_GB, use_store=True), lambda d: d),
}
run_models(spec_tuned)
PRED['M3 Kết hợp RF + GB (tuned)'] = 0.5 * PRED['M1 Random Forest (tuned)'] + 0.5 * PRED['M2 Gradient Boosting (tuned)']   # trung bình trên thang log

# hệ số hiệu chỉnh c: 1 tham số duy nhất, chọn theo RMSPE trung bình 4 fold
grid = np.round(np.arange(0.96, 1.0401, 0.005), 3)
def mean_rmspe_for_c(m_, c_): return score_table([m_], {m_: c_}).loc[m_, 'Trung bình']
C_BEST = {}
for m_ in ['M1 Random Forest (tuned)', 'M2 Gradient Boosting (tuned)', 'M3 Kết hợp RF + GB (tuned)']:
    vals = [mean_rmspe_for_c(m_, c_) for c_ in grid]; C_BEST[m_] = float(grid[int(np.argmin(vals))])
    print(f'{m_:<32} c tốt nhất = {C_BEST[m_]:.3f} | RMSPE TB: c=1.000 -> {mean_rmspe_for_c(m_, 1.0):.4f}  ->  c={C_BEST[m_]:.3f}: {min(vals):.4f}')

### 12.7 Bảng so sánh tổng hợp (trên validation)

In [ ]:
ALL_MODELS = BASELINES + MODELS_DEFAULT + ['M1 Random Forest (tuned)', 'M2 Gradient Boosting (tuned)', 'M3 Kết hợp RF + GB (tuned)']
final = score_table(ALL_MODELS, C_BEST)
b2 = final.loc['B2 cùng thứ + Promo', FOLD_NAMES]
final['Cải thiện so với B2 (TB)'] = ((final['Trung bình'] / final.loc['B2 cùng thứ + Promo', 'Trung bình'] - 1) * 100).round(1).astype(str) + '%'
final['Số fold thắng B2'] = [(final.loc[m_, FOLD_NAMES] < b2).sum() for m_ in final.index]
display(final)

MODEL_MAP_EN = {
    'B1 cùng thứ': 'B1 DOW Mean',
    'B2 cùng thứ + Promo': 'B2 DOW + Promo',
    'B3 dow_promo_mean (lag>=49)': 'B3 Historical Profile',
    'L1 Ridge (log)': 'L1 Ridge Regression',
    'M1 Random Forest': 'M1 Random Forest (default)',
    'M2 Gradient Boosting': 'M2 Gradient Boosting (default)',
    'M2b GB không dùng Store': 'M2b GB without Store ID',
    'M1 Random Forest (tuned)': 'M1 Random Forest (tuned)',
    'M2 Gradient Boosting (tuned)': 'M2 Gradient Boosting (tuned)',
    'M3 Kết hợp RF + GB (tuned)': 'M3 Ensemble RF+GB (tuned)'
}
FOLDS_EN = {
    'F1 (gần test nhất)': 'F1',
    'F2': 'F2',
    'F3 (có lễ Phục sinh)': 'F3',
    'F4 (cùng mùa năm trước)': 'F4'
}
plot_final = final[FOLD_NAMES].rename(index=MODEL_MAP_EN, columns=FOLDS_EN)
fig, ax = plt.subplots(figsize=(11, 4.2))
plot_final.plot(kind='bar', ax=ax, width=0.85)
ax.set_ylabel('Validation RMSPE')
ax.set_title('Model Performance Across Folds')
ax.legend(title='Validation Fold', fontsize=8)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()


**Nhận xét M02 (đọc bảng ở trên):**
**Kết quả chính (RMSPE trung bình 4 fold, validation):** B1 0,227 → **B2 0,162** → B3 0,169 → Linear 0,151 → Random Forest 0,143 → **Gradient Boosting 0,113** (tuned: **0,112**).

1. **Ghép theo Promo là cải thiện lớn nhất trong các baseline** (B1 → B2 giảm 29%), đúng như luật R4.3. B3 (`dow_promo_mean` cách ≥ 49 ngày) kém B2 một chút (0,169 so với 0,162), nghĩa là chấp nhận mất ít thông tin để có luật chống rò rỉ.
2. **Mọi model ML đều thắng B2 ở cả 4 fold**, nhưng mức thắng rất khác nhau: Linear hơn B2 khoảng 7%, Random Forest khoảng 12%, **Gradient Boosting khoảng 30%** (cột *Cải thiện so với B2*, dấu âm = tốt hơn).
3. **Gradient Boosting ổn định hơn nhiều:** độ lệch chuẩn giữa các fold **0,0065** (Random Forest 0,0244, Linear 0,0266). Rõ nhất ở F3 (có Phục sinh): GB 0,121 so với RF 0,185 và Linear 0,197. Nguyên nhân chưa kiểm chứng riêng; một khả năng là GB nhận NaN trực tiếp và có dùng `Store`.
4. **Biến `Store` có ích ở cả 4 fold:** bỏ `Store` thì GB 0,1174 so với 0,1132. Nghĩa là các biến lịch sử chưa mô tả hết đặc thù từng cửa hàng.
5. **Tuning cho cải thiện nhỏ nhưng có vẻ thật:** trên mẫu 201 cửa hàng, cấu hình `rs2` (127 lá, learning rate 0,05, 500 cây, `reg_lambda` 5) tốt nhất (0,1076 so với mặc định 0,1090). Khi huấn luyện lại trên toàn bộ cửa hàng: **0,1122 so với 0,1132 của mặc định**, chênh 0,0010 (≈ 2,5 lần mức nhiễu seed 0,0004). Chỉ thử 6 cấu hình ngẫu nhiên (khoảng 0,1076-0,1139 trên mẫu) nên **không khẳng định đây là tối ưu**. Hệ số `c = 0,99` chỉ cải thiện thêm 0,0002 (0,1122 → 0,1120).
6. **Random Forest tuning được ít** (0,1434 → 0,1407, kèm `c` = 0,98). Bản **kết hợp RF + GB (0,1190) kém hơn GB đứng một mình (0,1120)** vì RF kéo trung bình xuống, nên **không dùng bản kết hợp**.
7. **Số baseline ở F3:** B2 là 0,2113 ở đây nhưng 0,1627 ở mục 6.2 vì mục 12 chấm trên mọi dòng (dòng baseline thiếu số thì lùi về phương án dự phòng, ~0,1% dòng ở F3); mình chưa tách riêng để xác nhận là nguyên nhân duy nhất.
8. **Thời gian:** GB mặc định 19-30 giây/fold, GB tuned 36-55 giây/fold, RF 25-48 giây/fold (máy 1 lõi).

**Chốt:** mô hình đi tiếp là **Gradient Boosting tuned (`rs2`, dùng `Store`, c = 0,99), 38 biến**.

---
## 13. E02 - Đánh giá Model 2 trên tệp validation
**Mục tiêu:** trả lời 4 câu hỏi (1) model tốt nhất có thật sự thắng baseline ở **mọi fold** không; (2) sai nhiều ở **nhóm nào**; (3) sai số có **tăng theo ngày dự báo thứ h** không; (4) kết quả có **hợp lý** (không rò rỉ, không lệch hệ thống) không. Đây vẫn là **đánh giá sơ bộ trên validation**; điểm trên tệp test của Kaggle sẽ có sau (mục 14).

### 13.1 Chọn mô hình tốt nhất và điểm từng fold

In [ ]:
CANDIDATES_FINAL = ['M1 Random Forest (tuned)', 'M2 Gradient Boosting (tuned)', 'M3 Kết hợp RF + GB (tuned)']
BEST_MODEL = final.loc[CANDIDATES_FINAL, 'Trung bình'].astype(float).idxmin()
BEST_C = C_BEST[BEST_MODEL]
print(f'Mô hình tốt nhất theo RMSPE trung bình 4 fold: {BEST_MODEL} (hệ số c = {BEST_C})')

VAL['valid'] = valid_val
VAL['pred_best'] = postprocess(PRED[BEST_MODEL], VAL.Open.values, BEST_C)
VAL['pred_B2']   = postprocess(PRED['B2 cùng thứ + Promo'], VAL.Open.values, 1.0)
VAL['StoreTypeL'] = VAL.StoreType.map(dict(enumerate(CATS['StoreType']))); VAL['AssortmentL'] = VAL.Assortment.map(dict(enumerate(CATS['Assortment'])))
VAL['holiday'] = (VAL.StateHoliday > 0).astype(int)

show = ['B2 cùng thứ + Promo', 'M2 Gradient Boosting (tuned)', BEST_MODEL]
tab = score_table(list(dict.fromkeys(show)), C_BEST)
print('RMSPE trên validation (tất cả dòng):'); display(tab)
print('RMSPE trên validation, loại các dòng ngày lễ nhà nước (để giống test hơn - R12.3):')
display(score_table(list(dict.fromkeys(show)), C_BEST, mask=(VAL.StateHoliday.values == 0)))

### 13.2 Sai số theo từng nhóm: model yếu ở đâu?
RMSPE của model tốt nhất (cột `best`) và của baseline B2 theo từng nhóm. Cột `n` là số dòng được chấm, `CH` là số cửa hàng - **nhóm nhỏ (nhất là StoreType b) chỉ nêu như trường hợp cá biệt**.

In [ ]:
# cờ bổ sung: cửa hàng thiếu dữ liệu 2014, đối thủ mới mở (<=6 tháng), cửa hàng vừa đóng cửa dài (>=7 ngày không phải Chủ nhật trong 28 ngày trước mốc T)
closed_ns = ((O == 0) & (ALL_DATES.dayofweek != 6)[:, None]).astype(float)
recent_closure = {}
for name, T0 in FOLD_ORIGINS.items():
    w = closed_ns.loc[T0 - pd.Timedelta(days=27):T0].sum()
    recent_closure[name] = set(w[w >= 7].index)
VAL['gap_store'] = VAL.Store.isin(gap_stores).astype(int)
VAL['new_competitor'] = ((VAL.MonthsSinceCompetitionOpened <= 6) & (VAL.CompetitionSinceUnknown == 0)).astype(int)
VAL['recent_closure'] = [int(s in recent_closure[f]) for s, f in zip(VAL.Store.values, VAL.fold.values)]
VAL['h_bucket'] = pd.cut(VAL.h, [0, 7, 14, 21, 28, 35, 42, 48], labels=['1-7', '8-14', '15-21', '22-28', '29-35', '36-42', '43-48'])

def _rm(g, col): 
    y = g.Sales.values; p = g[col].values; return np.sqrt(np.mean(((y - p) / y) ** 2))
def by_group(col, label=None):
    g = VAL[VAL.valid].groupby(col, observed=True)
    out = pd.DataFrame({'n': g.size(), 'CH': g.Store.nunique(), 'best': g.apply(lambda x: _rm(x, 'pred_best')), 'B2': g.apply(lambda x: _rm(x, 'pred_B2'))})
    out['best/B2'] = (out.best / out.B2).round(2); out.index.name = label or col
    return out.round(4)
for col, lab in [('StoreTypeL', 'StoreType'), ('AssortmentL', 'Assortment'), ('DayOfWeek', 'Thứ (1=T2..7=CN)'), ('Promo', 'Promo'),
                 ('IsPromo2ActiveThisMonth', 'Promo2 đang active'), ('holiday', 'Ngày lễ nhà nước'), ('h_bucket', 'Ngày dự báo thứ h'),
                 ('gap_store', 'CH thiếu dữ liệu 2014'), ('has_ly364', 'Có ly364 (cùng kỳ năm trước)'), ('new_competitor', 'Đối thủ mới mở <= 6 tháng'),
                 ('recent_closure', 'Vừa đóng cửa dài (>=7 ngày) trước mốc T')]:
    print(f'--- Theo {lab}'); display(by_group(col, lab))

### 13.3 Đánh giá sai số tích lũy theo chiều dài khoảng dự báo (Cumulative Forecast Horizon Analysis)
Đánh giá độ chính xác của mô hình tốt nhất (pred_best) khi áp dụng cho các khoảng thời gian lập kế hoạch thực tế khác nhau: **7 ngày (1 tuần), 14 ngày (2 tuần), 28 ngày (4 tuần) và 48 ngày (toàn bộ chu kỳ)**. 

> **Làm rõ phương pháp luận & cơ chế đánh giá:**
> 1. Phép đo này tính toán **sai số RMSPE tích lũy thực tế** trực tiếp từ bảng VAL của mô hình tốt nhất (pred_best), theo công thức: $\text{RMSPE}(h \le H)$ với  \in \{7, 14, 28, 48\}$, được tính riêng cho từng fold rồi lấy trung bình qua 4 fold.
> 2. **Giải mã lý do 14 ngày đạt sai số thấp nhất:** Trong 14 ngày đầu, quán tính kinh doanh bám rất sát mốc chốt dữ liệu và đặc biệt là **chưa bị ảnh hưởng bởi tuần lễ Phục sinh** (vốn xảy ra ở tuần thứ 4 của Fold 3, đẩy sai số tuần đó lên 0.1356). Do đó, chu kỳ 14 ngày đạt độ chính xác cao nhất để tối ưu hóa việc đặt hàng và bổ sung tồn kho an toàn cho chuỗi cung ứng.


In [ ]:
# Đánh giá sai số RMSPE tích lũy theo từng mốc Horizon (Cumulative Forecast Horizon Analysis)
# Tính toán trực tiếp từ kết quả dự báo của mô hình tốt nhất (pred_best) trên tập validation qua 4 fold

horizons = {
    '7 Days (1 Week)': 7,
    '14 Days (2 Weeks)': 14,
    '28 Days (4 Weeks)': 28,
    '48 Days (Full Cycle)': 48
}

horizon_fold_scores = {}
horizon_scores = {}

for label, max_h in horizons.items():
    scores_per_fold = {}
    for f in FOLD_NAMES:
        mask = VAL.valid & (VAL.fold == f) & (VAL.h <= max_h)
        score = rmspe(VAL.loc[mask, 'Sales'].values, VAL.loc[mask, 'pred_best'])
        scores_per_fold[f] = score
    horizon_fold_scores[label] = scores_per_fold
    horizon_scores[label] = float(np.mean(list(scores_per_fold.values())))

# 1. Vẽ biểu đồ so sánh RMSPE theo Horizon
fig, ax = plt.subplots(figsize=(8.5, 4.4))
min_val = min(horizon_scores.values())
colors = ['#55A868' if v == min_val else '#4C72B0' for v in horizon_scores.values()]
bars = ax.bar(list(horizon_scores.keys()), list(horizon_scores.values()), width=0.52, color=colors)

ax.set_ylabel('Validation RMSPE (Mean 4 Folds)')
ax.set_title('Forecast Performance Across Cumulative Horizons', fontsize=11, fontweight='bold')
ax.set_ylim(0, max(horizon_scores.values()) * 1.25)
plt.grid(axis='y', alpha=0.3)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.002, f'{yval:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# 2. Hiển thị bảng chi tiết điểm từng fold và điểm trung bình
df_horizon = pd.DataFrame(horizon_fold_scores).T[FOLD_NAMES]
df_horizon['Mean RMSPE'] = list(horizon_scores.values())
print('Bảng chi tiết sai số RMSPE theo Horizon qua 4 fold:')
display(df_horizon.round(4))


### 13.4 Bias (lệch hệ thống) và mức độ tập trung của sai số
- **Bias**: tổng dự báo / tổng thực tế - 1 (dương = dự báo cao hơn thực tế).
- **Tập trung sai số:** RMSPE bình phương sai số phần trăm nên *vài dòng sai cực lớn* có thể chi phối toàn bộ điểm. Bảng dưới cho biết bao nhiêu % tổng bình phương sai số đến từ 0,5% dòng sai nhiều nhất.

In [ ]:
v = VAL[VAL.valid].copy(); v['pe'] = (v.pred_best - v.Sales) / v.Sales; v['se'] = v.pe ** 2
bias = v.groupby('fold').apply(lambda x: pd.Series({'Bias tổng (dự báo/thực tế - 1)': x.pred_best.sum() / x.Sales.sum() - 1,
                                                    'Trung vị sai số %': x.pe.median(), 'Trung vị |sai số|%': x.pe.abs().median()})).loc[FOLD_NAMES]
display((bias * 100).round(2).astype(str) + '%')
v_sorted = v.sort_values('se', ascending=False)
top_n = max(int(len(v) * 0.005), 1)
print(f'0,5% dòng sai nhiều nhất ({top_n} dòng) chiếm {v_sorted.se.head(top_n).sum() / v.se.sum() * 100:.1f}% tổng bình phương sai số.')
print(f'RMSPE nếu bỏ 0,5% dòng sai nhiều nhất: {np.sqrt(v_sorted.se.iloc[top_n:].mean()):.4f} (so với {np.sqrt(v.se.mean()):.4f} khi giữ nguyên)')

cols_show = ['fold', 'Store', 'Date', 'DayOfWeek', 'Promo', 'Sales', 'pred_best', 'pe', 'has_ly364', 'recent_closure', 'gap_store']
print('\n15 dòng sai nhiều nhất:'); display(v_sorted[cols_show].head(15).assign(pe=lambda d: (d.pe * 100).round(0).astype(int).astype(str) + '%').round(0))

st_err = v.groupby('Store').agg(n=('se', 'size'), RMSPE=('se', lambda s: np.sqrt(s.mean())), doanh_số_TB=('Sales', 'mean'), loại=('StoreTypeL', 'first'),
                                CH_thiếu_2014=('gap_store', 'max'), vừa_đóng_cửa=('recent_closure', 'max'))
print('\n15 cửa hàng sai nhiều nhất (>=100 dòng chấm):'); display(st_err[st_err.n >= 100].sort_values('RMSPE', ascending=False).head(15).round(3))

### 13.5 Quy sang đơn vị kinh doanh (phục vụ D02)
Điểm RMSPE khó hình dung với người làm vận hành. Hai chỉ số dễ hiểu hơn: **tỷ lệ ngày-cửa hàng dự báo sai trong ±10% / ±20%**, và **sai số khi gộp theo tuần** (đặt hàng thường theo tuần chứ không theo ngày).

In [ ]:
within = pd.DataFrame({'±10%': (v.pe.abs() <= 0.10).groupby(v.fold).mean(), '±20%': (v.pe.abs() <= 0.20).groupby(v.fold).mean()}).loc[FOLD_NAMES]
within.loc['Gộp 4 fold'] = [(v.pe.abs() <= 0.10).mean(), (v.pe.abs() <= 0.20).mean()]
print('Tỷ lệ dòng (ngày-cửa hàng) có sai số tuyệt đối trong ngưỡng:'); display((within * 100).round(1).astype(str) + '%')

v['week'] = v.Date.dt.to_period('W-SUN')
wk = v.groupby(['fold', 'Store', 'week']).agg(act=('Sales', 'sum'), pred=('pred_best', 'sum'), nd=('Sales', 'size')).reset_index()
wk = wk[wk.nd >= 5]                                        # chỉ tuần có >= 5 ngày mở cửa được chấm
wk['ape'] = (wk.pred - wk.act).abs() / wk.act
print(f"Gộp theo tuần (tuần có >=5 ngày mở cửa, {len(wk):,} tuần-cửa hàng): sai số % trung vị = {wk.ape.median()*100:.1f}% | trong ±10%: {(wk.ape <= 0.10).mean()*100:.1f}% | trong ±20%: {(wk.ape <= 0.20).mean()*100:.1f}%")

### 13.6 Biểu đồ thực tế vs dự báo

In [ ]:
f1 = v[v.fold == FOLD_NAMES[0]]
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].hexbin(np.log1p(f1.Sales), np.log1p(f1.pred_best), gridsize=60, bins='log', cmap='Blues')
lim = [np.log1p(f1.Sales).min(), np.log1p(f1.Sales).max()]
axs[0].plot(lim, lim, 'r--', lw=1)
axs[0].set_xlabel('Actual log(1+Sales)')
axs[0].set_ylabel('Predicted log(1+Sales)')
axs[0].set_title('Actual vs Predicted Sales')

axs[1].hist((v.pe * 100).clip(-60, 60), bins=80, color='#4472C4')
axs[1].axvline(0, color='r', ls='--')
axs[1].set_title('Percentage Error Distribution')
axs[1].set_xlabel('Percentage Error (%)')

g_ = by_group('StoreTypeL', 'StoreType')[['best', 'B2']].rename(columns={'best': 'Best Model', 'B2': 'B2 Baseline'})
g_.plot(kind='bar', ax=axs[2], rot=0)
axs[2].set_title('RMSPE by StoreType')
axs[2].set_ylabel('RMSPE')
axs[2].legend(fontsize=8)
plt.tight_layout()
plt.show()


### 13.7 Checklist "model có hợp lý không" (tự động, theo mục 10.2)

In [ ]:
b2_tab, best_tab = score_table(['B2 cùng thứ + Promo'], C_BEST).loc['B2 cùng thứ + Promo'], score_table([BEST_MODEL], C_BEST).loc[BEST_MODEL]
gain = best_tab['Trung bình'] / b2_tab['Trung bình'] - 1
h_ratio = hh['best'].loc[25:].mean() / hh['best'].loc[:24].mean()
checks = [
 ('1. Thắng baseline B2 ở CẢ 4 fold',                       all(best_tab[f] < b2_tab[f] for f in FOLD_NAMES), {f: f'{best_tab[f]:.4f} vs {b2_tab[f]:.4f}' for f in FOLD_NAMES}),
 ('2. Không "tốt đến đáng ngờ" (cải thiện < 50% so với B2)',  gain > -0.5,                                      f'cải thiện {abs(gain)*100:.1f}%'),
 ('3. Sai số không tăng theo ngày dự báo (nửa sau / nửa đầu < 1.15)', h_ratio < 1.15,                           f'tỷ lệ = {h_ratio:.3f}'),
 ('4. Trung vị sai số % mỗi fold nằm trong ±5% (không lệch hệ thống)', bool((bias.iloc[:, 1].abs() < 0.05).all()), bias.iloc[:, 1].round(4).to_dict()),
 ('5. Không có dự báo âm hoặc NaN',                           bool((VAL.pred_best >= 0).all() and VAL.pred_best.notna().all()), 'ok'),
 ('6. Không có biến bị cấm (Customers/Sales thô/Id) trong FEATURES', not (set(FEATURES) & FORBIDDEN),         'ok'),
 ('7. Độ lệch giữa các fold hợp lý (std/mean < 0.25)',        best_tab['Độ lệch chuẩn'] / best_tab['Trung bình'] < 0.25, f"{best_tab['Độ lệch chuẩn']/best_tab['Trung bình']:.3f}"),
]
for name_, ok_, detail in checks: print(('ĐẠT  ' if ok_ else 'CHƯA ĐẠT ') + name_ + '  ->  ' + str(detail))

### 13.8 Kiểm tra logic và độ tin cậy trước khi tin kết quả
Mục này trả lời thẳng câu hỏi *"pipeline đã thật sự ổn để huấn luyện 1 mô hình chung chưa?"* bằng các phép kiểm tra chạy được, không chỉ bằng lập luận.
1. **Căn chỉnh dữ liệu:** dự báo và Sales thật phải ứng đúng cùng một dòng (lỗi lệch dòng sẽ làm mọi con số vô nghĩa).
2. **Biến lịch sử có thật sự đóng góp không?** Bỏ toàn bộ biến lịch sử Sales rồi chạy lại: nếu điểm gần như không đổi thì phần "thông minh" của pipeline không có tác dụng.
3. **Cửa hàng ngoài test (259 CH) giúp hay hại?** Chỉ huấn luyện trên 856 cửa hàng của test rồi so sánh.
4. **"1 mô hình chung" có bỏ sót đặc thù từng cửa hàng không?** Xem sai số (lệch hệ thống) của từng cửa hàng có **lặp lại giữa các fold** không: nếu cùng một cửa hàng luôn bị dự báo cao (hoặc thấp) ở nhiều fold thì mô hình chung chưa bắt được đặc thù của cửa hàng đó.
5. **Vùng dự báo có hợp lý không:** dự báo của model so với mức bán thông thường của chính cửa hàng đó.

In [ ]:
# (1) căn chỉnh dòng giữa VAL, PRED và TB
for f_, T0_ in FOLD_ORIGINS.items():
    _, m_ = fold_masks(T0_); rows_ = np.where(m_)[0]
    assert np.array_equal(TB.Sales.values[rows_], VAL.Sales.values[ROWS[f_]], equal_nan=True) and np.array_equal(TB.Store.values[rows_], VAL.Store.values[ROWS[f_]]), f'Lệch dòng ở {f_}'
print('(1) Căn chỉnh dòng: ĐẠT (VAL khớp TB theo từng fold)')

# (2) và (3): ablation
HIST_FEATS = ['dow_promo_mean', 'dow_promo_n', 'roll28', 'roll91', 'open28', 'promo_uplift91', 'ly364', 'ly728', 'has_ly364', 'chain_delta']
FEAT_NOHIST = [f for f in FEATURES if f not in HIST_FEATS]
t0 = time.time()
ABL = {'Bộ biến đầy đủ (GB mặc định)': score_table(['M2 Gradient Boosting']).loc['M2 Gradient Boosting', FOLD_NAMES].to_dict(),
       'Bỏ toàn bộ biến lịch sử Sales': gb_fold_scores(FEAT_NOHIST),
       'Chỉ huấn luyện trên 856 CH của test': gb_fold_scores(FEATURES, train_stores=test_stores)}
abl_df = pd.DataFrame(ABL).T[FOLD_NAMES]; abl_df['Trung bình'] = abl_df.mean(axis=1)
abl_df['so với đầy đủ'] = ((abl_df['Trung bình'] / abl_df.loc['Bộ biến đầy đủ (GB mặc định)', 'Trung bình'] - 1) * 100).round(1).astype(str) + '%'
print(f'(2)(3) Ablation xong trong {time.time()-t0:.0f}s'); display(abl_df.round(4))

In [ ]:
# (4) sai số lệch hệ thống theo từng cửa hàng có lặp lại giữa các fold không?
vv = VAL[VAL.valid].copy(); vv['le'] = np.log(vv.pred_best / vv.Sales)                 # sai số log: > 0 = dự báo cao hơn thực tế
sb = vv.groupby(['Store', 'fold']).le.agg(['mean', 'size']).reset_index(); sb = sb[sb['size'] >= 20]
piv = sb.pivot(index='Store', columns='fold', values='mean')[FOLD_NAMES]
print('Độ lệch chuẩn của sai số log trung bình giữa các cửa hàng, theo fold:', piv.std().round(3).to_dict())
cm = piv.corr().round(2); cm.index = cm.columns = ['F1', 'F2', 'F3', 'F4']
print('Tương quan giữa sai số lệch hệ thống của cùng một cửa hàng ở các fold (gần 0 = không lặp lại; gần 1 = lặp lại):'); display(cm)
full4 = piv.dropna()
persist_pos = int(((full4 > 0.05).all(axis=1)).sum()); persist_neg = int(((full4 < -0.05).all(axis=1)).sum())
print(f'Cửa hàng có đủ 4 fold: {len(full4)} | dự báo CAO hơn thực tế >5% ở cả 4 fold: {persist_pos} | THẤP hơn >5% ở cả 4 fold: {persist_neg}')

# (5) dự báo so với mức bán thông thường của chính cửa hàng (trung vị lịch sử các ngày mở cửa trước mốc T)
ratios = []
for f_, T0_ in FOLD_ORIGINS.items():
    med_ = np.expm1(logS.loc[:T0_].median())                                          # trung vị theo cửa hàng (ngày mở cửa)
    d_ = vv[vv.fold == f_]; ratios.append((d_.pred_best / d_.Store.map(med_)).dropna())
ratios = pd.concat(ratios)
print(f'(5) Dự báo / trung vị lịch sử của chính cửa hàng: P1 = {ratios.quantile(.01):.2f}, trung vị = {ratios.median():.2f}, P99 = {ratios.quantile(.99):.2f} | ngoài khoảng [0.3, 3]: {((ratios < 0.3) | (ratios > 3)).mean()*100:.2f}% dòng')

**Nhận xét E02 (đọc kết quả ở trên):**
**Điểm trên validation (chưa phải điểm Kaggle):** GB tuned trung bình **0,1120** (F1 0,1110 / F2 0,1020 / F3 0,1191 / F4 0,1159) so với B2 là 0,1620 (cải thiện 30,9%). Loại các ngày lễ nhà nước (giống test hơn): **0,1111**. Cả 7 mục checklist đều ĐẠT. Các điểm này được dùng để **chọn mô hình, tham số và c trên chính 4 fold**, nên hơi lạc quan; điểm Kaggle mới là phép thử độc lập.

**Kiểm tra logic (mục 13.8) - điều quan trọng nhất cần đọc:**
- **Căn chỉnh dòng, rò rỉ ở 4 mốc, khóa duy nhất: đều ĐẠT.**
- **Biến lịch sử Sales đóng góp rất ít:** bỏ toàn bộ biến lịch sử thì RMSPE trung bình chỉ tệ hơn **0,8%** (0,1141 so với 0,1132). Theo fold: F2 tệ hơn rõ (0,1100 so với 0,1034), F1 và F4 tệ hơn nhẹ, còn **F3 lại tốt hơn khi bỏ** (0,1133 so với 0,1211). Nghĩa là phần lớn độ chính xác đến từ lịch, Promo, đặc điểm cửa hàng và `Store`, không phải từ các biến lag mà F02 đã thiết kế công phu. Lý do chưa rõ (khả năng: `Store` đã mã hóa mức nền cửa hàng, và NaN của `ly364`/`ly728` làm nhiễu); chưa kiểm chứng. Đáng thử tiếp: bộ biến lịch sử rút gọn.
- **Cửa hàng ngoài test không giúp:** chỉ huấn luyện trên 856 cửa hàng của test cho 0,1124 (tốt hơn 0,6%, khoảng 2 lần mức nhiễu). Chênh lệch nhỏ, nên chưa nên kết luận đổi cách huấn luyện, nhưng cũng cho thấy 259 cửa hàng kia không cần thiết.
- **"1 mô hình chung" không để lại lệch cố định theo cửa hàng:** tương quan sai số lệch hệ thống của cùng một cửa hàng giữa các fold chỉ từ **−0,05 đến 0,23**; trong 676 cửa hàng có đủ 4 fold, **0 cửa hàng** luôn bị dự báo cao >5% và **2 cửa hàng** luôn bị thấp >5%.
- **Dự báo nằm trong vùng hợp lý:** tỷ lệ dự báo / trung vị lịch sử của chính cửa hàng có P1 = 0,58, trung vị = 1,02, P99 = 1,73; chỉ 0,08% dòng nằm ngoài [0,3; 3].

**Model yếu ở đâu:**
- **Nhóm nhỏ nhưng khó:** ngày **Chủ nhật mở cửa** (722 dòng, 28 cửa hàng) RMSPE **0,206** (B2: 0,974); **ngày lễ nhà nước** (119 dòng) **0,463** (B2: 1,511). Test chỉ có 26 dòng mở cửa trong ngày lễ nên ảnh hưởng đến điểm Kaggle nhỏ.
- **Thiếu thông tin lịch sử:** dòng không có `ly364` 0,1219 (10.105 dòng) so với 0,1112 khi có; đối thủ mới mở ≤ 6 tháng 0,1266 (66 cửa hàng); cửa hàng vừa đóng cửa dài 0,1276 (chỉ 259 dòng, 7 cửa hàng).
- **180 cửa hàng thiếu dữ liệu 2014:** 0,1163 so với 0,1112 của nhóm còn lại, và đây là nhóm model cải thiện ít nhất so với B2 (tỷ lệ 0,86 so với 0,65). Nhóm này chiếm 21% dòng test.
- **Theo loại cửa hàng:** a 0,112 / b 0,094 / c 0,114 / d 0,113 (b chỉ 12 cửa hàng). Promo=1 (0,114) và Promo=0 (0,110) gần như ngang nhau.

**Theo ngày dự báo thứ h:** nửa sau cao hơn nửa đầu **2,7%** (0,1130 so với 0,1100), độ dốc +0,00026/ngày, **không tăng đều**: cao nhất ở h = 22-28 (0,1356) rồi giảm. B2 cũng cao nhất đúng đoạn này (0,222), nên có vẻ là đặc điểm dữ liệu của các fold, không phải model xấu đi theo thời gian; chưa kiểm chứng riêng.

**Bias:** tổng dự báo lệch thấp **−1,0% (F1), −3,8% (F2), −4,9% (F3), −1,1% (F4)**; trung vị sai số % lần lượt −0,9% / −3,4% / −3,6% / −0,2%. F1 và F4 (gần test nhất) lệch rất nhỏ; F2, F3 lệch thấp khoảng 3-5%. Một phần do `c = 0,99`.

**Sai số tập trung vào ít dòng:** 0,5% dòng sai nhiều nhất (647 dòng) chiếm **16,1%** tổng bình phương sai số; bỏ chúng thì RMSPE còn 0,1029 (so với 0,1120). 15 dòng sai nhiều nhất **đều là dự báo cao hơn thực tế từ 164% đến 340%**, ví dụ Store 685 ngày 08/09/2014 bán 1.128 nhưng dự báo 4.966. Có thể đó là những ngày bán thấp bất thường (đóng cửa một phần, hết hàng...) mà không biến nào mô tả được, nhưng đây là suy đoán, chưa kiểm chứng. Cửa hàng sai nhiều nhất: **415 (0,396), 909 (0,376), 183 (0,305)**; 4 cửa hàng trong top 15 thuộc nhóm thiếu dữ liệu 2014.

**Đơn vị kinh doanh:** **68,4%** ngày-cửa hàng sai trong ±10% và **94,0%** trong ±20%. Gộp theo tuần (tuần có ≥ 5 ngày mở, 21.998 tuần-cửa hàng): sai số % trung vị **4,2%**, **87,5%** trong ±10%, **98,3%** trong ±20%, phù hợp với việc đặt hàng theo tuần.

**File nộp Kaggle:** đã huấn luyện lại trên toàn bộ dữ liệu có nhãn đến 31/07/2015 (3 seed, 174 giây) và ghi `submission_F02_best_model.csv` (41.088 dòng, 5.984 dòng đóng cửa gán 0). Doanh số TB dự báo cho dòng mở cửa là **6.812**, thấp hơn mức 48 ngày cuối của train (**7.073**) khoảng 3,7%; chưa đủ cơ sở để nói đúng hay sai (test là tháng 8-9, 48 ngày cuối train là tháng 6-7). **Kỳ vọng hợp lý nhưng chưa phải đảm bảo:** nếu tổng quát hóa như validation thì điểm Kaggle quanh **0,102-0,119** (khoảng giữa các fold); điểm thực có thể nằm ngoài khoảng này vì test có 21% dòng thuộc 180 cửa hàng thiếu dữ liệu 2014 và chỉ có 1 ngày lễ.

---
## 14. Chuẩn bị nộp Kaggle (đánh giá trên tệp test đến SAU)
Sau khi chốt mô hình ở mục 12-13: **huấn luyện lại trên toàn bộ dữ liệu có nhãn đến 31/07/2015** (không có dòng test nào tham gia), dự báo 48 ngày test, và ghi file nộp. **Chưa có điểm số nào ở bước này** - điểm RMSPE thật chỉ có sau khi nộp lên Kaggle.

**Cách nộp (lưu ý):** Rossmann Store Sales là cuộc thi **nộp file dự báo (CSV), không phải nộp code**: Kaggle chỉ nhận file có hai cột `Id,Sales` và chấm RMSPE trên Sales thật mà nhóm không nhìn thấy. Notebook này chỉ là nơi **tạo ra** file đó. Cách nộp: vào trang cuộc thi trên Kaggle → *Submit Predictions* → tải lên `submission_F02_best_model.csv` (nếu cuộc thi đã đóng thì dùng *Late Submission* khi trang còn hiển thị nút này; nhóm nên kiểm tra trực tiếp trên trang cuộc thi).

In [ ]:
N_SEEDS = 1 if QUICK else 3
m_tr_all = TB.train_ok.values; m_te = (TB.is_test == 1).values
tr_all, te_all = TB.loc[m_tr_all], TB.loc[m_te]
t0 = time.time(); parts = []
if 'RF' in BEST_MODEL or 'Kết hợp' in BEST_MODEL:
    rf_pred = np.mean([RandomForestRegressor(**BEST_RF, n_jobs=N_JOBS, random_state=SEED + s).fit(rf_fill(tr_all), tr_all.y_log.values).predict(rf_fill(te_all)) for s in range(N_SEEDS)], axis=0)
    parts.append(('rf', rf_pred))
if 'Gradient' in BEST_MODEL or 'Kết hợp' in BEST_MODEL:
    gb_pred = np.mean([GB(BEST_GB, use_store=True, seed=SEED + s).fit(tr_all, tr_all.y_log.values).predict(te_all) for s in range(N_SEEDS)], axis=0)
    parts.append(('gb', gb_pred))
pred_log_test = np.mean([p for _, p in parts], axis=0)
pred_test = postprocess(pred_log_test, te_all.Open.values, BEST_C)
SUB = make_submission(te_all.Id.values, pred_test)
SUB.to_csv(OUT_DIR / 'submission_F02_best_model.csv', index=False)
print(f'Đã huấn luyện lại ({time.time()-t0:.0f}s, {N_SEEDS} seed) và ghi submission_F02_best_model.csv: {SUB.shape}')

# kiểm tra trước khi nộp
assert list(SUB.columns) == ['Id', 'Sales'] and set(SUB.Id) == set(test.Id) and SUB.Sales.notna().all() and (SUB.Sales >= 0).all()
tt = te_all.assign(pred=pred_test)
print('Dòng Open=0 được gán Sales=0:', bool((tt[tt.Open == 0].pred == 0).all()), '| số dòng Open=0:', int((tt.Open == 0).sum()))
last48 = TB[(TB.is_test == 0) & (TB.Date > TRAIN_END - pd.Timedelta(days=HORIZON)) & (TB.Open == 1) & (TB.Sales > 0) & TB.Store.isin(test_stores)]
print(f'Doanh số TB mỗi ngày-cửa hàng (dòng mở cửa): dự báo test = {tt[tt.Open == 1].pred.mean():,.0f} | 48 ngày cuối của train = {last48.Sales.mean():,.0f}')

**Khi đã nộp Kaggle**, ghi điểm vào biến bên dưới để so sánh với ước lượng trên validation:

In [ ]:
KAGGLE_RMSPE = None          # <- điền điểm RMSPE Kaggle trả về (ví dụ 0.123). Giữ None nếu chưa nộp.
lo, hi = best_tab[FOLD_NAMES].min(), best_tab[FOLD_NAMES].max()
print(f'Ước lượng trên validation: trung bình 4 fold = {best_tab["Trung bình"]:.4f}, khoảng {lo:.4f} - {hi:.4f} (F1 {best_tab[FOLD_NAMES[0]]:.4f}, F4 {best_tab[FOLD_NAMES[3]]:.4f})')
if KAGGLE_RMSPE is None: print('Chưa có điểm Kaggle -> chưa kết luận được model có tổng quát hóa tốt trên tệp test hay không.')
else:
    pos = 'TRONG' if lo <= KAGGLE_RMSPE <= hi else 'NGOÀI'
    print(f'Điểm Kaggle = {KAGGLE_RMSPE:.4f} -> {pos} khoảng validation; chênh so với trung bình validation: {KAGGLE_RMSPE - best_tab["Trung bình"]:+.4f}')

---
# PHẦN D - xAI: M03 (giải thích mô hình) và E03 (kiểm tra độ tin cậy của giải thích)

## 15. M03 - Giải thích mô hình Gradient Boosting đã chốt
**xAI trả lời câu hỏi gì?** Ở E02 ta biết model dự báo *tốt đến đâu*. xAI trả lời: model **dựa vào cái gì** để dự báo, **theo chiều nào** và **vì sao một dự báo cụ thể ra con số đó**.

**Hai lưu ý trước khi đọc bất kỳ kết quả nào:**
1. **xAI giải thích MÔ HÌNH.** Nếu model "dùng" biến Promo nhiều, điều đó nghĩa là *Promo giúp model dự báo*, không chứng minh *Promo làm tăng doanh số*.
2. **Giải thích trên dữ liệu validation, không phải trên dữ liệu đã học.** Trên dữ liệu đã học, model "thuộc lòng" nên mọi giải thích bị méo. Mọi con số dưới đây dùng **fold F1** (48 ngày gần test nhất), mô hình được huấn luyện chỉ với dữ liệu ≤ mốc T của fold.

Mô hình được giải thích: **Gradient Boosting tuned** (cấu hình `BEST_GB`, 38 biến, dùng `Store`), chưa nhân hệ số `c` (hệ số `c` chỉ là chỉnh nhỏ ở bước cuối).

### 15.0 Cấu hình và các hàm dùng chung

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time
from scipy import stats
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
assert HAS_LGB, 'Phần xAI dùng TreeSHAP có sẵn của LightGBM (pred_contrib) nên cần cài lightgbm'

XAI_QUICK = False                         # True = chạy nhanh để kiểm tra code (mẫu nhỏ, ít cây)
N_SHAP = 4000 if XAI_QUICK else 8000     # số dòng validation dùng để tính SHAP
N_PERM = 4000 if XAI_QUICK else 8000     # số dòng dùng cho permutation importance
N_REP  = 1    if XAI_QUICK else 2         # số lần xáo trộn cho mỗi biến / nhóm biến
FOLD_XAI = FOLD_NAMES[0]                  # F1: fold gần test nhất
XAI_PARAMS = dict(BEST_GB)
if XAI_QUICK: XAI_PARAMS['n_estimators'] = 60

FEATURE_GROUPS = {
 'Lịch sử Sales (mức nền, cùng kỳ năm trước)': ['dow_promo_mean', 'dow_promo_n', 'roll28', 'roll91', 'open28', 'promo_uplift91', 'ly364', 'ly728', 'has_ly364'],
 'Mùa vụ toàn chuỗi':       ['chain_delta'],
 'Định danh cửa hàng':      ['Store'],
 'Đặc điểm cửa hàng':       ['StoreType', 'Assortment'],
 'Promo ngắn hạn':          ['Promo', 'IsPromoWeek', 'Promo_prev7', 'Promo_next7'],
 'Lịch (thứ, tháng, tuần...)': ['DayOfWeek', 'Month', 'WeekOfYear', 'DayOfMonth', 'DaysToMonthEnd'],
 'Ngày lễ':                 ['StateHoliday', 'SchoolHoliday', 'DaysSinceLastHoliday', 'IsDayBeforeHoliday', 'IsDayAfterHoliday'],
 'Đối thủ':                 ['CompetitionDistance', 'CompetitionDistanceMissing', 'MonthsSinceCompetitionOpened', 'CompetitionSinceUnknown', 'CompetitorNotOpenYet'],
 'Promo2':                  ['Promo2', 'Promo2Started', 'IsPromo2ActiveThisMonth', 'Promo2WeeksSinceStart'],
 'Đóng cửa dài ngày':       ['DaysSinceReopen', 'LastClosureLen'],
}
assert sorted(sum(FEATURE_GROUPS.values(), [])) == sorted(FEATURES), 'FEATURE_GROUPS phải phủ đúng toàn bộ FEATURES'
GROUP_OF = {f: g for g, fs in FEATURE_GROUPS.items() for f in fs}
DOW_NAME = {1: 'T2', 2: 'T3', 3: 'T4', 4: 'T5', 5: 'T6', 6: 'T7', 7: 'CN'}

def fit_gb_df(tr, feats, params=None, seed=SEED):
    return GB(params or XAI_PARAMS, use_store=('Store' in feats), seed=seed, feats=feats).fit(tr, tr.y_log.values)

def fold_data(fold):
    # train (<= T) và validation (48 ngày sau T, cửa hàng của test, chỉ dòng được chấm: Open=1 và Sales>0)
    T0 = FOLD_ORIGINS[fold]; m_tr, m_va = fold_masks(T0); tr, va = TB.loc[m_tr], TB.loc[m_va]
    return tr, va[(va.Open == 1) & (va.Sales > 0)]

def score_pred(pred_log, df): return rmspe(df.Sales.values, postprocess(pred_log, df.Open.values))     # RMSPE thang gốc, c = 1

def perm_imp(predict, X, meta, units, n_rep=N_REP, seed=0):
    # Permutation importance: xáo trộn (đồng thời) các cột của từng "đơn vị" (biến hoặc nhóm biến) rồi đo RMSPE tăng bao nhiêu.
    r = np.random.default_rng(seed); base = score_pred(predict(X), meta); out = {}
    for u, cs in units.items():
        ds = []
        for _ in range(n_rep):
            Xp = X.copy(); idx = r.permutation(len(X))
            for c in cs: Xp[c] = X[c].values[idx]
            ds.append(score_pred(predict(Xp), meta) - base)
        out[u] = float(np.mean(ds))
    return base, pd.Series(out)

def shap_vals(model, X):
    # TreeSHAP có sẵn của LightGBM: mỗi dòng ra 1 giá trị cho từng biến (cộng lại + base = dự báo trên thang log)
    c = model.m.predict(X[model.feats], pred_contrib=True)
    return pd.DataFrame(c[:, :-1], columns=model.feats, index=X.index), c[:, -1]

t0 = time.time()
tr1, va1 = fold_data(FOLD_XAI)
M1 = fit_gb_df(tr1, FEATURES)
va1s = va1.sample(min(N_SHAP, len(va1)), random_state=SEED)
SV1, BASE1 = shap_vals(M1, va1s)
vperm = va1.sample(min(N_PERM, len(va1)), random_state=SEED + 1)
pred_fn = lambda d: M1.predict(d)
print(f'Đã huấn luyện GB trên fold {FOLD_XAI} ({len(tr1):,} dòng) và tính SHAP cho {len(va1s):,} dòng validation trong {time.time()-t0:.0f}s')
print(f'RMSPE của mô hình trên validation {FOLD_XAI} (c = 1): {score_pred(M1.predict(va1), va1):.4f}')
chk = SV1.sum(axis=1).values + BASE1; print('Kiểm tra tính cộng của SHAP (tổng SHAP + base = dự báo log): sai lệch tối đa =', float(np.abs(chk - M1.predict(va1s)).max()))

### 15.1 Chọn phương pháp xAI: so sánh các ứng viên & tác động khi Test không có Sales thật
Có nhiều cách giải thích; mỗi cách trả lời một câu hỏi khác nhau và có điểm yếu riêng. 

> **Lưu ý kiến trúc cốt lõi từ đặc thù đề bài Kaggle:** Tập `test.csv` hoàn toàn không có cột nhãn `Sales` (chỉ Kaggle chấm điểm ngầm). Điều này chia các phương pháp xAI làm 2 nhóm rõ rệt:
> 1. **Nhóm phụ thuộc hàm mất mát (Loss-based xAI như Permutation Importance, Worst-case Error Analysis):** Bắt buộc cần $Sales$ thật để tính $\Delta \text{RMSPE}$. Do đó **KHÔNG THỂ CHẠY TRÊN TẬP TEST**, bắt buộc phải neo 100% tại các tập Out-of-fold Validation (F1 đến F4).
> 2. **Nhóm phụ thuộc đầu ra mô hình (Output-based xAI như TreeSHAP, PDP, ICE, Beeswarm, Waterfall):** Chỉ cần mô hình $f(X)$ và ma trận thuộc tính $X$. Do đó **HOÀN TOÀN KHÔNG BỊ ẢNH HƯỞNG**, có thể chạy trên cả Test lẫn Validation để diễn giải kế hoạch kinh doanh tương lai.

| Phương pháp | Trả lời câu hỏi nào | Cần nhãn Sales thật? (Chạy được trên Test?) | Ưu điểm | Điểm yếu cần biết | Quyết định & Lý do sử dụng |
|---|---|:---:|---|---|---|
| **Importance có sẵn của cây (gain)** | Biến nào được cây dùng nhiều để chia? | Không cần (Chạy được trên Test) | Miễn phí, nhanh | Thiên vị biến có nhiều mức/giá trị (như `Store`); đo trên dữ liệu đã học, không phản ánh khả năng tổng quát | **Chỉ để so sánh** tham chiếu, không dùng làm kết luận chính |
| **Permutation importance** (theo biến & theo nhóm) | Nếu phá hỏng biến này thì RMSPE tệ đi bao nhiêu? | **BẮT BUỘC CẦN (KHÔNG CHẠY ĐƯỢC TRÊN TEST)** | Đo **trực tiếp trên metric RMSPE của bài toán**, dùng được cho mọi mô hình (RF/GB), tính chuẩn trên Validation | Biến tương quan chia nhau điểm → cần xáo theo nhóm; không chạy được trên tập Test thiếu nhãn | **CÓ (Chính)**: Đo mức độ quan trọng sống còn của biến trên tập Validation |
| **SHAP (TreeSHAP)** | Mỗi biến đẩy dự báo lên hay xuống bao nhiêu, ở từng dòng? | **Không cần** (Chạy tốt trên cả Test & Val) | Cho biết cả **chiều (+/-)** và **độ lớn**, giải thích chi tiết từng dòng, cộng lại đúng bằng dự báo $f(X)$ | Giải thích mô hình chứ không phải nhân quả ngoài đời; chỉ dành cho mô hình dạng cây | **CÓ (Chính)**: Đo chiều tác động, bóc tách động lực và giải thích từng dòng dự báo |
| **PDP / ICE** | Khi chỉ đổi một biến, dự báo đổi thế nào? | **Không cần** (Chạy tốt trên cả Test & Val) | Dễ hiểu cho cấp quản lý, hiển thị trực quan dạng hàm phi tuyến | PDP là trung bình nên có thể che khác biệt giữa các cửa hàng (ICE khắc phục điều này) | **CÓ (Chính)**: Kiểm tra tính hợp lý kinh doanh (Promo dốc lên, mở lại dốc xuống) |
| **Mô hình thay thế (surrogate)** | Xấp xỉ mô hình phức tạp bằng cây đơn giản | Không cần | Dễ vẽ cây quyết định cho slide trình bày | Chỉ là xấp xỉ gián tiếp, sai số lớn; thêm một tầng giả định không cần thiết | **LOẠI BỎ (Không dùng)**: Dễ gây sai lệch bản chất thuật toán |


In [ ]:
t0 = time.time()
Xp1 = vperm[FEATURES]
base_perm, perm_feat = perm_imp(pred_fn, Xp1, vperm, {f: [f] for f in FEATURES})
rank_gain = pd.Series(M1.m.booster_.feature_importance('gain'), index=M1.feats).reindex(FEATURES)
rank_shap = SV1.abs().mean().reindex(FEATURES)
RANKS = {'gain (cây)': rank_gain, 'SHAP (mean |SHAP|)': rank_shap, 'permutation (ΔRMSPE)': perm_feat}
top10 = pd.DataFrame({k: v.sort_values(ascending=False).head(10).index.tolist() for k, v in RANKS.items()}); top10.index = range(1, 11)
print(f'Permutation theo từng biến xong trong {time.time()-t0:.0f}s | RMSPE nền trên mẫu = {base_perm:.4f}')
print('Top 10 biến theo từng cách xếp hạng:'); display(top10)
sp = pd.DataFrame({a: {b: stats.spearmanr(RANKS[a], RANKS[b])[0] for b in RANKS} for a in RANKS}).round(2)
print('Tương quan thứ hạng (Spearman) giữa các cách xếp hạng trên 38 biến:'); display(sp)

**Nhận xét 15.1 (So sánh mức độ đồng thuận giữa các phương pháp):**
- **Ba cách xếp hạng có sự đồng thuận rất cao:** Spearman giữa gain và SHAP là **0,89**, giữa SHAP và permutation là **0,87**, giữa gain và permutation là **0,73**. Cả ba phương pháp đều xếp `dow_promo_mean` (mức nền doanh số cùng thứ, cùng trạng thái Promo) ở vị trí **số 1 tuyệt đối**.
- 4 biến cốt lõi gồm `dow_promo_mean`, `roll91`, `Store`, `Promo` đều xuất hiện vững chắc trong top-5 của cả ba bảng xếp hạng.
- **Quyết định phương pháp luận cốt lõi:**
  1. **Permutation theo nhóm biến (Mục 15.3):** Dùng làm thước đo chính để đo mức độ phụ thuộc sống còn của mô hình (đo trực tiếp trên metric RMSPE của bài toán).
  2. **TreeSHAP (Mục 15.4 - 15.5, 15.7):** Dùng để bóc tách chiều tác động (+/-), tính % đòn bẩy kinh doanh và giải thích chi tiết từng dòng dự báo.
  3. **PDP / ICE (Mục 15.6):** Dùng để kiểm tra hình dạng quan hệ phi tuyến và độ phân hóa giữa các nhóm cửa hàng.
  4. **Gain:** Chỉ dùng làm số liệu tham khảo nội bộ cấu trúc cây.

### 15.3 Mức độ quan trọng theo NHÓM biến
Biến tương quan chia nhau điểm quan trọng (ví dụ `Promo` và `IsPromoWeek`), nên cách đọc đúng là **theo nhóm có nghĩa kinh doanh**. Hai cách đo:
- **Permutation theo nhóm:** xáo trộn *đồng thời* mọi biến trong nhóm, đo RMSPE tăng bao nhiêu (cao = model phụ thuộc nhóm đó khi dự báo).
- **SHAP theo nhóm:** cộng SHAP của các biến trong nhóm ở từng dòng rồi lấy trung bình giá trị tuyệt đối (đơn vị: log, ≈ % dự báo bị đẩy lên/xuống).

In [ ]:
base_g, perm_grp = perm_imp(pred_fn, Xp1, vperm, FEATURE_GROUPS)
GROUP_TAB = pd.DataFrame({'ΔRMSPE': perm_grp, '% of Baseline RMSPE': perm_grp / base_g * 100,
                          'Feature Count': {g: len(fs) for g, fs in FEATURE_GROUPS.items()}}).sort_values('ΔRMSPE', ascending=False)
display(GROUP_TAB.round(4))

GROUP_EN = {
    'Lịch sử Sales (mức nền, cùng kỳ năm trước)': 'Sales History',
    'Lịch (thứ, tháng, tuần...)': 'Calendar',
    'Định danh cửa hàng': 'Store Identity',
    'Promo ngắn hạn': 'Short-term Promo',
    'Ngày lễ': 'Holidays',
    'Mùa vụ toàn chuỗi': 'Chain Trend',
    'Promo2': 'Promo2',
    'Đối thủ': 'Competition',
    'Đóng cửa dài ngày': 'Store Closures',
    'Đặc điểm cửa hàng': 'Store Characteristics'
}
s_perm = GROUP_TAB['ΔRMSPE'].rename(index=GROUP_EN).iloc[::-1]

fig, ax = plt.subplots(figsize=(8.5, 4.4))
s_perm.plot(kind='barh', ax=ax, color='#4472C4')
ax.set_title('Permutation Feature Importance (Level 1)')
ax.set_xlabel('ΔRMSPE when permuting feature group')
ax.set_xlim(0, s_perm.max() * 1.18)
for p in ax.patches:
    w = p.get_width()
    ax.annotate(f'+{w:.4f}', (w, p.get_y() + p.get_height() / 2), xytext=(5, 0), textcoords='offset points', va='center', fontsize=8.5, fontweight='bold')
plt.tight_layout()
plt.show()


### 15.4 Biểu đồ SHAP tổng quát (beeswarm) - biến nào đẩy dự báo lên hay xuống?
Mỗi chấm là một dòng validation. **Vị trí ngang** = biến đó đẩy dự báo (thang log) lên (phải) hay xuống (trái) bao nhiêu; **màu** = giá trị của biến (đỏ = cao, xanh = thấp).

In [ ]:
# Lọc bỏ nhóm Lịch sử Sales để tập trung hoàn toàn vào các biến Đòn bẩy Điều hành (Actionable Levers)
sales_hist = FEATURE_GROUPS['Lịch sử Sales (mức nền, cùng kỳ năm trước)']
actionable_feats = [f for f in rank_shap.sort_values(ascending=False).index if f not in sales_hist][:12]

# 1. Bar Chart: Mean Absolute SHAP Value
fig, ax = plt.subplots(figsize=(8.5, 4.6))
s_top = rank_shap.loc[actionable_feats].iloc[::-1]
s_top.plot(kind='barh', ax=ax, color='#ED7D31')
ax.set_title('Mean Absolute SHAP Value')
ax.set_xlabel('mean |SHAP| (log scale)')
ax.set_xlim(0, s_top.max() * 1.18)
for p in ax.patches:
    w = p.get_width()
    ax.annotate(f'{w:.4f}', (w, p.get_y() + p.get_height() / 2), xytext=(5, 0), textcoords='offset points', va='center', fontsize=8.5, fontweight='bold')
plt.tight_layout()
plt.show()

# 2. Beeswarm Plot: SHAP Beeswarm Summary
fig, ax = plt.subplots(figsize=(10.5, 6))
rj = np.random.default_rng(1)
top_f = actionable_feats[::-1]
for i, f in enumerate(top_f):
    v = va1s[f].astype(float); pct = v.rank(pct=True).fillna(0.5).values
    ax.scatter(SV1[f].values, i + rj.uniform(-0.28, 0.28, len(v)), c=pct, cmap='coolwarm', s=2.5, alpha=0.55)
ax.set_yticks(range(len(top_f)))
ax.set_yticklabels(top_f, fontsize=9.5)
ax.axvline(0, color='k', lw=0.9)
ax.set_xlabel('SHAP Value', fontsize=10)
ax.set_title('SHAP Beeswarm Summary', fontsize=11, fontweight='bold')
ax.set_xlim(-0.23, 0.27)
cbar = plt.colorbar(plt.cm.ScalarMappable(cmap='coolwarm'), ax=ax, fraction=0.03, pad=0.02)
cbar.set_ticks([0.05, 0.95])
cbar.set_ticklabels(['Low', 'High'])
cbar.set_label('Feature Value')
plt.tight_layout()
plt.show()


### 15.5 Chiều và độ lớn của các tác động chính
Đọc SHAP ở dạng **% chênh lệch**: vì dự báo ở thang log nên `exp(hiệu SHAP) - 1` là chênh lệch phần trăm giữa hai trạng thái, **trong mô hình**, giữ các biến khác như nhau. Đây là *"model dự báo cao hơn bao nhiêu"*, không phải *"Promo làm doanh số tăng bao nhiêu"*.

In [ ]:
X1s = va1s[FEATURES]
promo_cols = FEATURE_GROUPS['Promo ngắn hạn']
g_promo = SV1[promo_cols].sum(axis=1)
eff_p = g_promo.groupby(X1s.Promo.values).mean()
print(f'Promo: nhóm biến Promo đẩy dự báo của ngày có Promo cao hơn ngày không Promo: {np.expm1(eff_p[1] - eff_p[0]) * 100:+.1f}% (SHAP nhóm; mẫu {len(X1s):,} dòng)')
print(f"      riêng biến Promo: {np.expm1(SV1['Promo'].groupby(X1s.Promo.values).mean().diff().iloc[-1]) * 100:+.1f}%")
byT = pd.DataFrame({'SHAP nhóm Promo': g_promo.values, 'Promo': X1s.Promo.values, 'StoreType': va1s.StoreType.map(dict(enumerate(CATS['StoreType']))).values})
t_ = byT.groupby(['StoreType', 'Promo'])['SHAP nhóm Promo'].mean().unstack(); t_['% chênh (Promo=1 so với 0)'] = (np.expm1(t_[1] - t_[0]) * 100).round(1); t_['n dòng'] = byT.groupby('StoreType').size()
print('\nTheo StoreType (nhóm n nhỏ chỉ nêu như trường hợp cá biệt):'); display(t_.round(3))
dow = SV1['DayOfWeek'].groupby(X1s.DayOfWeek.values).mean(); dow_n = X1s.groupby('DayOfWeek').size()
dowdf = pd.DataFrame({'thứ': [DOW_NAME[k] for k in dow.index], 'SHAP trung bình (log)': dow.values, '% so với TB các thứ': (np.expm1(dow.values - np.average(dow.values, weights=dow_n.loc[dow.index].values)) * 100), 'n dòng': dow_n.loc[dow.index].values})
print('\nTác động của thứ trong tuần (biến DayOfWeek):'); display(dowdf.round(3))
for f_, lab in [('IsPromo2ActiveThisMonth', 'Promo2 đang active'), ('Promo2', 'cửa hàng có Promo2')]:
    e = SV1[f_].groupby(X1s[f_].values).mean()
    if len(e) > 1: print(f'{lab}: SHAP của biến {f_} khác nhau giữa 1 và 0 là {np.expm1(e.get(1, np.nan) - e.get(0, np.nan)) * 100:+.2f}% (rất nhỏ nghĩa là model gần như không dùng)')
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, f_ in zip(axs, ['dow_promo_mean', 'chain_delta', 'MonthsSinceCompetitionOpened']):
    ax.scatter(va1s[f_], SV1[f_], s=3, alpha=0.3); ax.set_xlabel(f_); ax.set_ylabel('SHAP Value (log)'); ax.set_title(f'{f_} SHAP Dependence')
plt.tight_layout(); plt.show()


### 15.6 PDP / ICE: nếu chỉ đổi một biến thì dự báo đổi thế nào?
Với 2.000 dòng validation: mỗi đường mờ (**ICE**) là một dòng, giữ nguyên mọi thứ, chỉ thay giá trị một biến; đường đậm (**PDP**) là trung bình. Trục dọc là *% thay đổi dự báo so với giá trị đầu tiên của lưới*.

In [ ]:
def pdp_ice(model, X, feat, grid, n_ice=80, seed=0):
    r = np.random.default_rng(seed); preds = []
    for g_ in grid:
        Xg = X.copy(); Xg[feat] = g_; preds.append(model.predict(Xg))
    P = np.array(preds).T                                  # dòng x lưới, thang log
    rel = np.expm1(P - P[:, [0]]) * 100                    # % thay đổi so với điểm đầu lưới
    return rel.mean(axis=0), rel[r.choice(len(rel), min(n_ice, len(rel)), replace=False)]
Xpd = va1.sample(min(2000, len(va1)), random_state=3)[FEATURES]
specs = [('Promo', [0, 1]), ('DayOfWeek', [1, 2, 3, 4, 5, 6]), ('MonthsSinceCompetitionOpened', list(np.round(np.nanpercentile(TB.MonthsSinceCompetitionOpened, np.linspace(5, 95, 12)), 1))),
         ('CompetitionDistance', list(np.round(np.nanpercentile(TB.CompetitionDistance, np.linspace(5, 95, 12)), 0))), ('chain_delta', list(np.round(np.nanpercentile(TB.chain_delta, np.linspace(5, 95, 12)), 3))),
         ('DaysSinceReopen', [0, 7, 14, 30, 60, 90, 120])]
fig, axs = plt.subplots(2, 3, figsize=(15, 7.5)); PDP = {}
for ax, (f_, grid) in zip(axs.ravel(), specs):
    pdp, ice = pdp_ice(M1, Xpd, f_, grid); PDP[f_] = pd.Series(pdp, index=grid)
    for line in ice: ax.plot(grid, line, color='#4472C4', alpha=0.12, lw=0.8)
    ax.plot(grid, pdp, color='#C00000', lw=2.5, marker='o', ms=5); ax.axhline(0, color='k', lw=0.6); ax.set_title(f_); ax.set_ylabel('% Prediction Change')
    # Gắn Data Labels cho các điểm quan trọng
    if f_ == 'Promo':
        for x_val, y_val in zip(grid, pdp):
            ax.annotate(f'{y_val:+.1f}%', (x_val, y_val), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9, fontweight='bold', color='#C00000')
    elif f_ == 'DayOfWeek':
        for x_val, y_val in zip(grid, pdp):
            ax.annotate(f'{y_val:+.1f}%', (x_val, y_val), textcoords='offset points', xytext=(0, 8 if y_val >= 0 else -14), ha='center', fontsize=8.5, fontweight='bold', color='#C00000')
    elif f_ == 'DaysSinceReopen':
        for x_val, y_val in zip(grid[:4], pdp[:4]):
            ax.annotate(f'{y_val:.1f}%', (x_val, y_val), textcoords='offset points', xytext=(0, 8 if y_val >= -20 else -14), ha='center', fontsize=8.5, fontweight='bold', color='#C00000')
plt.tight_layout(); plt.show()
print('PDP (% thay đổi dự báo so với giá trị đầu lưới):'); 
for f_, s_ in PDP.items(): print(f'  {f_:<30}', {k: round(float(v), 1) for k, v in s_.items()})


### 15.7 Giải thích từng dự báo cụ thể
Chọn 2 dòng validation của fold F1: **điển hình** (sai số gần trung vị) và **sai nhiều nhất**. (Ngày lễ được xem riêng ở mục 16.5 vì F1 không có ngày lễ.) Bảng cho biết dự báo "xuất phát" từ mức nền (base) rồi mỗi biến cộng/trừ bao nhiêu (thang log, nhân lại bằng `exp` ra %).

In [ ]:
pe1 = (np.expm1(M1.predict(va1)) - va1.Sales.values) / va1.Sales.values
rows_sel = {'Điển hình (sai số gần trung vị)': va1.index[np.argmin(np.abs(np.abs(pe1) - np.median(np.abs(pe1))))], 'Sai nhiều nhất': va1.index[np.argmax(np.abs(pe1))]}
def explain_row(model, df, idx, title):
    X = df.loc[[idx], FEATURES]; sv, base = shap_vals(model, X); pred_log = float(model.predict(X)[0])
    s = sv.iloc[0].sort_values(key=np.abs, ascending=False).head(8)
    print(f'--- {title}: Store {int(df.loc[idx, "Store"])}, {df.loc[idx, "Date"].date()}, thứ {DOW_NAME[int(df.loc[idx, "DayOfWeek"])]}, Promo={int(df.loc[idx, "Promo"])}')
    print(f'    Thực tế {df.loc[idx, "Sales"]:,.0f} | dự báo {np.expm1(pred_log):,.0f} | mức nền (trung bình của model) {np.expm1(base[0]):,.0f}')
    display(pd.DataFrame({'giá trị biến': df.loc[idx, s.index].values, 'đóng góp (log)': s.values, '% so với mức hiện tại': (np.expm1(s.values) * 100)}, index=s.index).round(3))
for t_, i_ in rows_sel.items(): explain_row(M1, va1, i_, t_)

### 15.8 Góc nhìn thứ hai: xem "động lực" khi đã trừ mức nền của cửa hàng
Biến lịch sử và `Store` mã hóa "cửa hàng này vốn bán nhiều hay ít" nên **chiếm gần hết điểm quan trọng**, che mất các yếu tố kinh doanh khác. Để xem rõ các động lực (Promo, thứ, mùa vụ, đối thủ...), mình huấn luyện thêm **một mô hình phụ** dự báo *phần chênh so với mức nền* `y_log − dow_promo_mean`, chỉ dùng các biến **ngoài nhóm lịch sử và ngoài `Store`**, rồi giải thích mô hình phụ đó. **Đây là góc nhìn thứ hai, không thay thế mô hình chính**; hai góc nhìn cần được báo cáo cùng nhau.

In [ ]:
t0 = time.time()
DRV_FEATS = [f for f in FEATURES if GROUP_OF[f] not in ('Lịch sử Sales (mức nền, cùng kỳ năm trước)', 'Định danh cửa hàng')]
def offset(df): return df.dow_promo_mean.fillna(df.roll91).fillna(df.roll28).fillna(GLOBAL_MED).values
trd = tr1.copy(); trd['y_res'] = trd.y_log.values - offset(trd)
drv = GB(XAI_PARAMS, use_store=False, seed=SEED, feats=DRV_FEATS).fit(trd.assign(y_log=trd['y_res']), trd['y_res'].values)
pred_drv = offset(va1) + drv.predict(va1)
print(f'Mô hình phụ (offset + phần chênh) trên validation {FOLD_XAI}: RMSPE = {score_pred(pred_drv, va1):.4f} | mô hình chính: {score_pred(M1.predict(va1), va1):.4f} | chỉ offset (B3): {score_pred(offset(va1), va1):.4f}  ({time.time()-t0:.0f}s)')
SVd, _ = shap_vals(drv, va1s)
drv_rank = SVd.abs().mean().sort_values(ascending=False)
drv_grp = pd.DataFrame({g: SVd[[f for f in fs if f in DRV_FEATS]].sum(axis=1) for g, fs in FEATURE_GROUPS.items() if any(f in DRV_FEATS for f in fs)}).abs().mean().sort_values(ascending=False)
print('Biến "động lực" quan trọng nhất (mean |SHAP|, log):'); display(drv_rank.head(10).round(4).to_frame('mean |SHAP|'))
print('Theo nhóm:'); display(drv_grp.round(4).to_frame('mean |SHAP nhóm|'))

**Nhận xét M03 (kết quả giải thích):**
1. **Mô hình dự báo chủ yếu bằng "mức nền lịch sử" của chính cửa hàng.** Nhóm lịch sử Sales làm RMSPE tăng **0,457** khi bị xáo trộn (+420% so với RMSPE nền 0,109), gấp hơn 12 lần nhóm đứng sau (Lịch 0,036; Định danh cửa hàng 0,032; Promo ngắn hạn 0,029). Biến đơn quan trọng nhất là `dow_promo_mean` (mean |SHAP| 0,199, gấp khoảng 6 lần biến thứ hai `roll91` 0,035).
2. **Điều này KHÔNG mâu thuẫn với E02** (ở E02, bỏ biến lịch sử rồi huấn luyện lại chỉ tệ hơn 0,8%). Hai phép đo trả lời hai câu hỏi khác nhau: permutation đo *model đã học đang dựa vào biến đó đến đâu* (không cho học lại); ablation đo *nếu không có biến đó, model học lại từ thứ khác được bao nhiêu*. Kết quả gợi ý thông tin của biến lịch sử phần lớn **trùng với thứ khác, nhiều khả năng là `Store`** (nhóm Định danh cửa hàng cũng đứng thứ ba, 0,032). Giải thích này hợp lý nhưng **chưa kiểm chứng trực tiếp**.
3. **Promo:** nhóm biến Promo đẩy dự báo ngày có Promo cao hơn ngày không Promo khoảng **+9,4%** (riêng biến `Promo` +6,3%; PDP khi chỉ đổi cờ Promo: +7,1%), gần như giống nhau giữa các loại cửa hàng (a 9,4%, b 6,7% với n = 119, c 9,6%, d 9,4%). **Con số này thấp hơn nhiều so với +41,4% trong dữ liệu thô** (E03). Lý do hợp lý: phần lớn tác động Promo đã nằm sẵn trong `dow_promo_mean` (trung bình của các ngày *cùng trạng thái Promo*), nên SHAP chia công cho hai biến. Vì vậy +9,4% đọc là "phần Promo bổ sung ngoài mức nền đã ghép Promo", **không phải tổng tác động của Promo** (và không phải nhân quả).
4. **Thứ trong tuần:** T2 cao nhất (+4,1% so với trung bình các thứ), T3 gần 0, T4-T7 thấp hơn 0,7-1,4%, Chủ nhật −3,9% (chỉ 39 dòng).
5. **Promo2 và đối thủ gần như không được dùng:** SHAP của `IsPromo2ActiveThisMonth` khác nhau giữa 1 và 0 chỉ −0,50%, của `Promo2` −0,01%; nhóm Promo2 làm RMSPE tăng chỉ 0,0019 và nhóm Đối thủ 0,0008; PDP của `CompetitionDistance` phẳng (0-0,5%). Với mục tiêu dự báo, hai nhóm này không cần thiết (phù hợp EDA: khoảng cách đối thủ gần như không liên quan Sales).
6. **Đóng cửa dài ngày (R15):** PDP cho thấy model học được hiệu ứng sau khi mở lại: dự báo thấp hơn **46,7%** sau 7 ngày và **57%** sau 30 ngày so với đúng ngày mở lại. Nhưng nhóm này rất hiếm (ΔRMSPE chỉ 0,0008) nên ảnh hưởng đến điểm số nhỏ; PDP cũng có thể tạo tổ hợp phi thực tế khi đặt "ngày mở lại" cho dòng không hề vừa mở lại.
7. **`chain_delta` (R16):** quan trọng thấp (mean |SHAP| 0,011, hạng 9) và PDP cho chiều **ngược trực giác** (giá trị cao hơn thì dự báo thấp hơn, tối đa −2,3%). Chưa giải thích được; **đừng diễn giải nó là "mùa vụ" khi trình bày**.
8. **Vị trí trong tháng:** `DaysToMonthEnd` (0,021) và `DayOfMonth` (0,013) nằm trong top 8 biến, và ở góc nhìn "động lực" `DayOfMonth` là biến số 1. Gợi ý có hiệu ứng đầu/cuối tháng (ví dụ ngày nhận lương), nhưng đây là **suy đoán, chưa kiểm chứng bằng EDA**.
9. **Giải thích hai dòng cụ thể:** dòng điển hình (Store 723, 21/07/2015) thực tế 6.706, dự báo 6.271; không biến nào áp đảo (đóng góp lớn nhất chỉ ±3,5%). Dòng sai nhiều nhất (Store 909, 01/07/2015) thực tế **3.547**, dự báo **13.837**: `dow_promo_mean` đẩy dự báo lên +57%, `ly364` bị thiếu (cửa hàng thuộc nhóm mất dữ liệu 2014). Model tin vào mức nền lịch sử cao còn thực tế bán thấp bất thường, và **không biến nào cho biết vì sao** (khớp với suy đoán ở E02 về những ngày bất thường).
10. **Góc nhìn thứ hai (mô hình phụ trừ mức nền):** RMSPE 0,1238, tốt hơn chỉ-dùng-mức-nền (B3: 0,1453) khoảng 15% nhưng kém mô hình chính (0,1127) khoảng 10%. "Động lực" xếp theo nhóm: **Lịch** (0,0415), Ngày lễ (0,0183), Đối thủ (0,0085), Mùa vụ chuỗi (0,0078), Promo2 (0,0055). Nhóm Promo ngắn hạn chỉ 0,0025 vì Promo đã nằm trong mức nền (xem điểm 3). **Hai góc nhìn cần được báo cáo cùng nhau**, không thay thế nhau.

---
## 16. E03 - Kiểm tra độ tin cậy của giải thích
**Vì sao cần bước này?** Một giải thích chỉ có giá trị nếu nó **ổn định**: đổi fold, đổi hạt giống, đổi mô hình mà kết luận đảo ngược thì kết luận đó là nhiễu. Mình kiểm tra 5 điều (tiêu chí đạt ghi cạnh từng mục):

| # | Kiểm tra | Tiêu chí đạt |
|---|---|---|
| 1 | Ổn định giữa 4 fold (giải thích trên 4 mô hình huấn luyện ở 4 mốc T khác nhau) | top-5 biến trùng ≥ 4/5 với F1; Spearman thứ hạng ≥ 0,8; chiều tác động Promo không đổi dấu |
| 2 | Ổn định giữa các hạt giống | Spearman ≥ 0,9 |
| 3 | Ổn định giữa 2 mô hình khác loại (Gradient Boosting và Random Forest) | Spearman thứ hạng nhóm ≥ 0,7 |
| 4 | **Cảm biến nhiễu:** thêm một biến ngẫu nhiên (vô nghĩa) vào mô hình | Biến nhiễu phải nằm cuối bảng xếp hạng |
| 5 | **Đối chiếu với dữ liệu thô:** chiều/độ lớn tác động Promo mà model "dùng" so với % chênh trực tiếp trong dữ liệu | Cùng chiều, cùng bậc độ lớn |
(Các ngưỡng là quy ước do nhóm đặt ra, không phải chuẩn khoa học.)

In [ ]:
t0 = time.time(); FOLD_MODELS = {FOLD_XAI: (M1, va1, SV1, va1s, vperm)}
for fold in FOLD_NAMES[1:]:
    tr_f, va_f = fold_data(fold); m_f = fit_gb_df(tr_f, FEATURES)
    vs_f = va_f.sample(min(N_SHAP, len(va_f)), random_state=SEED); vp_f = va_f.sample(min(N_PERM, len(va_f)), random_state=SEED + 1)
    FOLD_MODELS[fold] = (m_f, va_f, shap_vals(m_f, vs_f)[0], vs_f, vp_f)
print(f'Đã huấn luyện và giải thích {len(FOLD_MODELS)} mô hình (mỗi fold một mô hình) trong {time.time()-t0:.0f}s')

fold_shap = pd.DataFrame({f: v[2].abs().mean() for f, v in FOLD_MODELS.items()}).reindex(FEATURES)
fold_perm_grp = pd.DataFrame({f: perm_imp(lambda d, m=v[0]: m.predict(d), v[4][FEATURES], v[4], FEATURE_GROUPS)[1] for f, v in FOLD_MODELS.items()})
print('mean |SHAP| theo biến - top 10 của từng fold:'); display(pd.DataFrame({f: fold_shap[f].sort_values(ascending=False).head(10).index.tolist() for f in FOLD_NAMES}, index=range(1, 11)))
print('Permutation theo nhóm (ΔRMSPE) theo fold:'); display(fold_perm_grp.round(4))
sp_f = pd.DataFrame({a: {b: stats.spearmanr(fold_shap[a], fold_shap[b])[0] for b in FOLD_NAMES} for a in FOLD_NAMES}).round(2)
sp_f.index = sp_f.columns = ['F1', 'F2', 'F3', 'F4']; print('Spearman thứ hạng mean |SHAP| (38 biến) giữa các fold:'); display(sp_f)
spg = pd.DataFrame({a: {b: stats.spearmanr(fold_perm_grp[a], fold_perm_grp[b])[0] for b in FOLD_NAMES} for a in FOLD_NAMES}).round(2)
spg.index = spg.columns = ['F1', 'F2', 'F3', 'F4']; print('Spearman thứ hạng NHÓM (permutation) giữa các fold:'); display(spg)
top5 = {f: set(fold_shap[f].sort_values(ascending=False).head(5).index) for f in FOLD_NAMES}
overlap = {f: len(top5[f] & top5[FOLD_XAI]) for f in FOLD_NAMES}; print('Số biến trùng trong top-5 so với F1:', overlap)
promo_eff = {}
for f, (m_, vf, sv_, vs_, vp_) in FOLD_MODELS.items():
    gp = sv_[promo_cols].sum(axis=1); e = gp.groupby(vs_.Promo.values).mean(); promo_eff[f] = float(np.expm1(e[1] - e[0]) * 100)
print('Tác động Promo mà model dùng (SHAP nhóm Promo, Promo=1 so với 0) theo fold (%):', {k: round(v, 1) for k, v in promo_eff.items()})
dow_eff = pd.DataFrame({f: v[2]['DayOfWeek'].groupby(v[3].DayOfWeek.values).mean() for f, v in FOLD_MODELS.items()})
dow_corr = {f: round(float(dow_eff[[FOLD_XAI, f]].dropna().corr().iloc[0, 1]), 2) for f in FOLD_NAMES[1:]}
print('Tương quan mẫu hình "thứ trong tuần" (SHAP của DayOfWeek) giữa các fold so với F1:', dow_corr)

### 16.2 Ổn định giữa các hạt giống và giữa hai loại mô hình

In [ ]:
t0 = time.time(); seed_shap = {SEED: rank_shap}
for sd in ([SEED + 1] if XAI_QUICK else [SEED + 1, SEED + 2]):
    ms = fit_gb_df(tr1, FEATURES, seed=sd); seed_shap[sd] = shap_vals(ms, va1s)[0].abs().mean().reindex(FEATURES)
print(f'Huấn luyện thêm {len(seed_shap)-1} hạt giống trong {time.time()-t0:.0f}s')
sp_seed = {sd: round(float(stats.spearmanr(seed_shap[SEED], seed_shap[sd])[0]), 3) for sd in seed_shap if sd != SEED}
ov_seed = {sd: len(set(seed_shap[SEED].sort_values(ascending=False).head(5).index) & set(seed_shap[sd].sort_values(ascending=False).head(5).index)) for sd in seed_shap if sd != SEED}
print('Spearman mean |SHAP| so với seed gốc:', sp_seed, '| số biến trùng top-5:', ov_seed)

# Random Forest (không có Store, điền NaN theo rf_fill) so với Gradient Boosting: so sánh theo NHÓM (không gồm 'Định danh cửa hàng')
t0 = time.time(); grp_nostore = {g: [f for f in fs if f != 'Store'] for g, fs in FEATURE_GROUPS.items() if any(f != 'Store' for f in fs)}
cmp_models = {}
for fold in [FOLD_NAMES[0], FOLD_NAMES[3]]:
    tr_f, va_f = fold_data(fold); rf = RandomForestRegressor(**BEST_RF, n_jobs=N_JOBS, random_state=SEED).fit(rf_fill(tr_f), tr_f.y_log.values)
    vp_f = FOLD_MODELS[fold][4]; gb_f = FOLD_MODELS[fold][0]
    pg = perm_imp(lambda d: gb_f.predict(d), vp_f[FEATURES], vp_f, grp_nostore)[1]
    pr = perm_imp(lambda d: rf.predict(rf_fill(d)), vp_f[NOSTORE], vp_f, grp_nostore)[1]
    cmp_models[fold] = pd.DataFrame({'GB': pg, 'RF': pr})
    print(f'Fold {fold}: Spearman thứ hạng nhóm GB vs RF = {stats.spearmanr(pg, pr)[0]:.2f}')
print(f'(RF xong trong {time.time()-t0:.0f}s)'); display(pd.concat(cmp_models, axis=1).round(4))

### 16.3 Cảm biến nhiễu và đối chiếu với dữ liệu thô
**Cảm biến nhiễu:** thêm một cột số ngẫu nhiên (không liên quan gì đến Sales) vào mô hình. Nếu cách giải thích "tử tế" thì cột này phải xếp **gần cuối** (SHAP gần 0, permutation gần 0). Nếu nó xếp cao nghĩa là phương pháp (hoặc model) đang tạo ra "quan trọng giả".

**Đối chiếu dữ liệu thô:** tính trực tiếp từ dữ liệu validation: trong cùng một cửa hàng và cùng một thứ, doanh số ngày có Promo cao hơn ngày không Promo bao nhiêu %. So với con số mà model "dùng" (SHAP nhóm Promo). Chỉ so sánh *chiều và bậc độ lớn*, vì hai con số không đo cùng một thứ (model giữ các biến khác cố định).

In [ ]:
t0 = time.time(); rn = np.random.default_rng(99)
trn, van = tr1.copy(), va1.copy(); trn['noise'] = rn.normal(size=len(trn)); van['noise'] = rn.normal(size=len(van))
FN = FEATURES + ['noise']; mn = fit_gb_df(trn, FN)
vsn = van.loc[va1s.index]; svn, _ = shap_vals(mn, vsn); shap_rank_n = svn.abs().mean().sort_values(ascending=False)
vpn = van.loc[vperm.index]; _, perm_n = perm_imp(lambda d: mn.predict(d), vpn[FN], vpn, {f: [f] for f in FN}); perm_rank_n = perm_n.sort_values(ascending=False)
print(f'Biến nhiễu: hạng SHAP = {list(shap_rank_n.index).index("noise") + 1}/{len(FN)} (mean |SHAP| = {shap_rank_n["noise"]:.5f}); hạng permutation = {list(perm_rank_n.index).index("noise") + 1}/{len(FN)} (ΔRMSPE = {perm_n["noise"]:+.5f})  ({time.time()-t0:.0f}s)')

# đối chiếu Promo trong dữ liệu thô (validation F1): trong từng cửa hàng và thứ T2-T6, chênh lệch log-doanh số giữa ngày có và không Promo
d_ = va1[va1.DayOfWeek <= 5].assign(ly=lambda x: np.log(x.Sales))
g_ = d_.groupby(['Store', 'DayOfWeek', 'Promo']).ly.mean().unstack('Promo').dropna()
emp = float(np.expm1((g_[1] - g_[0]).mean()) * 100)
print(f'Dữ liệu thô (validation {FOLD_XAI}): trong cùng cửa hàng và cùng thứ, ngày có Promo cao hơn ngày không Promo trung bình {emp:+.1f}% (từ {len(g_):,} cặp cửa hàng-thứ)')
print(f'Model dùng (SHAP nhóm Promo, fold {FOLD_XAI}): {promo_eff[FOLD_XAI]:+.1f}%')

### 16.4 Bảng kết quả kiểm tra độ tin cậy (tự động chấm theo tiêu chí ở đầu mục)

In [ ]:
chk_rows = []
others = [f for f in FOLD_NAMES if f != FOLD_XAI]
chk_rows.append(('1a. Top-5 biến trùng ≥ 4/5 với F1 (cả 3 fold còn lại)', all(overlap[f] >= 4 for f in others), overlap))
chk_rows.append(('1b. Spearman thứ hạng biến (SHAP) giữa fold ≥ 0,8', bool((sp_f.values[np.triu_indices(4, 1)] >= 0.8).all()), float(sp_f.values[np.triu_indices(4, 1)].min())))
chk_rows.append(('1c. Spearman thứ hạng NHÓM (permutation) giữa fold ≥ 0,8', bool((spg.values[np.triu_indices(4, 1)] >= 0.8).all()), float(spg.values[np.triu_indices(4, 1)].min())))
chk_rows.append(('1d. Chiều tác động Promo không đổi dấu giữa các fold', len({np.sign(v) for v in promo_eff.values()}) == 1, {k: round(v, 1) for k, v in promo_eff.items()}))
chk_rows.append(('2.  Hạt giống: Spearman ≥ 0,9', all(v >= 0.9 for v in sp_seed.values()), sp_seed))
rf_sp = {f: float(stats.spearmanr(df_['GB'], df_['RF'])[0]) for f, df_ in cmp_models.items()}
chk_rows.append(('3.  GB vs RF: Spearman thứ hạng nhóm ≥ 0,7', all(v >= 0.7 for v in rf_sp.values()), {k: round(v, 2) for k, v in rf_sp.items()}))
rk_s, rk_p = list(shap_rank_n.index).index('noise') + 1, list(perm_rank_n.index).index('noise') + 1
chk_rows.append(('4.  Biến nhiễu nằm ở nửa dưới bảng xếp hạng (cả SHAP và permutation)', rk_s > len(FN) / 2 and rk_p > len(FN) / 2, {'hạng SHAP': rk_s, 'hạng permutation': rk_p, 'tổng biến': len(FN)}))
chk_rows.append(('5.  Promo: model và dữ liệu thô cùng chiều', np.sign(emp) == np.sign(promo_eff[FOLD_XAI]), {'dữ liệu thô %': round(emp, 1), 'model %': round(promo_eff[FOLD_XAI], 1)}))
leak_ok = not (set(FEATURES) & FORBIDDEN)
chk_rows.append(('6.  Không có biến bị cấm (Customers/Sales thô/Id) trong danh sách biến được giải thích', leak_ok, 'ok' if leak_ok else 'CÓ LỖI'))
E03 = pd.DataFrame(chk_rows, columns=['Kiểm tra', 'Đạt?', 'Chi tiết']); E03['Đạt?'] = E03['Đạt?'].map({True: 'ĐẠT', False: 'CHƯA ĐẠT'})
display(E03)

**Nhận xét E03 (độ tin cậy của giải thích):**
**Kết quả chấm tự động: 7 mục ĐẠT, 2 mục CHƯA ĐẠT** (ngưỡng do nhóm đặt ra, không phải chuẩn khoa học).

**Đạt:**
- **Giữa 4 fold:** top-5 biến trùng 5/5, 4/5, 4/5, 4/5 với F1; Spearman thứ hạng mean |SHAP| từ 0,86 đến 0,95; tác động Promo cùng chiều và cùng độ lớn (+8,7% đến +9,8%); mẫu hình "thứ trong tuần" tương quan 0,90 / 0,80 / 0,99 với F1.
- **Giữa các hạt giống:** Spearman 0,986 và 0,993; top-5 trùng 4/5 và 5/5.
- **Cảm biến nhiễu:** biến ngẫu nhiên xếp 37/39 theo SHAP (mean |SHAP| 0,00006) và 30/39 theo permutation (ΔRMSPE +0,00001, gần 0). Lưu ý: nhiều biến thật cũng có ΔRMSPE gần 0 nên hạng 30/39 của permutation không mang nhiều thông tin.
- **Không có biến bị cấm** (`Customers`, `Sales` thô, `Id`) trong danh sách biến.

**Chưa đạt (và cách hiểu):**
- **1c - thứ hạng NHÓM giữa các fold:** Spearman thấp nhất 0,71 (< 0,8; giữa F2 và F4). Nhóm đứng đầu (lịch sử Sales) luôn là số 1 ở cả 4 fold (0,43-0,48). Sự khác biệt nằm ở các nhóm nhỏ: nhiều nhóm chỉ cách nhau vài phần nghìn RMSPE nên thứ hạng rất dễ đổi, và nhóm Ngày lễ thay đổi theo việc fold có ngày lễ hay không (0,045 ở F2, 0,028 ở F3, nhưng 0,004 ở F1 và F4). Nên kết luận hợp lý là *"thứ hạng nhóm đứng đầu thì chắc, thứ hạng các nhóm nhỏ thì không đáng tin"*.
- **3 - Gradient Boosting so với Random Forest:** Spearman thứ hạng nhóm là 0,95 ở F1 nhưng **chỉ 0,57 ở F4** (< 0,7). Cả hai đều xếp lịch sử Sales số 1, nhưng Random Forest gần như **không dùng** Promo (0,003 so với 0,034 của GB) và lịch (0,004 so với 0,023). Hai mô hình "giải thích" khác nhau ở các nhóm phía dưới, nên **mọi con số về Promo, lịch ở trên chỉ nói về Gradient Boosting**, không nên nói là "đặc tính của dữ liệu". (Random Forest cũng là mô hình kém hơn: RMSPE 0,141 so với 0,112.)

**Đối chiếu với dữ liệu thô (mục 5 của bảng kiểm):** chiều giống nhau, nhưng **độ lớn lệch bốn lần**: dữ liệu thô +41,4% (cùng cửa hàng, cùng thứ, T2-T6; 4.280 cặp cửa hàng-thứ), còn model dùng +9,4%. Bảng kiểm chỉ chấm theo chiều nên báo ĐẠT; đừng báo đây là "khớp". Lý do lệch xem Nhận xét M03 điểm 3 (hợp lý, chưa kiểm chứng riêng).

**Nghịch lý Simpson của ngày lễ (mục 16.5):** SHAP của `StateHoliday` trên các dòng ngày lễ ở F2 + F3 là **−27,2% (StoreType a, 39 dòng), −25,1% (d, 10 dòng) và +2,4% (b, 70 dòng)**. Chiều khớp với U02 mục 2.6 (b tăng, a/d giảm) nhưng số dòng rất ít, chỉ dùng 2 fold và `d` chỉ 10 dòng, nên chỉ là **gợi ý**, không phải kết luận.

### 16.5 Kiểm tra bổ sung: nghịch lý Simpson của ngày lễ (`StateHoliday` theo `StoreType`)
U02 mục 2.6 ghi: ngày lễ làm doanh số tăng ở StoreType b nhưng giảm ở a/d. Xem model có học được điều đó không. Chỉ có ít dòng ngày lễ trong validation (F2, F3), nên đọc kết quả như **gợi ý**, không phải kết luận.

In [ ]:
hol = []
for fold in [FOLD_NAMES[1], FOLD_NAMES[2]]:
    m_, vf, _, _, _ = FOLD_MODELS[fold]; vh = vf[vf.StateHoliday > 0]
    if len(vh) == 0: continue
    sv_, _ = shap_vals(m_, vh); hol.append(pd.DataFrame({'fold': fold, 'StoreType': vh.StoreType.map(dict(enumerate(CATS['StoreType']))).values, 'SHAP StateHoliday (log)': sv_['StateHoliday'].values}))
hol = pd.concat(hol); tab_h = hol.groupby('StoreType')['SHAP StateHoliday (log)'].agg(['mean', 'size']); tab_h['% đẩy dự báo'] = (np.expm1(tab_h['mean']) * 100).round(1)
print('SHAP của biến StateHoliday trên các dòng ngày lễ (F2 + F3), theo StoreType:'); display(tab_h.round(3))

**Kết luận xAI (tổng hợp M03 + E03):**
**Điều có thể nói khá chắc (qua 4 fold, 3 hạt giống và biến nhiễu):**
1. Gradient Boosting dự báo chủ yếu bằng **mức bán nền của chính cửa hàng** (`dow_promo_mean`: cùng thứ, cùng trạng thái Promo), sau đó điều chỉnh bởi lịch, `Store` và Promo.
2. **Promo** đẩy dự báo lên cùng chiều ở mọi fold và mọi loại cửa hàng, **khoảng +9%** ngoài mức nền; **thứ Hai cao nhất**, Chủ nhật thấp nhất.
3. **Promo2 và đối thủ gần như không được dùng** khi dự báo.
4. Các kết luận trên **ổn định** giữa 4 fold (SHAP Spearman 0,86-0,95) và giữa các hạt giống (≥ 0,986), và biến nhiễu không bị xếp cao.

**Điều còn yếu hoặc chưa kiểm chứng:**
- Con số +9% **không phải tác động của Promo** (dữ liệu thô +41%; phần còn lại nằm trong mức nền) và không phải nhân quả.
- Thứ hạng các **nhóm nhỏ** không ổn định giữa fold, và Random Forest "giải thích" khác Gradient Boosting (Spearman 0,57 ở F4): đừng khái quát các con số cho "mọi mô hình".
- `chain_delta` có chiều ngược trực giác; hiệu ứng đầu/cuối tháng, và lời giải thích vì sao bỏ biến lịch sử mà điểm gần không đổi (ở E02) đều là **suy đoán chưa kiểm chứng**.
- Ngày lễ và Chủ nhật chỉ dựa trên rất ít dòng.

**Dùng cho Business memo (D02):** dự báo của công ty nên được hiểu là *"mức bán điển hình của từng cửa hàng theo thứ và theo Promo, điều chỉnh nhẹ theo lịch"*; tác động của Promo2 và đối thủ nằm ngoài những gì mô hình này dùng.